# Verified, editable, publication-quality network graphs

This Colab-CPU-ready notebook contains the complete source internally and requires **no uploads or notebook inputs**. Run it top to bottom. Each graph is rendered as a high-resolution PNG and linked SVG, while the notebook itself is saved without embedded image outputs.

The renderer now applies a deterministic publication-quality pass before drawing: weighted layouts make stronger relationships shorter, node collisions are relaxed, numeric edge labels are routed through low-conflict positions, category annotations are repositioned, and legends are typeset from the available physical space. Graphs 1 and 4 use a three-column publication index so long entries do not collide.

The existing source, topic rules, graph definitions, edge values, degree-based node-size rules, and validation remain auditable in editable cells.

## Edit map

| Graph | Frame / titles / legend | Nodes and legend entries | Edges and edge labels | Description boxes | Render |
|---|---|---|---|---|---|
| Graph 1 | [edit](#scrollTo=g1_frame) | [edit](#scrollTo=g1_nodes) | [edit](#scrollTo=g1_edges) | [edit](#scrollTo=g1_notes) | [run](#scrollTo=g1_render) |
| Graph 2 | [edit](#scrollTo=g2_frame) | [edit](#scrollTo=g2_nodes) | [edit](#scrollTo=g2_edges) | [edit](#scrollTo=g2_notes) | [run](#scrollTo=g2_render) |
| Graph 3 | [edit](#scrollTo=g3_frame) | [edit](#scrollTo=g3_nodes) | [edit](#scrollTo=g3_edges) | [edit](#scrollTo=g3_notes) | [run](#scrollTo=g3_render) |
| Graph 4 | [edit](#scrollTo=g4_frame) | [edit](#scrollTo=g4_nodes) | [edit](#scrollTo=g4_edges) | [edit](#scrollTo=g4_notes) | [run](#scrollTo=g4_render) |

The configuration dictionaries contain the exact rendered values, not placeholders.

## Publication-quality revisions made

- Rebalanced the canvas and legend panels; Graphs 1 and 4 now use a wider, three-column publication index with physical-space-aware wrapping and row spacing.
- Replaced the sparse fixed layouts at render time with deterministic strength-weighted spring layouts, then relaxed node collisions while preserving the degree-to-radius rule.
- Reduced oversized publication-node scaling and tuned node-label, edge-label, note, title, subtitle, footer, and legend typography for print readability.
- Added automatic edge-label routing with curved alternatives and collision costs against nodes, other labels, and unrelated edges.
- Repositioned community, year, stage, and theme annotations near their groups without forcing excessive graph whitespace.
- Added rendered overlap diagnostics for edge labels, annotations, and legend entries; node overlaps are now a validation error rather than a warning.
- Kept the notebook Colab CPU compatible and dependency-light; no Graphviz, GPU, uploads, or interactive inputs are required.


In [1]:
# Colab-safe setup: pure Python/NetworkX; no Graphviz or file upload is required.
from __future__ import annotations

from pathlib import Path
import gc
import html
import hashlib
from collections import OrderedDict
import importlib.util
import math
import os
import re
import subprocess
import sys
import textwrap
import warnings

REQUIRED = {
    'numpy': 'numpy', 'pandas': 'pandas', 'networkx': 'networkx',
    'matplotlib': 'matplotlib', 'dateutil': 'python-dateutil',
    'scipy': 'scipy', 'sklearn': 'scikit-learn', 'PIL': 'pillow',
}
missing = [package for module, package in REQUIRED.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing])

import numpy as np
import pandas as pd
import networkx as nx
import matplotlib
matplotlib.use('Agg')
matplotlib.rcParams['svg.fonttype'] = 'none'
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch, Rectangle, PathPatch
from matplotlib.path import Path as MplPath
from dateutil import parser as dateparser
from scipy.stats import spearmanr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from PIL import Image as PILImage
from IPython.display import HTML, display

IN_COLAB = 'google.colab' in sys.modules
OUTPUT_DIR = Path('/content/cqd_network_graph_outputs') if IN_COLAB else Path.cwd() / 'cqd_network_graph_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# EDITABLE EXPORT SETTINGS
FIGSIZE = (24.0, 17.0)       # physical figure size in inches
EXPORT_DPI = 160             # 3840 × 2720 pixels; reliable on a standard Colab CPU runtime
DISPLAY_CLICKABLE_SVG = os.environ.get('CQD_DISPLAY_SVG', '0') == '1'


# ONE-COMMAND NODE-SIZE CONTROL
# Scale factors preserve every graph's existing relative node-size relationships.
# 1.00 = unchanged, 1.75 = 75% larger, 0.80 = 20% smaller.
NODE_RADIUS_SCALES = {}


def set_network_node_scales(*, graph01=1.00, graph02=1.00, graph03=1.00, graph04=1.00):
    """Set node-radius multipliers for all four network graphs in one command."""
    requested = {
        'graph1': float(graph01),
        'graph2': float(graph02),
        'graph3': float(graph03),
        'graph4': float(graph04),
    }
    invalid = {name: value for name, value in requested.items()
               if not np.isfinite(value) or value <= 0}
    if invalid:
        raise ValueError(f'All node-radius scales must be finite and greater than zero: {invalid}')
    NODE_RADIUS_SCALES.clear()
    NODE_RADIUS_SCALES.update(requested)
    print('Node-radius scales:', NODE_RADIUS_SCALES)


# Edit this single command to resize nodes in any or all network graphs.
set_network_node_scales(graph01=1.20, graph02=0.95, graph03=0.95, graph04=1.15)
STRENGTH_LENGTH_RHO_MAX = -0.25
EXPECTED_README_SHA256 = '8f9acb01cd2f521c3d2d4e209e194d81662ef4e317641c37fcd6cee2a759d1f7'

print(f'Outputs: {OUTPUT_DIR}')
print(f'PNG canvas: {int(FIGSIZE[0]*EXPORT_DPI)} × {int(FIGSIZE[1]*EXPORT_DPI)} pixels')


Node-radius scales: {'graph1': 1.2, 'graph2': 0.95, 'graph3': 0.95, 'graph4': 1.15}
Outputs: /content/cqd_network_graph_outputs
PNG canvas: 3840 × 2720 pixels


### Node-size control

All four network graphs are still controlled by one command in the setup cell:

`set_network_node_scales(graph01=1.20, graph02=0.95, graph03=0.95, graph04=1.15)`

The publication defaults reduce crowding while preserving every graph's relative degree-based size encoding. `1.00` retains the unscaled radius rule.

## Embedded README source

The next cell stores the complete uploaded README in a readable, editable cell. Runtime validation uses a canonical checksum that ignores trailing whitespace because IPython may trim whitespace-only lines.

In [2]:
README_TEXT = r"""<div align="center">
  <p>Clinical Trial Funding Application v2.0, RFA-RM-27-001, Kawchak K.</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>July 12, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents an NIH-adaptable clinical trial funding application and funder-outreach template for the NIH Director's Pioneer Award opportunity RFA-RM-27-001. The proposed five-year project, “Daraxonrasib Phase 1 LLM-Advised Robotic Whipple Trial in KRAS-Mutated PDAC,” identifies Kevin Kawchak and ChemicalQDevice as the applicant and applicant organization. The package is a drafting resource rather than a substitute for SF424 forms, ASSIST, Grants.gov Workspace, an agency portal, or another required submission system; the live solicitation should be verified before submission.

<div align="left">

<br>

Kawchak, K. (2026). Clinical Trial Funding Application v2.0, RFA-RM-27-001, Kawchak K. Zenodo. https://doi.org/10.5281/zenodo.21317266
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.21317266-blue)](https://doi.org/10.5281/zenodo.21317266)

---


<div align="center">
  <p>Clinical Trial Funding Application, RFA-RM-27-001, Kawchak K.</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>July 7, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents an NIH-adaptable clinical trial funding application and funder-outreach template for the NIH Director's Pioneer Award opportunity RFA-RM-27-001. The proposed five-year project, “Daraxonrasib Phase 1 LLM-Directed Robotic Whipple in KRAS-Mutated PDAC,” identifies Kevin Kawchak and ChemicalQDevice as the applicant and applicant organization. The package is a drafting resource rather than a substitute for SF424 forms, ASSIST, Grants.gov Workspace, an agency portal, or another required submission system; the live solicitation should be verified before submission.

<div align="left">

<br>

Kawchak, K. (2026). Clinical Trial Funding Application, RFA-RM-27-001, Kawchak K. Zenodo. https://doi.org/10.5281/zenodo.21232965
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.21232965-blue)](https://doi.org/10.5281/zenodo.21232965)

---


<div align="center">
  <p>Investigational New Drug Application - Daraxonrasib, Phase 1, LLM-Directed Robotic Whipple in KRAS-Mutated PDAC</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>July 1, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents a draft initial Investigational New Drug application under 21 CFR §312.23 for a Phase 1, first-in-human, prospective, open-label, single-arm study of perioperative daraxonrasib (RMC-6236) with an LLM-directed eight-arm robotic pancreaticoduodenectomy for patients with resectable or borderline-resectable KRAS G12-mutated pancreatic ductal adenocarcinoma. The application describes the proposed regulatory submission, associated FDA forms, clinical rationale, and mechanistic basis for daraxonrasib as a RAS(ON) multi-selective inhibitor. It is an independent research and adoption guide, not medical or regulatory advice and not an active or agency-endorsed IND.

<div align="left">

<br>

Kawchak, K. (2026). Investigational New Drug Application - Daraxonrasib, Phase 1, LLM-Directed Robotic Whipple in KRAS-Mutated PDAC. Zenodo. https://doi.org/10.5281/zenodo.21097442
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.21097442-blue)](https://doi.org/10.5281/zenodo.21097442)

---


<div align="center">
  <p>Phase 1 Pancreatic Cancer Trial: Efficient LLM Document Guidance</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 29, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

A Phase 1 oncology trial advances only when a sequence of large regulatory and clinical documents is authored, reviewed, and approved, while patients wait for that paperwork to be assembled. This paper illustrates a repository based large language model (LLM), driven by a single master prompt that writes and executes a schedule of sub-prompts committed to GitHub in real time; which can hasten the entire Phase 1 process by generating documents through an auditable "mermaid" to "draft" to "full" to "final" pipeline. The method was conducted on a prior pancreatic ductal adenocarcinoma program: an on-premises LLM directed robotic Whipple with perioperative daraxonrasib (RMC-6236), whose Phase 1 and Phase 2 protocols were themselves single-prompt multi-file builds. The following trial aspects were addressed: the initial IND and IRB package; protocol amendments with synchronized consent; cohort-review packages; the complete clinical-hold response; the Phase 2-to-3 briefing package; and the CSR and NDA/BLA modules. Only the administrative and preparation time bucket compresses, while clinical follow-up and fixed regulatory review clocks do not. Papers finish in 1-4 days, based on prompt and input status. Twenty-four colored Mermaid converted figures ground the prose in LaTeX. Paper repository, directory, and file name samples were verified manually by the author. For terminal patients, probable benefit exceeds probable risk.
<div align="left">

<br>

Kawchak, K. (2026). Phase 1 Pancreatic Cancer Trial: Efficient LLM Document Guidance. Zenodo. https://doi.org/10.5281/zenodo.21018646
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.21018646-blue)](https://doi.org/10.5281/zenodo.21018646)

---


<div align="center">
  <p>Oncology Trial PI LLM Adoption Guide</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 25, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

Repository based LLMs enable the creation of large scale documents more efficiently due to increased file generation capabilities, less errors, and a greater capacity to process an author's prompt containing specific instructions. This process allows for adequate incorporation of source information and iterative refinements of document drafts; while using a lower number of prompts. Repository level LLMs excel in understanding document context when a finite number of appropriately sized, high quality inputs are uploaded to the author's repository. Files generated in sections are advantageous for enabling direct access in future project builds. Single prompt multi-project outputs with a sub-prompt topics schedule informs LLM generation and execution of sub-prompts possible for creating machine readable diagrams and three sets of self learning and iteratively refined clinical trial paper files. Both clinical trial site and sponsor simulations can also be conducted at scale to observe patient, robotic, and AI interactions. LLM proficiency is possible for at least glioblastoma, pancreatic ductal adenocarcinoma, and lung adenocarcinoma. This practical guide builds from the HHS June 22, 2026 initiative to restore American leadership in clinical trials.
<div align="left">

<br>

Kawchak, K. (2026). Oncology Trial PI LLM Adoption Guide. Zenodo. https://doi.org/10.5281/zenodo.20843290
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20843290-blue)](https://doi.org/10.5281/zenodo.20843290)

---


<div align="center">
  <p>A Phase 2, Daraxonrasib + LLM Guided Robotic PDAC Whipple Procedure, Clinical Trial Protocol</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 23, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents a draft Phase 2 multicenter randomized controlled trial protocol for an on-premises LLM-guided robotic Whipple procedure with perioperative daraxonrasib (RMC-6236) in KRAS-mutated pancreatic ductal adenocarcinoma. The protocol builds on a Phase 1 predicate and addresses IND, IDE, Physical AI, good clinical practice, electronic records, single-IRB governance, trial registration, risk-benefit assessment, objectives, and endpoints. It is an independent research draft, not an active or approved IND/IDE protocol and not medical or regulatory advice.

Table of Contents

1 Statement of Compliance
1.1 Drug Arm Compliance (IND, 21 CFR Part 312)
1.2 Device Arm Compliance (IDE, 21 CFR Part 812)
1.3 Physical AI Overlay (21 CFR Part 312, Subpart J)
1.4 Good Clinical Practice and Electronic Records
1.5 Multicenter Single IRB and Co-Investment Governance
1.6 Trial Registration
1.7 Statement of Compliance Pathway
1.8 Signatures
2 Protocol Summary
2.1 Synopsis
2.2 Schema
2.3 Schedule of Activities (SoA)
3 Introduction
3.1 Study Rationale
3.2 Background
3.3 Risk/Benefit Assessment
3.3.1 Known Potential Risks
3.3.2 Known Potential Benefits
3.3.3 Assessment of Potential Risks and Benefits
4 Objectives and Endpoints
4.1 Primary Objective and Endpoint
4.2 Key Secondary Objectives and Endpoints
4.3 Secondary Objectives and Endpoints
4.4 Exploratory Objectives and Endpoints

<div align="left">

<br>

Kawchak, K. (2026). A Phase 2, Daraxonrasib + LLM Guided Robotic PDAC Whipple Procedure, Clinical Trial Protocol. Zenodo. https://doi.org/10.5281/zenodo.20807027
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20807027-blue)](https://doi.org/10.5281/zenodo.20807027)

---


<div align="center">
  <p>On-Premises LLM-Directed Robotic Pancreaticoduodenectomy with Perioperative Daraxonrasib (RMC-6236) in KRAS-Mutated Pancreatic Ductal Adenocarcinoma</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 21, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents a draft Phase 1, first-in-human, combined IND/IDE clinical trial protocol for an on-premises LLM-directed robotic pancreaticoduodenectomy with perioperative daraxonrasib (RMC-6236) in KRAS-mutated pancreatic ductal adenocarcinoma. The protocol integrates drug-arm, device-arm, Physical AI, good clinical practice, electronic-record, trial-registration, risk-benefit, objective, endpoint, and study-design requirements. It is an independent research draft, not an active or approved IND/IDE protocol and not medical or regulatory advice.

Table of Contents

1 Statement of Compliance
1.1 Drug Arm Compliance (IND, 21 CFR Part 312)
1.2 Device Arm Compliance (IDE, 21 CFR Part 812)
1.3 Physical AI Overlay (21 CFR Part 312, Subpart J)
1.4 Good Clinical Practice and Electronic Records
1.5 Trial Registration
1.6 Statement of Compliance Pathway
1.7 Signatures
2 Protocol Summary
2.1 Synopsis
2.2 Schema
2.3 Schedule of Activities (SoA)
3 Introduction
3.1 Study Rationale
3.2 Background
3.3 Risk/Benefit Assessment
3.3.1 Known Potential Risks
3.3.2 Known Potential Benefits
3.3.3 Assessment of Potential Risks and Benefits
4 Objectives and Endpoints
4.1 Primary Objectives and Endpoints
4.2 Secondary Objectives and Endpoints
4.3 Exploratory Objectives and Endpoints
5 Study Design
5.1 Overall Design
5.2 Scientific Rationale for Study Design
5.3 Justification for Dose
5.4 End of Study Definition

<div align="left">

<br>

Kawchak, K. (2026). On-Premises LLM-Directed Robotic Pancreaticoduodenectomy with Perioperative Daraxonrasib (RMC-6236) in KRAS-Mutated Pancreatic Ductal Adenocarcinoma. Zenodo. https://doi.org/10.5281/zenodo.20780121
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20780121-blue)](https://doi.org/10.5281/zenodo.20780121)

---


<div align="center">
  <p>Earning the Congress's Vote: A New Oncology Trial Framework for Enacting H. R. 9510</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 17, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

H. R. 9510, the Verification Before Generation in Physical AI Oncology Trials Act of 2026, would amend the Federal Food, Drug, and Cosmetic Act to require that an automated verification, validation, and uncertainty quantification examination clear robot-patient interaction code before that code is generated or executed in an oncology clinical investigation. This paper is a passage framework: it organizes the case for the bill as the eight questions a member of Congress asks before a yes vote, pairs with the corresponding National Platform, and the public record. The eight questions are the mandate for a statute, Congress's authority to act, the safety the bill creates, the fiscal score, the benefit to constituents, bipartisanship, the supporting coalition, and the passage path itself. The answers are concrete: a ten-gate safety floor bound to published consensus standards; an authorization of 58 million dollars over five years with no new mandatory spending and a verification cost roughly nineteen times lower than conventional review; and a platform that, in validated simulation, treated 168 patients with 29 robots at 99.7 percent uptime with zero patient harm events. The framework is written for the staffer, the member, and the committee counsel who must decide whether to mark up, cosponsor, and vote for the bill in both chambers. It is a new, analogous companion with one objective: making H. R. 9510 most likely to pass in both the House and the Senate.

Disclaimer: This work is independent and not endorsed or sponsored by the Congress, FDA, HHS, any Member, committee, trial sponsor, or medical society; it was generated by Claude Code Opus 4.8 Max using the author's prompt, followed by manual edits. The number “H. R. 9510” is illustrative. This is not enacted law and not legal advice.
<div align="left">

<br>

Kawchak, K. (2026). Earning the Congress's Vote: A New Oncology Trial Framework for Enacting H. R. 9510. Zenodo. https://doi.org/10.5281/zenodo.20726461
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20726461-blue)](https://doi.org/10.5281/zenodo.20726461)

---


<div align="center">
  <p>Earning the Clinician's Trust: New Framework for Verified Physical AI Oncology Trials</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 16, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

This framework is written for the clinicians who will be asked to supervise autonomous Physical AI in oncology clinical trials, and who must decide, at the bedside, when that reliance is earned. It rests on a single mechanism, verification before generation, in which a software agent proposes a clinical action and a ten-gate verification, validation, and uncertainty quantification examination either accepts it under audit, escalates it to a qualified human, or blocks it before it can reach a patient. The decision to trust is organized as eight plain questions a clinician already asks of any tool: is it competent, is it safe, is it transparent, am I in control, is it equitable, is it reliable, who is accountable, and does it fit my workflow. Each question pairs a clinical concern with a credible, cited fact, and closes on the concrete mechanism, a gate, a record, a role, or a control, that answers it. The aim is not maximal trust but calibrated trust: reliance proportioned to demonstrated capability, avoiding both automation bias and algorithm aversion. The framework draws on a documented lineage, including a surgical-humanoid simulation that passed 172 of 172 automated tests across a ten-gate suite, and on the literature of trust in automation, clinical decision support, and algorithmic bias. Three questions recur throughout: would I let this system near my own family member, can I defend this decision at tumor board, and could I show an auditor exactly what happened and why. The conclusion is an adoption decision a clinician can sign.

Disclaimer: This work is independent and not endorsed or sponsored by trial sponsors, FDA, CROs, sites, IRBs, regulators, or medical societies; was generated Claude Code Opus 4.8 Max using the author's prompt; followed by manual edits.
<div align="left">

<br>

Kawchak, K. (2026). Earning the Clinician's Trust: New Framework for Verified Physical AI Oncology Trials. Zenodo. https://doi.org/10.5281/zenodo.20710602
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20710602-blue)](https://doi.org/10.5281/zenodo.20710602)

---


<div align="center">
  <p>From H. R. 9510 to Federal Law: A Narrative Case for Verified Physical AI Oncology Trials</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 14, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

This review makes a case for legislators who shape medical and artificial intelligence law, but have little to no robotics or frontier large language model (LLM) application experience. It is built on a single mechanism, verification before generation, in which a software agent proposes a clinical action and a ten-gate verification, validation, and uncertainty quantification examination either accepts it, escalates it to a qualified human, or blocks it before it can reach a patient. The argument is organized as eight emotional pillars that legislative-advocacy research finds most persuasive: compassion, fear of preventable harm, moral outrage, hope, responsibility, protection of vulnerable people, trust, and urgency. Each pillar pairs a human appeal with a credible, cited fact. The review draws on a documented engineering lineage, including a surgical-humanoid assurance run that passed 172 of 172 automated tests across a ten-gate suite, and on the published advocacy literature describing how testimony, coalition building, and policy entrepreneurship move a bill through markup and reconciliation. The conclusion is that Physical AI Trial Bill H. R. 9510 2026 should be enacted into Federal law.

Disclaimer: This work is independent and not endorsed or sponsored by trial sponsors, FDA, CROs, sites, IRBs, regulators, or medical societies; was generated Claude Code Opus 4.8 Max using the author's prompt; followed by manual edits.
<div align="left">

<br>

Kawchak, K. (2026). From H. R. 9510 to Federal Law: A Narrative Case for Verified Physical AI Oncology Trials. Zenodo. https://doi.org/10.5281/zenodo.20685379
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20685379-blue)](https://doi.org/10.5281/zenodo.20685379)

---


<div align="center">
  <p>H. R. 9510 (Bill v5.0) 2026</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 10, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This legislative research draft proposes amending the Federal Food, Drug, and Cosmetic Act to require verification, validation, and uncertainty quantification before robot-patient interaction code is generated or executed in a Physical AI oncology clinical investigation. Bill v5.0 adds a financial-data record and provisions addressing transparency, user fees, authorization of appropriations, and budgetary effects. The measure was prepared with an autonomous artificial-intelligence coding agent under human direction. It is an independent draft, not enacted law or legal advice, and its financial figures are illustrative unless tied to a cited statute or notice.

<div align="left">

<br>

Kawchak, K. (2026). H. R. 9510 (Bill v5.0) 2026. Zenodo. https://doi.org/10.5281/zenodo.20619762
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20619762-blue)](https://doi.org/10.5281/zenodo.20619762)

---


<div align="center">
  <p>H. R. 9510 (Bill v4.0) 2026</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 7, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This legislative research draft proposes amending the Federal Food, Drug, and Cosmetic Act to require verification, validation, and uncertainty quantification before robot-patient interaction code is generated or executed in a Physical AI oncology clinical investigation. Bill v4.0 presents the full operative amendment with gray-scale Mermaid figures, full-width tables, and twelve submission deliverables in LaTeX appendices. The measure was prepared with an autonomous artificial-intelligence coding agent under human direction and is an independent draft, not enacted law or legal advice.

<div align="left">

<br>

Kawchak, K. (2026). H. R. 9510 (Bill v4.0) 2026. Zenodo. https://doi.org/10.5281/zenodo.20576907
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20576907-blue)](https://doi.org/10.5281/zenodo.20576907)

---


<div align="center">
  <p>H. R. 9510 (Bill v3.0) 2026</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 4, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This legislative research draft proposes amending the Federal Food, Drug, and Cosmetic Act to require verification, validation, and uncertainty quantification before robot-patient interaction code is generated or executed in a Physical AI oncology clinical investigation. Bill v3.0 is a visual amendment containing text figures, full-width tables, an engineering-evidence appendix, a submission-deliverables appendix, and an explainability-standard appendix. The measure was prepared with an autonomous artificial-intelligence coding agent under human direction and is an independent draft, not enacted law or legal advice.

<div align="left">

<br>

Kawchak, K. (2026). H. R. 9510 (Bill v3.0) 2026. Zenodo. https://doi.org/10.5281/zenodo.20535429
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20535429-blue)](https://doi.org/10.5281/zenodo.20535429)

---


<div align="center">
  <p>Verification Before Generation Act of 2026</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 1, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents a draft federal measure to amend the Federal Food, Drug, and Cosmetic Act by requiring an automated verification, validation, and uncertainty quantification process before robot-patient interaction code is generated or executed in a Physical AI oncology clinical investigation. The measure documents an AI-assisted drafting process conducted under human direction. It is an independent research draft, not enacted law or legal advice.

<div align="left">

<br>

Kawchak, K. (2026). Verification Before Generation Act of 2026. Zenodo. https://doi.org/10.5281/zenodo.20485580
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20485580-blue)](https://doi.org/10.5281/zenodo.20485580)

---


<div align="center">
  <p>VVUQ Physical AI Oncology Trial Bill</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 30, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents a draft bill for the 119th Congress that would require automated verification, validation, and uncertainty quantification before robot-patient interaction code is generated or executed in a Physical AI oncology clinical trial. The draft adapts public-domain federal regulatory material and licensed ICH material into a proposed legislative framework. It is independent, is not endorsed by regulators or clinical-trial organizations, and is not enacted law or legal advice.

<div align="left">

<br>

Kawchak, K. (2026). VVUQ Physical AI Oncology Trial Bill. Zenodo. https://doi.org/10.5281/zenodo.20454870
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20454870-blue)](https://doi.org/10.5281/zenodo.20454870)

---


<div align="center">
  <p>Mobile Pancreatic Cancer Unitree H2 Surgical Humanoid with Priority VVUQ</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 28, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

In this study, an autonomous Claude Code agent carried an instruction specification through a generated codebase and full execution record regarding a 2030 Unitree H2-Surgical 1.0 humanoid performing a 60-second pancreaticoduodenectomy. The surgical humanoid code assurance process, not code generation, is the substantial and decision-bearing work; holding verification, validation, and uncertainty quantification to a higher standard than the code itself. This process is what will make physical AI oncology trials faster, less expensive, and more beneficial to patients. The run passed 172 of 172 automated tests, 64 of them in the ten-gate assurance suite. Each gate required a verification fraction of exactly 1.0, validation agreement up to 1.00, relative error as tight as 0.01, and a coefficient-of-variation bound as tight as 0.05; five decision cases resolved to ten ACCEPT, three BLOCK, and one ESCALATE, and all 32 of 32 sweep iterations cleared every gate near a composite mean of 93.6. In a four-entrant, simulation-against-simulation tournament the mobile humanoid placed second (93.334) to the stationary eight-arm PancreSpeed cart (93.782). Parallel arms shorten the throughput-weighted score, so the single humanoid trails by under half a point, whereas the prior PDAC paper featured the multi-arm baseline alone. The 1790line comparison. json and the 1000-row sample_h2_sensor.csv reproduced byte-forbyte from seed 20260525. Binding every gate to published consensus standards already used in real life makes the assurance argument traceable and defensible to a regulator.

Disclaimer: This work is independent and not endorsed or sponsored by trial sponsors, FDA, CROs, sites, IRBs, regulators, or

medical societies; and was generated using Artificial Intelligence.
<div align="left">

<br>

Kawchak, K. (2026). Mobile Pancreatic Cancer Unitree H2 Surgical Humanoid with Priority VVUQ. Zenodo. https://doi.org/10.5281/zenodo.20421754
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20421754-blue)](https://doi.org/10.5281/zenodo.20421754)

---


<div align="center">
  <p>VVUQ Oncology Clinical Trial LLM Verification Automation Priority over Existing Generated Code</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 25, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

Physical AI is entering oncology trials through in silico simulation, autonomous code generation, and eventually robotic procedures - settings in which an error that ships costs far more than one caught prior to production. This work reports an end to end pipeline in which Claude Code performs five operations in order. Specifically,

it generates code from instructions, executes that code, generates image instructions, generates figures, then assembles a draft, and finally a full paper. The central argument is that none of these generation steps decides whether a deliverable is trustworthy; the verification, validation, and uncertainty quantification (VVUQ) process does. These precautions can also make autonomous physical AI oncology trials faster, less expensive, and more rigorous than conventional verifications. Code execution supports the claim directly in the cancer-automated repository, passing 51 of 51 automated tests across 8 modules and accelerated a representative schedule by a factor of 2.5, from 30 to 12 days. Its VVUQ gate, centered on generated script vvuq/vvuq_gate. py exercised across 6 cases, accepted 1 of 5 - escalating the blocked cases to a human, and requiring a verification fraction of exactly 1.0.

Disclaimer: This work is independent and not endorsed or sponsored by any clinical trial sponsors, FDA, CRO, site, IRB, regulator, or medical society; and was generated with the aid of Artificial Intelligence.
<div align="left">

<br>

Kawchak, K. (2026). VVUQ Oncology Clinical Trial LLM Verification Automation Priority over Existing Generated Code. Zenodo. https://doi.org/10.5281/zenodo.20372501
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20372501-blue)](https://doi.org/10.5281/zenodo.20372501)

---


<div align="center">
  <p>Threefold Humanoid 24/7 Adverse Event Oncology Trial Response Team: 4-Site Rotation</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 20, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

The future consists of humanoids responding to on-site repository based large language models that provide commands factoring in real-time sensor data and Cartesian coordinates to administer synergistic patient adverse events treatment. This paper describes the PAT-NET-001 continental network of 4 trial locations (San Francisco SF-01, San Diego SD-01, Boston BO-01, Atlanta AT-01), each running Claude Code Opus 4.7 1M Max and 3 Unitree H2 EDU humanoid robots per site in coordinated camarade swarms. The production chain proceeds through instruction generation, code generation across Python, C++, and Rust, and execution along a DuckDB aggregate. This occurs across 32 deterministic iteration sweeps covering a 168 hour monitoring window with 84 adverse events of which 24 reach CTCAE grade 3 or higher. The swarm achieves a 67.5 to 76.8 second median response time, a camaraderie invariant Pass rate of 0.954 to 0.985, and a FDA RTCT 1 hour SLA compliance rate near 0.999. The camaraderie pattern reduces robot error potential by a factor of 3 through peer cross checking, role rotation, hand off within 2 seconds, and shared sensor evidence.

Disclaimer: This work is independent and not endorsed or sponsored by trial sponsors, FDA, CRO, site, IRB, regulator, or medical society; and was generated using Artificial Intelligence.
<div align="left">

<br>

Kawchak, K. (2026). Threefold Humanoid 24/7 Adverse Event Oncology Trial Response Team: 4-Site Rotation. Zenodo. https://doi.org/10.5281/zenodo.20303281
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20303281-blue)](https://doi.org/10.5281/zenodo.20303281)

---


<div align="center">
  <p>2030: 60 Second Pancreatic Ductal Adenocarcinoma Robotic Whipple Procedure & Daraxonrasib Simulation</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 15, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

On premises repository based large language models (LLMs) provide commands to a parallel coelomic oncology robot based on real time sensor data that is controlled via x, y, Z coordinates to administer patient treatment and minimize error. This thesis was first initiated through a five phase Claude Code workflow (create instructions, generate code, code execution, draft generation, and full paper production) yielding a 60 second 8 arm robotic pancreaticoduodenectomy on a hypothetical 2030 Medtronic PancreSpeed 1.0 platform paired with the Daraxonrasib pan KRAS inhibitor (FDA Breakthrough Therapy 06/25, RASolute 302 + RASolve 301). PancreSpeed 1.0 closes a 5x to 500x SOTA gap (1,200 mm/s tip, 100 kHz force, 0.05 mm RMS, 3 ms E stop, 18 N cumulative cap, 640 sensor channels). A 32 iteration deterministic Latin hypercube sweep at seed 20260513 yields mean composite 93.298 (std 1.225, 95% CI half width 0.462), and the 4 entrant LLM tournament places PancreSpeed 1.0 first at 93.735 versus 2030 da Vinci SP and Hugo RAS successors. The 1001 record Phase 5 first 100 ms sensor_sample_- 8arm. json1 is an exceptional processing feat; Daraxonrasib postop restart distributes as T+7d in 29 of 32 iterations. Practical adoption still requires bridging the synthetic patient gap, hypothetical hardware gap, and TRIPOD+AI plus CREMLS reporting floors.

Disclaimer: This work is independent and not endorsed or sponsored by FDA, or any sponsor, CRO, site, IRB, regulator, or medical society; and was generated using Artificial Intelligence.
<div align="left">

<br>

Kawchak, K. (2026). 2030: 60 Second Pancreatic Ductal Adenocarcinoma Robotic Whipple Procedure & Daraxonrasib Simulation. Zenodo. https://doi.org/10.5281/zenodo.20196639
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20196639-blue)](https://doi.org/10.5281/zenodo.20196639)

---


<div align="center">
  <p>2030: 60 Second Glioblastoma AI Robotic Surgery</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 11, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

In this platform, an on-premises repository-based large language model (LLM) simulation provides commands to standard oncology surgical robots based on real-time sensor data which are controlled via x, y, z coordinates to administer patient treatment while minimizing robot error. This was achieved through several Claude Code prompts for code instructions, code generation, and code execution. This thesis is positioned as an opportunistic extension of the U.S. Food and Drug Administration April 2026 Real-Time Clinical Trials announcement, now with the surgical theater where a hypothetical 2030 4-arm Medtronic robot closes a 5x-200x performance gap against the Medtronic ROSA ONE Brain v3.0 baseline on tip velocity (1,000 vs 50 mm/s), acceleration (10,000 vs 200 mm/s squared), E-stop latency (5 vs 50 ms), and force resolution (0.001 vs 0.01 N). A 54 x 1001 row sensor sample table, a 16-iteration deterministic sweep, and an on-premises LLM tournament were steps achieved to accelerate clinical trial robotic workflows. Practical real-life adoption still requires bridging the synthetic patient gap and adopting TRIPOD+AI plus CREMLS reporting floors.

Disclaimer: This work is independent and not endorsed or sponsored by trial sponsors, FDA, CRO, site, IRB, regulator, or medical society; and was generated using Artificial Intelligence.
<div align="left">

<br>

Kawchak, K. (2026). 2030: 60 Second Glioblastoma AI Robotic Surgery. Zenodo. https://doi.org/10.5281/zenodo.20113157
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20113157-blue)](https://doi.org/10.5281/zenodo.20113157)

---


<div align="center">
  <p>Patient Priority of Proposed U.S. Bills for Physical AI Oncology Clinical Trials</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 7, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work compares proposed U.S. legislative approaches intended to increase future cancer patients' control over participation in Physical AI oncology clinical trials. The draft focuses on patient self-selection, definitions, implementation, and the relationship between adapted human-subject protections, decentralized-trial guidance, and federal patient-choice provisions. It is an independent research draft and does not represent enacted legislation or third-party endorsement.

Table of Contents

1 H.R. 9501 (2026) — Cancer Patient Self-Selection of Physical AI Oncology Trials Act of 2026 (Adaption of 21 CFR Part 50, FDA Decentralized Trial Final Guidance 2024, and 42 U.S.C. 300gg-8)
1.1 Findings and Declarations
1.2 Definitions
1.3 Patient Self-Selection Rights
1.4 Implementation

<div align="left">

<br>

Kawchak, K. (2026). Patient Priority of Proposed U.S. Bills for Physical AI Oncology Clinical Trials. Zenodo. https://doi.org/10.5281/zenodo.20045457
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.20045457-blue)](https://doi.org/10.5281/zenodo.20045457)

---


<div align="center">
  <p>Accelerated Patient Prediction in Physical AI Oncology Clinical Trials: 4 Extensive Simulations</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 4, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

On 28 April 2026, the U.S. Food and Drug Administration announced two real-time clinical trial proofs-of-concept and a pilot Request for Information, framing the conduct of trials in which key safety signals no longer take years to reach the agency. The prevailing oncology AI patient-prediction baseline remains a heterogeneous set of narrow supervised models with ceilings such as Manz 2020 AUC 0.89, the SHIELD-RT prospective randomized trial, SCORPIO, and PROGPATH, set against the Huang 2025 null result that machine learning provides no significant gain over Cox regression on real-world structured survival data. This paper uses four author Physical AI oncology trial simulations - Simulation | in hour-00 through hour-55, Simulation 2 ten patient-journey stages, Simulation 3 a 24-hour autonomous sponsor, and Simulation 4 a 168-hour 7-day sponsor extension verified locally on a Core i5-6200U laptop with 4 GB RAM - to demonstrate that Claude Code Opus 4.7 Max produces working agentic code applicable to patient prediction better than supervised models in current trial practice and with more utility than the FDA RTCT proof-of-concept. The computational signature - 1M token contextual code-and-text awareness, hourly commit cadence, repository-scale forecasting - is the source of the advantage, without claiming clinical deployment readiness.

Disclaimer: This work is independent and is not endorsed or sponsored by any trial sponsor, FDA, CRO, site, IRB, regulator, or medical society; and was generated using Artificial Intelligence.
<div align="left">

<br>

Kawchak, K. (2026). Accelerated Patient Prediction in Physical AI Oncology Clinical Trials: 4 Extensive Simulations. Zenodo. https://doi.org/10.5281/zenodo.19994945
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.19994945-blue)](https://doi.org/10.5281/zenodo.19994945)

---


<div align="center">
  <p>Fully Automated Sponsor: Physical AI Oncology Clinical Trial Platform</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>April 5, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

This paper presents a fully autonomous AI-native sponsor operating system for Physical AI oncology clinical trials, replacing traditional human-staffed pharmaceutical sponsor functions with a multi agent software architecture. The system comprises twelve functional agents organized into four layers: governance, study execution, site and robotics interface, and trust infrastructure. Each agent automates a distinct sponsor responsibility, from portfolio management and protocol design through safety monitoring, regulatory submissions, and robotic procedure authorization, while maintaining compliance with adapted regulatory frameworks including 21 CFR Part 312, 21 CFR Part 50, and ICH E6(R3). The architecture integrates with national MCP server infrastructure, federated learning networks, and Physical AI trial sites equipped with ten categories of robotic systems. This version includes automated code generation results: 108 Python scripts generated from the paper’s LaTeX instructions by Claude Code Opus 4.6, with successful execution of a 24-hour simulation producing 288 sponsor decisions across 155 patients and 75 text diagrams across three perspectives. A human sponsor-of-record retains legal accountability and override authority for safety-critical decisions, ensuring that autonomous operation occurs within established regulatory boundaries.
<div align="left">

<br>

Kawchak, K. (2026). Fully Automated Sponsor: Physical AI Oncology Clinical Trial Platform. Zenodo. https://doi.org/10.5281/zenodo.19396256
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.19396256-blue)](https://doi.org/10.5281/zenodo.19396256)

---


<div align="center">
  <p>National Platform for Physical AI Oncology Trials</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 28, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work presents a draft national-platform framework for Physical AI oncology trials. It adapts public-domain federal regulatory material and licensed ICH guidance into a proposed unified resource for trial governance and implementation. The document is an independent research draft and is not affiliated with or endorsed by CFR, ICH, FDA, a trial sponsor, a contract research organization, a clinical site, an institutional review board, a regulator, or a medical society.

<div align="left">

<br>

Kawchak, K. (2026). National Platform for Physical AI Oncology Trials. Zenodo. https://doi.org/10.5281/zenodo.19244918
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.19244918-blue)](https://doi.org/10.5281/zenodo.19244918)

---


<div align="center">
  <p>Physical AI Oncology Clinical Trial Site Complete Documentation Package</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 24, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work compiles an eleven-document package for a proposed California Physical AI oncology clinical trial site. The package addresses legislation, regulations, building code, premises code, and site operations, including a draft California authorization and site-establishment act. It is an unofficial independent research draft and does not express third-party endorsement, sponsorship, affiliation, or authorization.

Contents

SB 1042 — California Physical AI Oncology Clinical Trial Authorization and Site Establishment Act of 2026
1 Legislative Findings and Declarations
2 Definitions
3 Authorization to Establish Physical AI Oncology Clinical Trial Sites

<div align="left">

<br>

Kawchak, K. (2026). Physical AI Oncology Clinical Trial Site Complete Documentation Package. Zenodo. https://doi.org/10.5281/zenodo.19176370
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.19176370-blue)](https://doi.org/10.5281/zenodo.19176370)

---


<div align="center">
  <p>A Cancer Patient's Journey Through a Regulated Physical AI Oncology Trial</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 21, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

This simulation presents the first fully autonomous, single-patient journey through a regulated Physical AI oncology clinical trial, orchestrated end-to-end by Claude Code Opus 4.6 in a single pull request comprising 13 commits completed in 72 minutes without user intervention. The patient journey follows PAT-2026-0042, a 58-year-old female with Stage IIIB non-small cell lung cancer (NSCLC) adenocarcinoma (ECOG 1, PD-L1 65%, TMB 14 mut/Mb), through 10 sequential clinical stages: prescreening, enrollment, digital twin initialization, robot qualification, robotic surgery, post-operative recovery, immunotherapy, federated learning, surveillance, and trial closeout. The journey is governed by three adapted regulatory frameworks: 21 CFR Part 312 Subpart J, 21 CFR Part 50 Subpart C, and ICH E6(R3) - which were previously adapted by the author for Physical AI systems. The orchestration infrastructure comprises 12 Python modules (master_journey.py, patient_state.py, and 10 stage-specific scripts), 30 ASCII progress diagrams across three perspectives (timeline, regulatory, clinical), 6 text-based deliverable diagrams, 6 regulatory tables, 4 guidance documents, and an FDA cost-savings analysis projecting 30-50% trial cost reductions ($390M-$650M savings against a $1.3B baseline). The da Vinci Xi surgical robot (USL 87.5) performed a 168-minute robotic lobectomy achieving RO resection with negative margins, while the Franka Emika (USL 88.75) managed laboratory automation and pharmacy dosing. Across 35 pembrolizumab immunotherapy cycles, the patient achieved complete response (CR) with recurrence risk declining from 35% to 3% over 36 months of event-free survival. The project demonstrates that fully autonomous AI workflows can produce clinically coherent, regulatory-compliant trial illustrations with real-time human monitorability through timestamped text diagrams and accessible Python scripts at each commit. This publication should not be considered a new or approved standard or regulation.
<div align="left">

<br>

Kawchak, K. (2026). A Cancer Patient's Journey Through a Regulated Physical AI Oncology Trial. Zenodo. https://doi.org/10.5281/zenodo.19119939
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.19119939-blue)](https://doi.org/10.5281/zenodo.19119939)

---


<div align="center">
  <p>Adaption: 21 CFR Part 312, End-to-End Physical AI Oncology Clinical Trial Unification</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 17, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work adapts 21 CFR Part 312, Investigational New Drug Application, for an end-to-end Physical AI oncology clinical trial unification framework. The public-domain eCFR text was reconstructed in Markdown and further adapted into LaTeX to organize investigational-drug requirements for the proposed Physical AI setting. The work is an independent modified regulatory draft and is not endorsed or sponsored by the Code of Federal Regulations or the FDA.

<div align="left">

<br>

Kawchak, K. (2026). Adaption: 21 CFR Part 312, End-to-End Physical AI Oncology Clinical Trial Unification. Zenodo. https://doi.org/10.5281/zenodo.19057628
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.19057628-blue)](https://doi.org/10.5281/zenodo.19057628)

---


<div align="center">
  <p>Adaption: 21 CFR Part 50, End-to-End Physical AI Oncology Clinical Trial Unification</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 16, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work adapts 21 CFR Part 50, Protection of Human Subjects, for an end-to-end Physical AI oncology clinical trial unification framework. The public-domain eCFR text was reconstructed in Markdown and further adapted into LaTeX to organize human-subject protections for the proposed Physical AI setting. The work is an independent modified regulatory draft and is not endorsed or sponsored by the Code of Federal Regulations or the FDA.

<div align="left">

<br>

Kawchak, K. (2026). Adaption: 21 CFR Part 50, End-to-End Physical AI Oncology Clinical Trial Unification. Zenodo. https://doi.org/10.5281/zenodo.19040707
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.19040707-blue)](https://doi.org/10.5281/zenodo.19040707)

---


<div align="center">
  <p>Adaption: ICH Harmonised Guideline, End-to-End Physical AI Oncology Clinical Trial Unification</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 12, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


This work adapts the ICH E6(R3) Guideline for Good Clinical Practice into a proposed Good Physical AI Clinical Practice framework for end-to-end Physical AI oncology trial unification. The source guideline's formatting was reconstructed in LaTeX and modified to address the proposed Physical AI setting. The work is an independent adaptation and is not endorsed or sponsored by ICH.

<div align="left">

<br>

Kawchak, K. (2026). Adaption: ICH Harmonised Guideline, End-to-End Physical AI Oncology Clinical Trial Unification. Zenodo. https://doi.org/10.5281/zenodo.18973368
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.18973368-blue)](https://doi.org/10.5281/zenodo.18973368)

---


<div align="center">
  <p>National MCP Servers for Physical AI Oncology Clinical Trial Systems</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 9, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

Prior oncology clinical trial infrastructure has been fragmented, site-specific, and inefficient across the critical dimensions of interoperability, auditability, privacy, and deployment. Each trial site has historically operated its own isolated systems for electronic health records, imaging archives, and audit logging, resulting in inconsistent data formats, duplicated regulatory effort, and limited cross-site collaboration. This paper presents the National MCP Physical AI Oncology Trials system, a proposed end-to-end architecture comprising five Model Context Protocol (MCP) servers that address these gaps through standardized, federated, and safety-governed capabilities. The five servers (Authorization, FHIR Clinical Data, DICOM Imaging, Audit Ledger, and Provenance) expose 23 tools across five hierarchical conformance levels, validated by 668 test functions. The system integrates AI and robotics advances intended to improve patient safety and clinical effectiveness, including emergency stop coordination, procedure state machines, and multi-party approval checkpoints. Quantitative analysis of the repository demonstrates 381 files across 88 directories, 34 integration adapters, 8 safety modules, 13 JSON schemas, and dual-language SDKs in Python and TypeScript. This end-to-end MCP Physical AI oncology trial system provides the foundation for national-scale standardization of robotic oncology clinical trials.
<div align="left">

<br>

Kawchak, K. (2026). National MCP Servers for Physical AI Oncology Clinical Trial Systems. Zenodo. https://doi.org/10.5281/zenodo.18916731
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.18916731-blue)](https://doi.org/10.5281/zenodo.18916731)

---


<div align="center">
  <p>TrialMCP: MCP Servers for Physical AI Oncology Clinical Trial Systems</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 5, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

Multi-site oncology clinical trials increasingly deploy autonomous robotic platforms for specimen handling, imaging guidance, and procedural automation. Integrating these Physical AI systems with clinical infrastructure—scheduling, electronic data capture (EDC), eConsent, imaging archives, and laboratory information systems—currently requires point-to-point integrations that grow quadratically with the number of platforms and sites. This paper presents TrialMCP, an open-source suite of five Model Context Protocol (MCP) servers that provides a standardized interoperability layer between autonomous trial robots and clinical systems. We describe the end-to-end operational workflow from token-based authentication through study status retrieval, imaging pointer acquisition, evidence logging, and provenance tracking. The architecture supports role-based access control with deny-by-default semantics across 23 MCP tools, HIPAA Safe Harbor de-identification, and SHA-256 hash-chained audit trails satisfying 21 CFR Part 11. Validation across 39 tests—spanning security, audit completeness, and integration scenarios—demonstrates operational readiness. We present deployment topology patterns for federated multi-site trials, an adoption pathway with three milestone phases, and quantitative success criteria including projected reductions in integration project count, site onboarding time, and audit preparation cycles. TrialMCP v0.2.0 establishes the M1 milestone: read-only clinical data servers with a complete authorization framework, positioning the system for production FHIR/DICOM proxy integration in subsequent milestones.
<div align="left">

<br>

Kawchak, K. (2026). TrialMCP: MCP Servers for Physical AI Oncology Clinical Trial Systems. Zenodo. https://doi.org/10.5281/zenodo.18870961
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.18870961-blue)](https://doi.org/10.5281/zenodo.18870961)

---


<div align="center">
  <p>Federated Learning Physical AI Oncology Trials Unification</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>February 27, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

The transition from conventional software-only artificial intelligence to physical AI systems incorporating robotic hardware in oncology clinical trials represents a paradigm shift requiring unified infrastructure for privacy, regulation, cross-framework interoperability, and multi-organization cooperation. This paper presents the PAI Oncology Trial FL platform (v1.1.0), a comprehensive federated learning framework comprising 235 Python modules (~86,800 lines of code) that unifies five critical infrastructure pillars: (1) Privacy Infrastructure implementing all 18 HIPAA Safe Harbor identifiers with HMAC-SHA256 pseudonymization, (2) Regulatory Infrastructure spanning FDA, IRB, ICHGCP, and multi-jurisdiction compliance across v0.6.0 and v0.9.1, (3) Cross-Framework Unification bridging NVIDIA Isaac Sim, MuJoCo, Gazebo, and PyBullet simulation environments, (4) Standards & Benchmarking for QI 2026 objectives including model conversion and registry pipelines, and (5) Multi-Organization Cooperation enabling federated training across academic medical centers, community hospitals, and pharmaceutical companies. End-to-end workflow demonstrations are presented across 31 example scripts, 6 agentic AI production examples implementing Model Context Protocol (MCP), ReAct reasoning, real-time monitoring, autonomous orchestration, safetyconstrained execution, and RAG-based compliance. A triple AI peer review process (v0.9.4-v0.9.9) using sequential Codex-to-Claude Code review-fix cycles resolved 31/31 code recommendations at 100% completion, establishing a dual-manufacturer trust benchmark for AI-generated clinical trial software. The platform demonstrates that unified federated learning infrastructure is a necessary precondition for transitioning the oncology industry to using robots in physical AI clinical trials.
<div align="left">

<br>

Kawchak, K. (2026). Federated Learning Physical AI Oncology Trials Unification. Zenodo. https://doi.org/10.5281/zenodo.18795507
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.18795507-blue)](https://doi.org/10.5281/zenodo.18795507)

---


<div align="center">
  <p>Unification Standard Level for Physical AI Oncology Trials</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>February 26, 2026</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

As physical AI systems advance toward clinical deployment in oncology, no standardized framework exi evaluate how ready a robotic platform is for unified, multi-site clinical trials. Current technology rea assessments (e.g., NASA TRL, MLTRL) do not capture the unique demands of cross-platform simulation s AI integration, inter-organizational robot progress sharing, and federated regulatory compliance required for multi-site oncology trials. This paper introduces the Unification Standard Level (USL), a 1.0-10.0 scoring framework that evaluates physical AI robots across four equally weighted dimensions: (A) Simulation Framework Switching, (B) Generative/Agentic AI Integration, (C) Cross-Robot Progress Sharing, and (D) Multi-Site Clinical Trial Collaboration. We apply USL to nine robots across three categories—collaborative robots (cobots), surgical robots, and humanoid robots—finding per-dimension scores ranging from 1.5 to 8.5 and final composite scores from 3.4 to 7.4. The Franka Emika Panda (USL 7.4) and da Vinci dVRK (USL 7.1) lead their respective categories, both driven by large open-source ecosystems. Clinical trial readiness (Dimension D) remains the weakest dimension for seven of nine robots evaluated, revealing a field-wide gap between research maturity and clinical deployment infrastructure. All scoring code, robot evaluation modules, and documentation are open-source at https: //github. com/kevinkawchak/physical-ai-oncology- trials under MIT license.
<div align="left">

<br>

Kawchak, K. (2026). Unification Standard Level for Physical AI Oncology Trials. Zenodo. https://doi.org/10.5281/zenodo.18778220
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.18778220-blue)](https://doi.org/10.5281/zenodo.18778220)

---


<div align="center">
  <p>Code Generation Competition: 16 Proprietary vs. Open-Source LLMs & Iterative Learning Based on FDA Adverse Event Reporting System</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>December 22, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">


Few effective goal-oriented iterative LLM code benchmarking studies exist. Successive high dimensional and complex problem improvements are desired versus conventional code assessments. Inspired by a recent CodeClash study, this tournament focuses primarily on the goal of generating functions to obtain a perfect competition task score based on three recent FDA FAERS files. Here, Opus 4.5 Extended was primarily utilized to build a novel Python evaluation engine measuring LLM code pair correctness, methodology, code quality, and algorithm effectiveness against a fixed reference standard and head-to-head. The notebook then automated Code A and Code B grading, and outputted their answers and reference standard of drug-reaction signals in csv files. The bracket was organized at scale: 16 LLMs - 8 proprietary LLMs on the left and 8 open-source LLMs on the right. The 8 Round 1 winners and corresponding notebooks were then re-introduced to each LLM with a competition prompt to generate the next round’s code submission. Iterative learning in the form of improved final scores was observed for several Round 2 winners, which was based on its prior round competition code, competitors’ code, and results. Gpt-5.2-pro and Gemini 2.5 Pro API were effective at iterative learning on the FAERS dataset goal; while Kimi K2 Thinking saw the biggest single round score increase at +0.405. Contestant models were from xAI, OpenAI, Gemini, Claude, DeepSeek, Kimi, GLM, MiniMax, and Qwen manufacturers.
<div align="left">

<br>

Kawchak, K. (2025). Code Generation Competition: 16 Proprietary vs. Open-Source LLMs & Iterative Learning Based on FDA Adverse Event Reporting System. Zenodo. https://doi.org/10.5281/zenodo.18029100
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.18029100-blue)](https://doi.org/10.5281/zenodo.18029100)

---


<div align="center">
  <p>AI Peer Review Acceleration of LLM-Generated Glioblastoma Clinical Trial Patient Matching ML, FDA/ICH/ISO, and FastAPI</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>November 30, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

Human peer review was an effective technique to address quality, originality, and errors of non-LLM generated work. Researcher-AI trust has since then grown due to performant large language model (LLM) benchmark improvements and author implementations across different model manufacturers for processing frequent, complex, and voluminous data. This single human-AI team has led to the emergence of readily available artificial intelligence peer review. A primary benefit of AI peer review is the ability to appropriately conduct iterative analyses and corrections throughout the entire manuscript process. This responsible method improves on over-worked and fatigued human recommendations provided after manuscript completion. Here, a maximum efficiency study was performed in 14 days with the author utilizing a prior glioblastoma clinical trial matching guidance and a TrialGPT paper as inputs to output a prompt for developing a Python deep learning pipeline. A 10 file dataset and clinical BERT model notebook were generated by Sonnet 4.5 Extended, followed by repetitions of code fixes and optimizations to yield a final notebook at a low test set performance of 67.3%. A triple AI peer review conducted by Sonnet, GPT 5.1, and Grok 4.1 all yielded the primary recommendation to upgrade to a tabular model, which was more suitable for the csv dataset.

ChatGPT then identified the most appropriate selection: an open source TabPFN-v2-clf model from Hugging Face. The Sonnet recommendations also included detailed instruction for five-fold cross-validation, SHAP explainability, and bias analysis incorporation. An AI peer review evaluation standard for ML workflows based criterion and efficiency metrics was generated by Opus 4.5 with highest efficiencies based on the least number of humans, prompts, and time at the lowest LLM cost. Subsequent evaluation of the current workflow with an overall quality of 87.5%, and 0.831 efficiency based on human, prompt, time, and cost metrics. The ChatGPT identified model and Sonnet review instructions were implemented into the prior notebook to yield a new notebook that saw an acceleration in research progress with a test set accuracy improvement to 94.0% with few-shot learning along with peer review corrections. A regulatory compliance prompt was created, followed by full document generations of FDA 21 CFR Part 820, ICH-GCP E6(R2), and ISO 14971 addressing quality, clinical practice, and risk management. Several attempts to create an effective FastAPI code repository by Sonnet were initially unsuccessful, however an Opus/Sonnet prompting strategy yielded a FastAPI interface with three successful glioblastoma patient clinical trial match predictions. AI as a senior software engineer and as a senior regulatory analyst were also employed as part of the AI peer review process.

The purpose of this AI peer review study was to place the world’s focus back on innovating active and real world medical AI advancements in a fraction of the time. Measuring the effect AI implementation has on large workforces can be challenging to quantify. The conversational human-AI R&D relationship has become increasingly solitude: therefore this single author project should be viewed as the standard for new workflows going into the future. This next evolution of peer review is especially suitable for hard working researchers in less favorable economic, educational, and affiliation conditions, such as the author of the paper. This process is afforded by low cost, fast, and easy to use state of the art LLMs from Anthropic, Google, OpenAI, and xAI; marking the pivot away from slow and irresponsible use of human peer review for LLM-generated data, and back to fast automation of patient health applications at scale. Many authors of less influence can now improve their speed of development and gain reputation in papers by simply providing prompts, LLMs, AI peer review recommendations, and corrections they made - without the need for paid review or journals. Interested parties can then view the author’s peer review findings alongside the paper. Trust for the AI peer review process was established over a series of human-AI works reflected in LLM code and non-code benchmarking, applications in literature, and prior author works using AI in review roles coordinated across judging, external validation, cross-verification, meta-verification, and tests according to FDA/ICH guidance.
<div align="left">

<br>

Kawchak, K. (2025). AI Peer Review Acceleration of LLM-Generated Glioblastoma Clinical Trial Patient Matching ML, FDA/ICH/ISO, and FastAPI. Zenodo. https://doi.org/10.5281/zenodo.17774560
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.17774560-blue)](https://doi.org/10.5281/zenodo.17774560)

---


<div align="center">
  <p>LLM-Generated Glioblastoma Drug Synergy Machine Learning: From Rapid Code Prototypes to Project Deliverables Package</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>November 14, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

Large language models (LLMs) are best utilized through their ability to rapidly prototype code which can be executed locally, causing minimal workflow disruption to existing drug development processes and addresses validation, verification, and transparency concerns. Essentially, LLM written code substituting human code mitigates concerns of LLM black box interpretability and explainability issues. Here, this study uses LLMs to develop, peer review, and make code corrections, yielding a feasible drug synergy machine learning pipeline. This was achieved utilizing Sonnet 4.5 Extended to provide a six machine learning model Python notebook and 1020x26 dataset of 153 unique glioblastoma drug interaction pairs in a single output; with ChatGPT 5 Thinking as a senior peer reviewer recommending fixes to label leakage, group-aware cross validation, and probability calibration. Subsequent Sonnet optimizations focused on an enhanced random forest model at an accuracy of 0.9804 and Macro-F1 of 0.9705 in predicting three drug synergy classes. The final Sonnet automated output bundle of notebook, summary, implementation guide, recommendations mapping, and README files ensured end-to-end reproducibility, auditability, and version control - mimicking a real-world ML release. Four non-Sonnet inference LLMs were used to provide exhaustive analyses regarding performance and industry relevance of the notebook and dataset. LLM code generation using widely available proprietary models has validation advantages over direct LLM prompting; with reduced workflow complexity over fine-tuned, agentic, and augmented retrieval methods. Therefore: easy to use, cost effective, and mainstream LLMs proficient in web search and multi-format uploads are ideal for generating code that can be run locally for everyday drug development tasks; with direct LLM prompts for conducting AI peer review based on complex and voluminous data - marking the transition from LLM benchmark competitions to full and reviewed LLM applications.

<div align="left">

<br>

Kawchak, K. (2025). LLM-Generated Glioblastoma Drug Synergy Machine Learning: From Rapid Code Prototypes to Project Deliverables Package. Zenodo. https://doi.org/10.5281/zenodo.17614396
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.17614396-blue)](https://doi.org/10.5281/zenodo.17614396)

---


<div align="center">
  <p>End-to-End Oncology Clinical Trial LLM Efficiency For Industry Adoption with FDA/ICH Regulations</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>October 26, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

The process of a new oncology treatment from drug discovery and preclinical studies through Phase III clinical trials and FDA review can take over a decade and cost hundreds of millions of dollars. Therefore, automated and easy to use large language models which have adapted to code generation for production use of sensitive data should be implemented into cancer trial workflows at scale. Here, LLMs were implemented throughout the study to create table templates, datasets based on author and online literature; and followed
by triple and quintuple LLM workflows for optimization, table population, peer review, and meta-analyses. Primary models utilized were Sonnet 4.5 Extended, Gemini 2.5 Pro, GPT-5 High, and Grok 4 Fast - with Sonnet being used the most to quickly solve increasingly larger problems and reach consensuses between other LLM responses. The iterated workflow features 12 tables from discovery through Phase III trials, NDA/BOA & FDA Review that identified potential prior LLM applications, with estimated LLM time and cost reductions
for trial activities. These itemized values for each table were summed by AI and provided in a cost summary of an entire glioblastoma drug development process. The 2025 trial baseline total was estimated to be 10-15 years at a cost of $282M-$837M, while the LLM/AI assisted workflow was projected to be 4.8-7.7 years at $215M-$673M, which is 52-73% faster and 24-50% less expensive. Key performance indicators of baseline vs. LLM, acceleration and cost reduction areas, regulatory framework summary, and call to action were also provided by Sonnet, marking a transition to broader adoption of LLMs for oncology clinical trials. Note: LLM outputs and LLM-generated code locally executed with patient data may require FDA approval.

<div align="left">

<br>

Kawchak, K. (2025). End-to-End Oncology Clinical Trial LLM Efficiency For Industry Adoption with FDA/ICH Regulations. Zenodo. https://doi.org/10.5281/zenodo.17451709

[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.17451709-blue)](https://doi.org/10.5281/zenodo.17451709)

---


<div align="center">
  <p>Accelerating FDA Compliance and Cost Efficiency of in silico Clinical Trials via AI Digital Twin Pancreatic Cancer Simulation</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>September 30, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

**Question of Interest:**  Can a bidirectional PDAC digital twin, subjected to M15-aligned verification, validation, uncertainty quantification, and applicability assessment, provide sufficiently credible comparative predictions of arm-level efficacy/safety (ORR, DCR, mPFS, mOS, HRs, G3+ AE, dropout) to prioritize Phase II platform-trial arms and inform design choices (e.g., eligibility mix, progression threshold, horizon), thereby reducing empirical iteration while preserving patient safety?

**Context of Use:** A cohort-level digital twin simulates a 10-arm PDAC platform trial (≈100 patients/arm, 36-month horizon, dt=1 day). Tumor dynamics use a two-compartment sensitive/resistant Emax framework with archetype-driven growth rates and a lognormal “sensitivity multiplier.” Survival uses a base hazard with post-progression and biomarker multipliers; toxicity generates G3+ events with dropout probability; a rule-based policy effects 1L→2L→BSC transitions. The twin executes a closed-loop sense–analyze–recommend–act–learn cycle with per-patient logs. Data/knowledge used: drug parameters and archetypes transferred from prior QSP work and literature; control-arm external targets from MPACT (Arm A), NAPOLI-1 (Arm G), and POLO (Arms J/K). Specific role of model outcomes: rank arms, estimate HRs and endpoint distributions, test design assumptions. Other evidence: an a priori VV40/FDA test suite executedVerification/Numerics (V-01 dt-convergence; V-02 seed reproducibility; V-03 zero-efficacy; V-04 zero-growth; V-05 boundary conditions; V-06 toxicity logic), Validation & Sensitivity (S-01..S-09, covering Emax/EC50/half-life, growth, resistance, dropout, hazard multipliers), UQ (UQ-01..03, sigma variation, age SD, 10-seed ensembles), and Applicability (A-01 population drift, A-03 RECIST threshold, A-04 horizon; A-02 dosing schedule reserved). External validation: Arm A close on ORR and mOS, low on mPFS; Arm G underpredicts mPFS/OS; J/K farther off-acknowledged calibration limitations.

**Model Influence: Medium:** Justification: The twin informs internal prioritization and design (arm ranking, sample size tuning, eligibility mix, sensitivity to rules) but is not the sole basis for regulatory or labeling decisions. Outputs are triangulated with literature comparators and expert judgment. Verification and UQ support reliable computation; partial external fit tempers influence.

**Consequence of Wrong Decision: Medium** Justification: If mis-prioritized, resources could shift to a less active arm and expose Phase II participants to suboptimal regimens; however, care remains within accepted standards under IRB/DSMB oversight, and no patient-facing recommendations are made by the software. Thus, consequences are meaningful for efficacy and development efficiency, but with mitigations for safety.

**Model Risk: Medium** Justification: Combining Medium Model Influence with Medium Consequence yields Medium risk (per M15 and ASME V&V 40 logic). Credibility activities were commensurate: code/calculation verification (dt stability; seed reproducibility; boundary/logic checks), model robustness via sensitivity to key biological/clinical assumptions (S-01..S-09), stochastic/UQ ensembles (UQ-03), and applicability to population mix, RECIST thresholds, and horizon (A-01, A-03, A-04). External validation is partly met for A and G with documented gaps, transparently constraining scope.

**Model Impact: Medium (with a path to High)** Justification: Relative to current regulatory expectations, the twin provides MIDD evidence suitable for planning and interaction: clearly stated Question/COU; risk-informed Model Evaluation; pre-specified Technical Criteria via the VV40 test suite; reproducible code and patient-level logs; ensemble uncertainty bands; and applicability analyses. This supports early FDA interactions (e.g., MIDD/Q-sub) for arm prioritization and design what-if analyses, potentially saving time and cost versus purely empirical iteration. Impact is not rated High because simultaneous multi-endpoint external calibration (A, G, J/K) and prospectively defined quantitative acceptance targets/CIs need strengthening; completion of A-02 dosing applicability and expanded external validation would elevate impact.

<div align="left">

<br>

Kawchak, K. (2025). Accelerating FDA Compliance and Cost Efficiency of in silico Clinical Trials via AI Digital Twin Pancreatic Cancer Simulation. Zenodo. https://doi.org/10.5281/zenodo.17239510
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.17239510-blue)](https://doi.org/10.5281/zenodo.17239510)

---


<div align="center">
  <p>QSP Metastatic Pancreatic Cancer AI Clinical Trial Simulation From Protocol to Prediction: Code, VVUQ, and Playbook</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>August 29, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

**Question:**  Is it plausible for artificial intelligence to generate QSP (quantitative systems pharmacology) pancreatic cancer clinical trial protocols, Python scripts, VVUQ (verification, validation, and uncertainty quantification), and playbook?

**Concepts:** Two prior author studies consisting of drug arms, baseline characteristics, and patient archetypes were incorporated into the initial text trial protocol. The protocol was then further optimized and converted into Python by ChatGPT 5 Pro Research (ChatGPT). Additional trial attributes and mathematical functions were added; followed by hyperparameter optimizations, increased adverse events functionality, and fine-tuning of time steps and tumor grid size primarily by Gemini 2.5 Pro (Gemini). Sensitivity analyses to biological parameters were then conducted; yielding the final model code with optimized parameters and further comparisons to established trials including POLO, NAPOLI-1, and MPACT. The code was then converted back into a plain text protocol by ChatGPT for interpretability regarding non-technical staff, and serving as a platform for future developments.

**Results:** Verifications of several mechanistic variables revealed numerically stable objective response rates (ORRs), with results assisting the finalization of time step dt = 0.05 and tumor volume grid size = 5. Uncertainty quantification of biological sensitive vs. resistant clone parameters also aided in robust ORRs for a combination therapy. Additional sensitivity analysis was performed regarding a KRAS inhibitor whose Emax potency increased by up to threefold, limiting ORR error to 11%. In effect, both numerical stability and biological credibility were demonstrated by the QSP pancreatic cancer simulation. Efforts to ground the control arms to external standards for this model had some success with mOS, ORR%, and DCR% performing more optimally; while mPFS, and Grade 3+ Adverse Events experienced less optimal results based on external validations.

**Outputs:** Initial text based protocols were iteratively refined by ChatGPT based on the author’s prior empirical trial, known requirements for QSP trials, and additional biological incorporations. Code trials were generated based on iterative prompting to optimize parameters and variables. The 10 arm, 7 archetype, 10,000 patient trial with up to 250 ordinary differential equations (ODEs) per patient was aimed towards rapid drug prototyping a Phase II trial in real time. Each trial run yielded a Python script, notebook, and a patient log file. These generated materials were then used throughout the study to build the QSP playbook, VVUQ documentation, and visualization text instructions. The instructions were then converted into Python scripts by Opus 4.1 Extended (Opus), further optimized by the author, and then executed in Google Colab.

**Impacts:** Financial assessments were established between industry QSP virtual trial costs at $2M vs historical Phase II and Phase III trials. The current study was performed by the author at a theoretical cost of $36,304 based on a $150/hr rate at 60 hr/wk for 4 weeks. A QSP cost reduction of up to 99.6% was found; supported by a 23 month time reduction and +27,500% ROI vs. a typical $10.2M Phase II trial. Due to the larger patient cohorts and low resource requirements, the current QSP study was $3.6/patient vs. an in-person trial of $59,500/patient, a 16,528x difference. These financial propositions would most likely be realized by preventing no-go arms from proceeding to in-person trials, such as Arm E; which had a low mOS at 6.6 month, a high Grade 3+ AEs at 92.5%, and second highest Drop % at 14.2%.

**Outcome:** Arm C, an oncogene-targeted therapy, was the most recommended drug combination by AI due to its high ORR at 70.8% and mOS as 11.9 (HR = 0.50). Arms H and I derived from the previous empirical trial were consistently top drug combinations due to competitive response rates and survival. AI models proved pivotal for specialized tasks as ChatGPT was the domain expert that automated text production, Gemini generated trial code solutions at scale, and Opus converted text instructions into effective Python dashboards. Ultimately, as trial code character numbers and complexity rose, models were able to address smaller challenges in the code to obtain useful updates. The numerical stability and sensitivity analyses throughout the VVUQ process can be considered among the study’s most impactful results, while endpoints were more challenging to align with external control arms. This may be due to an estimated 2.5 million tumor volume states being updated at each time-step, adding complexity for further AI modifications.

<div align="left">

<br>

Kawchak, K. (2025). QSP Metastatic Pancreatic Cancer AI Clinical Trial Simulation From Protocol to Prediction: Code, VVUQ, and Playbook. Zenodo. https://doi.org/10.5281/zenodo.17001137
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.17001137-blue)](https://doi.org/10.5281/zenodo.17001137)

---


<div align="center">
  <p>ChatGPT 100,000 Patient 24-Month In Silico Phase III 5-Arm Pancreatic Cancer Clinical Trial Triplicate</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>July 24, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

**Inquiry:**  Is it possible for ChatGPT to simulate three reproducible 100,000 patient pancreatic ductal adenocarcinoma (PDAC) Phase III clinical trial reports? If so, can the results be internally and externally validated, cross-verified using other AI models, and be compared both clinically and financially to other trials?

**Concept:** 5 arms based on the Daraxonrasib + Mitazalimab + liposomal Irinotecan drug combination, baseline characteristics, and patient archetypes were identified from a prior study: doi.org/10.5281/zenodo.15735068. Six artificial intelligence models were then implemented to address the clinical trial pipeline: o3ph: ChatGPT o3-pro Research, g25p: Google Gemini 2.5 Pro, grk4: Grok 4, grk3: Grok 3 Think, o3pr: ChatGPT o3-pro, and ops4: Opus 4 Extended. o3ph generated the ICH E3-aligned trial reports, log files, plus internal, and external validations. g25p, grk4, grk3, o3pr, and ops 4 provided cross verifications that highlighted trial-to-trial and model-to-model correlations. g25p utilized 24 generations in the study to produce a virtual trials overview, while o3ph provided a meta-analysis of pooled and scored data versus relevant virtual and on-site trials. o3ph also provided a financial assessment and value proposition of USD estimates against Phase II and Phase III studies. ops4 provided visualizations written in Python for the majority of the sections.

**Results:** 100,000 individual patients generated from three separate o3ph conversations followed multiplicative hazard ratios and per-arm monthly hazards set in the prompt. Key variables were independent of each other, which yielded distributions of uncensored results. Log file cumulative effects of the censored 100,000 patients yielded expected results in OS by Arm (A > D > E), ≥G3 AE (A > D > E), and PFS (A > D > E). Baseline characteristics by metric across trials were in close alignment, and internal validations between log files and trial reports exhibited similar performance. External validation vs. a Flatiron Health dataset for OS passed, while ECOG validation saw higher differences. These deviations, along with a KRAS-mutant labeling issue were high, but consistent in magnitude across the three trials.

**Outputs:** In order to consolidate trial information, validations, and cross-verifications, g25p processed 24 of these relevant outputs to create a virtual trials overview. The core trial information, technical specifications, reproducibility, and validation findings provided a concise output needed for subsequent comparisons to trials. The method used to pool the current study with prior studies was accomplished by o3ph utilizing the virtual trials overview alongside online clinical trial data to produce a 9,574 word meta-analysis. Results focused on PRODIGE-4 and NAPOLI-1 trials that were top two in OS, while Arm A was third. However, the Arm D doublet of Daraxonrasib + Mitazalimab was less toxic than the other trials, and was found to be more clinically feasible than FOLFIRINOX in PRODIGE-4.

**Impacts:** The financial assessment and value proposition performed by o3ph and visualized by ops4 placed an estimated  price of $36,330 on the current study (1 user at $150/hr working 60 hrs/wk). Estimates for other virtual trials ranged from $120,000-$600,000, while a real Phase II trial was $20.0M, and the Phase III trial estimate was $100.0M. Time-to decision was fastest for the 100K Triplicate at 1 month, while other studies ranged from 4.5 months to 5.0 years. The AI’s main financial decision was that Arm A (Daraxonrasib + Mitazalimab + liposomal Irinotecan) was not a strong enough candidate, and the results from the current study were estimated to save $19.96M to avoid a clinical trial failure. In addition, a $2.36M burn rate reduction was anticipated, with an overall cost reduction of 99.9997% vs. a Phase III trial per patient.

**Outcome:** The main benefit was that reproducibility was observed across a single trial or multiple trials, while individual patients likely varied based on raw exponential sampling. The o3ph feat was primarily in providing a trial report that was replicable between the other trials performed in separate conversations. Similarly, the g25p model’s processing of 24 outputs to create a virtual trials overview could not me accomplished by any other model due to token limitations. The overview served to inform the final meta-analysis and financial assessment by o3ph, providing tangible comparisons and planning tools for upcoming studies. All work was performed by one user in a 30 day window.

<div align="left">

<br>

Kawchak, K. (2025). ChatGPT 100,000 Patient 24-Month In Silico Phase III 5-Arm Pancreatic Cancer Clinical Trial Triplicate. Zenodo. https://doi.org/10.5281/zenodo.16415815
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.16415815-blue)](https://doi.org/10.5281/zenodo.16415815)

---


<div align="center">
  <p>End-to-End Pancreatic Ductal Adenocarcinoma Digital Twin Clinical Trial Proposals</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>June 24, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

**Question:** Can online clinical trial literature be utilized by artificial intelligence to generate well-informed pancreatic ductal adenocarcinoma (PDAC) digital twin trial proposals?

**Findings:** AI successfully generated 40 meta-analyses in PDAC top research areas at an average word length of 10,196. One verification per meta-analysis was performed using AI automation, and the combined 408,081 word dataset was utilized to generate reports. Reports were separately verified and further visualized by AI. A second dataset consisting of the reports was used by five different AI models to yield five proposals. The proposals were evaluated by five AI judges to determine the top three proposals. Proposal deliverable completion reached a maximum score of 9.60/10, while the top citation proficiency score was 9.20. Prospective trial impact reached a peak score of 8.94, while funding probabilities ranged from 8.54 to 8.66.

**Design:** The framework consisted of 7 AI model combinations; o3re: ChatGPT o3 Research, g25p: Google Gemini 2.5 Pro Preview, son4: Sonnet 4 Extended, grk3: xAI Grok 3, o3pr: ChatGPT o3-pro, ops4: Opus 4 Extended, and o3ch: ChatGPT o3; which were utilized according to model specific advantages. o3re autonomously searched the web and constructed detailed meta-analyses for Dataset 1. g25p was used to verify quantitative data across meta-analyses and reports, and also generated reports. son4 and ops4 were used to produce visualizations in Python for reports and proposals; while grk3 fixed code when necessary. The 6 reports were combined to form Dataset 2, which was an input for the o3pr, ops4, g25p, o3ch, and grk3 proposals. The same 5 models served as judges for the combined proposals, with average scores being used to determine the top 3 proposals. AI run time for core experiments was approximately 14 hours, with paper completion in 27 days.

**Results:** One verification for each meta-analysis was conducted by g25p at an average accuracy of 95%. The same model verified one table for each of the six reports at a self-reported 84.0% accuracy. Both verification accuracies are considered preliminary due to complexities in grounding multiple data types. Meta-analyses by o3re ranged from 6,440-13,033 words with an average time of 19.3 minutes. Reports by g25p averaged 2,538 words and 2.48 minutes, with a Pearson's r coefficient of 0.94. Proposals totaled 8,903 words in 22.32 minutes. Proposal analyses by each of the five models were within 1-2 minutes, except for o3pr at 17.77 minutes. Each proposal included 10 prompt specific deliverables and a main therapy recommendation from a report.

**Importance:** An area of of significance for all 5 proposals was the inclusion of a triple drug combination with the highest Therapeutic Synergy & Viability Score (TSVS) detailed in Report 1. Proposal A utilized the combination as follows: "The DT platform will prospectively simulate a first-in-human (FIH) study of the top-ranked therapeutic triplet—Daraxonrasib + Mitazalimab + Liposomal Irinotecan identified in Report 1 (predicted TSVS 8.15)." Proposal B also highlighted the therapy: "simulating 10,000 virtual pancreatic cancer patients to de-risk the clinical development of a novel three-drug combination: daraxonrasib (pan-KRAS inhibitor) + mitazalimab (CD40 agonist) + liposomal irinotecan." The drug combination is traceable to source data using meta-analysis MA-23 for Daraxonrasib, MA-15 for Mitazalimab, and a total of 8 Meta-Analyses utilizing Liposomal Irinotecan. This example of data fusion likely indicates functional utility between AI datasets.

**Conclusion:** Enhanced predictive performance and clinical relevance was achieved through a scalable multi-step workflow beginning with meta-analysis and report generations. Source data was utilized effectively across models, yielding AI aware PDAC digital twin trial proposals. Although automated AI verification accuracies were preliminary, reports were translated into top proposals, and screened visualizations were effective in discerning results. Vital trial proposal information obtained from charts revealed important distinctions between the three top proposals across 24 month timelines, multi part ROI analyses, and budgets ranging from $8.5M-$17.8M. Proposal A by o3pr included the most intriguing digital twin Phase I/early-Phase II trial, with high scores in deliverable completion, citation ability, and chances of funding.

<div align="left">

<br>

Kawchak, K. (2025). End-to-End Pancreatic Ductal Adenocarcinoma Digital Twin Clinical Trial Proposals. Zenodo. https://doi.org/10.5281/zenodo.15735068
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.15735068-blue)](https://doi.org/10.5281/zenodo.15735068)

---


<div align="center">
  <p>10 Year Glioblastoma Clinical Trial Meta-Analyses by Autonomous AI at Scale. Survival, HR, AE, and RoB scored in AI Reports and Charts, including Verifications</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>May 29, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">

**Question:**  Can glioblastoma ten year clinical trial survival times, hazard ratios, adverse events, and risk of bias from online publications be effectively pooled, scored, and visualized with insights using artificial intelligence?

**Findings:** Ten glioblastoma therapies from the past 10 years were compared using AI: including EF-14, EORTC 26101, CheckMate 143, and REGOMA clinical trials. Commonly used charts such as forest, funnel, and risk of bias assessment plots were generated based on human prompts. Where patient data was not available, AI utilized alternative plots or approximated individuals’ data. Diagnostic, therapeutic, and biomarker approaches were utilized to develop reports of pooled and scored metrics. Insights were visualized using OS, PFS, and Grade 3+ AE rates alongside AI generated residual heterogeneity, SCCS, and synergy scores.

**Meaning:** Glioblastoma tumors have a fast growth rate, are difficult to treat due to the blood-brain barrier; while only a limited number of effective therapies have been FDA approved, and are therefore often fatal. This study is an application of internet based clinical trial data that yielded rapid text and image based insights of pooled data. For instance, several insights were based on the 695 patient Stupp et al. TTFields EF-14 2017 trial (OS HR = 0.63), which had a low occurrence of high grade adverse events. On the other hand, Regorafenib was also evaluated, which had improved survival performance (OS HR = 0.50), but with 56% Grade 3-4 toxicity. The core AI experiments presented here took approximately 5.4 total hours to complete, with human verifications and paper completion within 34 days.

**Design:** Internet searches of top 10 glioblastoma 2015-2025 clinical trial areas utilized in standalone PRISMA-2020 aligned meta-analyses (MAs) were created by ChatGPT o3 Deep research (o3dr). For each meta-analysis, 10 charts were generated in Python by 3.7 Sonnet Extended (37se). All meta-analyses were combined into a 103,905 word dataset and processed into reports by Gemini 2.5 Pro (g25p). For each report, 37se generated 10 charts, as well as 11 charts for a 3 report combination. Grok 3 (grk3) provided efficient fixes to Python code if charts contained errors. Four standards served as templates across 27 prompt variants. Excerpts of meta-analysis and report text are included due to size, with full outputs accessible through supplementary files. Note: This study is intended for educational purposes only.

**Results:** In this work, glioblastoma clinical trials such as KEYNOTE-028 and DC Vaccine were summarized into meta-analyses by AI, with best responses rates 81% of the time. Insights from ongoing Neoantigen Vaccine trials included combining personalized vaccines with checkpoint inhibitors to "prevent the exhaustion of vaccine-induced T-cells, potentially leading to deeper tumor control." AI visualized clinical trial metrics in charts with best/approximation/error responses of 30%/50%/20%. Approximations were made primarily due to lack of data, and an error was recorded if at least one mistake was observed. Report creation based on the 103,905 word dataset yielded best responses of 90%, while best quality charts of pooled or scored endpoints occurred 55% of the time, with 25% being approximated. This work represented effective text generations, while images required screening due to data and chart type complexities.

**Importance:** Comparing several clinical trial endpoints using grade systems, and additional intuition across four AI software manufacturers was possible using high contextual awareness and tools that were not accessible prior to early 2025. Stupp et al.’s tumor-treating fields is a widely accepted study for glioblastoma, and their results were represented consistently in multimedia throughout the study. Other treatments such as Neoantigen Vaccine and DNX-2401 + Pembro were visualized effectively with similar OS HR and SCCS metrics, and smaller patient cohorts aided by a network graph.

**Conclusion:** Glioblastoma 10 year clinical trial survival times, hazard ratios, and other endpoints from online publications were effectively pooled and visualized with charts across ten disease areas using AI. Additional conclusions were made combining multiple MAs with scores, as TTFields emerged "as a consistently beneficial therapy (positive SCCS, high IRBS in MGMT+), while checkpoint inhibitors (Nivolumab, Pembrolizumab) show limited efficacy (low/negative SCCS, low IRBS)." The study was made feasible by utilizing unique advantages from four different AI software manufacturers. o3dr offered ten best-in-class autonomous web search and meta-analysis generations. 37se chart generation performance featuring endpoints and scores was unmatched by other software platforms. g25p was the only model capable at a reasonable cost of processing the large meta-analysis dataset to create key insights and tables with sample calculations. grk3 proved to be fast and consistent in correcting chart errors when necessary.



<div align="left">

<br>

Kawchak, K. (2025). 10 Year Glioblastoma Clinical Trial Meta-Analyses by Autonomous AI at Scale. Survival, HR, AE, and RoB scored in AI Reports and Charts, including Verifications. Zenodo. https://doi.org/10.5281/zenodo.15549830
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.15549831-blue)](https://doi.org/10.5281/zenodo.15549831)

---


<div align="center">
  <p>AI revolution toward the cure of lung adenocarcinoma</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>April 24, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Many advancements have been achieved by clinical researchers to progress the cure of lung cancer using artificial intelligence. More recently, Large Language Models (LLMs) have furthered these efforts with clinical decision support systems, however a more comprehensive method was required to improve the cure using multiple disciplines. In this seminal study, a Multi-LLM system was utilized to locate literature, create summaries, and provide in-depth charts. In specific, OpenAI ChatGPT 4.5 Deep research autonomously searched relevant articles from PubMed Central, MDPI, PLOS One, and other journals to provide lung adenocarcinoma summaries across 10 topics at an average of 3,200 words. Each of the summaries were visualized with charts using several Claude 3.7 Sonnet Extended code generations featuring clinical trial data. The 10 summaries were combined into a 32,000 word dataset which was inputted into 3.7 Sonnet Extended to produce 5 detailed reports, each requesting a cure to lung adenocarcinoma from different perspectives. Visualizations were obtained using Python generations of Kaplan-Meier survival curves, forest plots, and violin plots. Additionally, AI experiment runtimes were completed in 3.1 hours, accompanied by an unprecedented number of manual cross-study validations aimed towards revealing LLM transparency, reproducibility, and bias. The significance of the study is that the cure of lung adenocarcinoma was advanced with a workflow that included autonomous generations which were backed by comprehensive charts.


<div align="left">

<br>

Kawchak, K. (2025). AI revolution toward the cure of lung adenocarcinoma. Zenodo. https://doi.org/10.5281/zenodo.15278152
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.15278152-blue)](https://doi.org/10.5281/zenodo.15278152)

---


<div align="center">
  <p>Autonomous LLM Agent and scalable Reasoning LLM for generating cancer drug industry cost solutions</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>March 23, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Every once in a while, a new artificial intelligence technology is released that significantly improves research utility and results. This ’deep research’ tool released by OpenAI in February 2025 is a Large Language Model (LLM) web agent that autonomously queries sites such as PubMed Central (PMC), and generates high quality summaries with verifiable citations. A February 2025 article by Haman M. et al. reported deep research advantages in "analyzing 37 sources-35 of which were found on the PubMed website" using the OpenAI o3 model. Here, ChatGPT 4.5 Deep research summaries regarding five pharmaceutical industry financial categories were found to be 100% in-context with PMC articles, and averaged 1,400 words in 10 minutes with minor issues. Also impressive was the processing of these summaries by the Claude 3.7 Sonnet Extended reasoning model to produce a structured 1,900 word 37 citation report containing detailed economic solutions, which were supported by 6 paragraphs of key insights collaborating multiple author quotations in approx. one minute. In addition, the Claude model produced eleven professional images based on USD or ROI trends, anomalies, and forecasts in three Python scripts. The Claude model possessed an output length that scaled by 3.2x for the report and 6x for code generations vs. the manufacturer’s previous model, was 100% in-context with source data, and included interpretable reasoning summaries. The outputs from ChatGPT 4.5 Deep research served as inputs to 3.7 Sonnet Extended in mitigating model bias amplification that can occur when using results within a single software manufacturer. For transparency, comprehensive generation traceability analyses were conducted for the five summaries, the financial solutions report, and the eleven Python diagrams.


<div align="left">

<br>

Kawchak, K. (2025). Autonomous LLM Agent and scalable Reasoning LLM for generating cancer drug industry cost solutions. Zenodo. https://doi.org/10.5281/zenodo.15072843
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.15072843-blue)](https://doi.org/10.5281/zenodo.15072843)

---


<div align="center">
  <p>Cost containment of global monoclonal antibody drugs and cancer clinical trials via LLM focused reasoning</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>February 25, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Expenses related to monoclonal antibody drugs worldwide and cancer clinical studies must be reduced in order to increase pharmaceutical industry efficiency. These financial opportunities can be assisted using state-of-the-art Large Language Models (LLMs) for focused report generation and advanced reasoning of solutions and forecasts that are based on authors’ original findings. Here, Claude 3.5 Sonnet utilized 45 articles totaling approximately 357,000 words to effectively generate 45 reports. OpenAI ChatGPT o3-mini processed 15 of the reports to obtain comprehensive monoclonal antibody (mAb) cost solutions and financial forecasts. This included a financial recommendation of mAb biosimilars for a 55.2 percent price per dose decrease vs. a bevacizumab biologic due to Japan financial incentives, as reported by Itoshima H. et al. The 30 additional reports were based on cancer clinical trial cost-effectiveness studies, with ChatGPT o3-mini reasoning to produce tables regarding economic strategies and projections. This included a "Total drug cost avoidance of "$92,662,609" over 10 years" when sponsored clinical trial participation was employed for solid tumors using various mAb therapies, as detailed by Carreras M. et al. 2024. The primary advantages of this comprehensive approach were 1) Efficient report generations by 3.5 Sonnet of nearly 25,000 words in 20 minutes, 2) ChatGPT o3-mini’s linear dependency prompt structure reduced narrative drift with structured outputs in less than 5 minutes, and 3) Ethical AI principles were strengthened: financial data was limited by rigorous prompts, yielding outputs that were highly traceable to source data using either LLM, as opposed to relying on the models’ training data.

<div align="left">

<br>

Kawchak, K. (2025). Cost containment of global monoclonal antibody drugs and cancer clinical trials via LLM focused reasoning. Zenodo. https://doi.org/10.5281/zenodo.14968404
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.14968404-blue)](https://doi.org/10.5281/zenodo.14968404)

---


<div align="center">
  <p>Clinical decision support based on Bevacizumab cancer trials and pushing the limitations of advanced LLMs</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>January 27, 2025</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
An exhaustive study was needed to test the limits of leading Large Language Models (LLMs) using numerous real-world clinical trial outcomes. It was also necessary to provide extensive hallucination studies based on both extracting main points and providing novel AI clinical decision support. Here, 100 Bevacizumab cancer therapy articles representing over 900K words were summarized by the 3.5 Sonnet model into 49K words, which completed a detailed and complex problem of several cancer and study types to press the capabilities of the ChatGPT o1 reasoning model. Report summaries in general followed an effective format, with guardrails to de-identify patient information and numerical data sources attributions to ground the output to the input. The main takeaway was that both LLMs typically remained in-context with the input data when more structured prompts were used, while precise quotations and author name citations were less prominent. These errors were likely due to LLM pressure towards achieving coherence vs. exact recall based on manufacturer inference-time compute settings. Overall, ChatGPT o1 provided state-of-the-art evidence-based Bevacizumab insights regarding clinical efficacy across indications, dosing recommendations, combination therapies, and biomarker-driven selections.

<div align="left">

<br>

Kawchak, K. (2025). Clinical decision support based on Bevacizumab cancer trials and pushing the limitations of advanced LLMs. Zenodo. https://doi.org/10.5281/zenodo.14968162
[![DOI](https://img.shields.io/badge/DOI-10.5281%2Fzenodo.14968162-blue)](https://doi.org/10.5281/zenodo.14968162)

---


<div align="center">
  <p>Cancer vs. Conversational Artificial Intelligence</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>December 23, 2024</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Solving cancer mechanisms is challenging due to the complexity of the disease integrated with many approaches that researchers take. In this study, information retrieval was performed on 40 oncological papers to obtain authors' methods regarding the tumor immune microenvironment (TIME) or organ-specific research. 20 TIME summaries were combined and analyzed to yield valuable insights regarding how research based papers compliment information from review papers using Large Language Model (LLM) in-context comparisons, followed by code generation to illustrate each of the authors' methods in a knowledge graph. Next, the 20 combined organ-specific emerging papers impacting historical papers was obtained to serve as a source of data to update a mechanism by Zhang, Y., et al., which was further translated into code by the LLM. The new signaling pathway incorporated four additional authors' area of cancer research followed by the benefit they could have on the original Zhang, Y., et al. pathway. The 40 papers in the study represented over 600,000 words which were focused to specific areas totaling approximately 17,000 words represented by detailed and reproducible reports by Clau-3Opus. ChatGPT o1 provided advanced reasoning based on these authors' methods with extensive correlations and citations. Python or LaTeX code generated by ChatGPT o1 added methods to visualize Conversational AI findings to better understand the intricate nature of cancer research.

<div align="left">

<br>

Kawchak, K. (2024). Cancer vs. Conversational Artificial Intelligence. bioRxiv. https://doi.org/10.1101/2024.12.28.630597
[![DOI](https://img.shields.io/badge/DOI-10.1101/2024.12.28.630597-blue)](https://doi.org/10.1101/2024.12.28.630597)

---


<div align="center">
  <p>mAb Bioprocess Engineering In-Context Table Forecasts using Conversational AI Literature Insight Generations</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>November 27, 2024</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Bioprocess engineering has incorporated effective AI applications in recent years that consist of traditional approaches to training models on relevant data to then analyze and predict new and unseen data. The missing component has been the ability to process mixed data from an assortment of dissimilar information sources with high contextual awareness to inform the Human-AI team on how LLMs and other authors' methods will further improve performance. Here, real-time web search or document retrieval methods with a max speed multiplier of over 600x by 3.5 Sonnet were obtained vs. the manuscript author regarding monoclonal antibody (mAb) bioprocess engineering kinetics across several models. ChatGPT-4o with an average score of 9/10 was the leader in quality for this task with several detailed reports that were obtained using document search addressing a paper's specific weaknesses being improved with LLMs or two other author's methods. This protocol was applied systematically for each of the other two papers, supported by the other two relevant bioprocess papers. o1-preview's advanced reasoning set a new standard over five other models in processing either 136 extracellular or 101 intracellular metabolite tables, incorporating the analysis of 12 additional paper summaries across two prompts with two table revisions. For extracellular metabolites, o1-preview generated an 18 metabolite table including all metabolite forecasts that were expected to be breakthroughs due to future integration of a LLM or other author's recent methods. The model supported its forecasts with interpretable author citations and quotations for breakthrough metabolites, along with lists of author specific and metabolite specific insights that influenced its conclusions. For intracellular metabolites, o1-preview provided a full 101 metabolite table, matching the number of entries from the original Sukwattananipaat, P., et al. table, including confirmations for each metabolite regarding whether each forecasted value was expected to be a breakthrough. Overall, this work was represented by numerous speed advantages, literature insights to address paper weaknesses, and competent o1-preview in-context table analysis with supporting evidence from leading articles represented by two 9.5/10 scores to lead the first conversational AI mAb bioprocess engineering revolution.

<div align="left">

<br>

Kawchak, K. (2024). mAb Bioprocess Engineering In-Context Table Forecasts using Conversational AI Literature Insight Generations. ChemRxiv. https://doi.org/10.26434/chemrxiv-2024-jzbj0
[![DOI](https://img.shields.io/badge/DOI-10.26434%2Fchemrxiv--2024--jzbj0-blue)](https://doi.org/10.26434/chemrxiv-2024-jzbj0)

---


<div align="center">
  <p>Monoclonal Antibody Bioprocess Engineering Advancements Using Conversational Artificial Intelligence</p>
<div align="center">

<div align="center">
  <p>October 27, 2024</p>
  <p>Kevin Kawchak</p>
  <p>CEO ChemicalQDevice</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Processing high dimensional and complex monoclonal antibody (mAb) bioprocess data in industry is now more efficient due to conversational AI. The human in the loop approach to Large Language Model (LLM) inferencing with document retrieval and chained outputs is a probable benefit to existing biotechnology workflows. Potential risks of using natural language processing are minimized due to the utility of solving problems with vast amounts of structured and unstructured mixed data that can be verified by the Human-AI team. This novel work demonstrates o1-preview, ChatGPT-4o, L3.1-405B, and 3.5 Sonnet models’ fast and stateof-the-art solutions. In specific, o1-preview provided a response to 16 papers 110x faster than the manuscript author’s time after the number of words were set equal. In addition, ChatGPT-4o was 371x faster than an optimal human researcher to examine and provide an estimate regarding dimension reduction or combinatorial optimization for a recent paper by Kao, M., et al. The third LLM speed advantage of 336x by ChatGPT-4o vs. the manuscript author was achieved using monte carlo simulations and markov chain models performance forecasts and a current paper by Konoike, F., et al.

Part A featured the individual analysis of 5 recent mAb production papers, which emphasized the proficiency of o1-preview (9.9/10.0), ChatGPT-4o (9.2), and L3.1-405B (9.2) providing a forecast report. Example generations for o1-preview and L3.1-405B typically established connections between using dimension reduction or combinatorial optimization and improving bioprocesses. Part B models generated tables regarding how LLMs can improve numerical data from 5 different papers using monte carlo simulations or markov chain models. An example from ChatGPT-4o (9.0) was substantially more complete, accurate, and convincing than the table provided 3.5 Sonnet (8.0). Part C utilized the report format from Part A combined with the numerical approach from Part B across 6 additional papers, led by o1-preview (9.0) and ChatGPT-4o (8.5). The o1-preview example followed the prompt format well, citing cases of how LLMs will utilize reinforcement learning and bayesian optimization to improve mAb production. The work represents a standard for utilizing a considerable amount of bioprocess data to forecast new results, with the transition into LLMs providing near-real-time production data analysis aided by document retrieval to provide a synergistic effect with existing machine learning techniques.

<div align="left">

<br>

Kawchak, K. (2024). Monoclonal Antibody Bioprocess Engineering Advancements Using Conversational Artificial Intelligence. ChemRxiv. https://doi.org/10.26434/chemrxiv-2024-3m7m1
[![DOI](https://img.shields.io/badge/DOI-10.26434%2Fchemrxiv--2024--3m7m1-blue)](https://doi.org/10.26434/chemrxiv-2024-3m7m1)

---


<div align="center">
  <p>Paclitaxel Biosynthesis AI Breakthrough</p>
<div align="center">

<div align="center">
  <p>October 3, 2024</p>
  <p>Kevin Kawchak</p>
  <p>CEO ChemicalQDevice</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Paclitaxel, C<sub>47</sub>H<sub>51</sub>NO<sub>14</sub>, biosynthesis is an active area of research due to ongoing progress towards more sustainable and environmentally friendly production of the drug compound. Recent literature details the characterization of enzymes that play a role in synthesis, optimization of growth media, and RNA related regulatory mechanisms. The method of PhD students spending excessive time performing literature reviews to discover new findings is obsolete due to faster and high quality state of the art conversational AI. In this study, approximate AI times were obtained regarding how long would it take the fastest human researcher to read, analyze, extract information, and type a high quality 250 word answer; with the fastest time of 1,380 seconds being used as a standard reference. The slowest AI generation in the study was 79.19s by ChatGPT-4o, which was still over 17x faster than the optimal human performance time. Here, a paclitaxel biosynthesis breakthrough was illustrated twice using LLMs and LMMs. In the first instance, full length papers were summarized by AI models – with the finding that AI provided more detailed answers across entire papers, generating over 10x longer descriptions and 12x faster times compared to the manuscript author’s methods to summarize abstracts.

The outputs of individual AI generated answers yielded a 10 Paper Summary with 6,322 words, and served as the input for eight separate prompts, which provided valuable insight regarding both emerging and historical views of paclitaxel retrobiosynthesis, engineering microorganisms, as well as top 10 new research recommendations, and top 10 challenges for this area. The second paclitaxel biosynthesis advancement was demonstrated with a speed of 752 seconds for 36 generations compared to the single optimal human response of 1,380 seconds. Top models received an average AI judge score of 9.5 by ChatGPT-4o for Part A; a score of 9.3 by o1-preview, L3.1-405B, and ChatGPT-4o for Part B; and a score of 9.3 by ChatGPT-4o and 3.5 Sonnet, followed by a score of 9.2 for Wiz8x22B for Part C. These superior results have primarily been afforded by OpenAI, Claude.ai, and Meta AI new model releases in late 2024 that have helped to advance the paclitaxel biosynthesis field. The presence of speedups with more detailed answers over optimal human responses is supported by advanced cloud hardware that processes high dimensional and complex data continually to solve combinatorial problems such as those in this study using 15 different prompts across 163 generations.

<div align="left">

<br>

Kawchak, K. (2024). Paclitaxel Biosynthesis AI Breakthrough. ChemRxiv. https://doi.org/10.26434/chemrxiv-2024-pqjd3
[![DOI](https://img.shields.io/badge/DOI-10.26434%2Fchemrxiv--2024--pqjd3-blue)](https://doi.org/10.26434/chemrxiv-2024-pqjd3)

---


<div align="center">
  <p>High Dimensional and Complex Spectrometric Data Analysis of an Organic Compound using Large Multimodal Models and Chained Outputs</p>
<div align="center">

<div align="center">
  <p>September 12, 2024</p>
  <p>Kevin Kawchak</p>
  <p>CEO ChemicalQDevice</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Large Multimodal Models (LMMs) possess the ability to analyze chemical spectra of an organic compound using state of the art conversational AI. These outputs can then be chained together and introduced as a text input for other LLMs or LMMs to predict the compound name. Here, a challenging 15 carbon molecule problem with 13 complex and high dimensional chemical spectra were analyzed as images by unmodified versions of Claude 3.5 Sonnet and OpenAI ChatGPT-4o models. ScholarGPT judged the responses across the 13 spectra with an average score of 9.01/10, and the highest response scores per individual spectra for 3.5 Sonnet or GPT-4o were used as the text-based chain. For Part B, the chain was then combined with two different prompt formats and the molecular formula to 8 different LMMs or LLMs which produced new compound predictions. 3.5 Sonnet had the highest proficiency in utilizing the formula simultaneously with complex data for three identical compound generations across two prompts, but was likely limited by the quality regarding the chain of 13, primarily with data from 6 2D NMR Spectra. 3.5 Sonnet's compound prediction was then further improved in Part C by utilizing manual chained explanations of the spectra by the author to yield what is believed to be the correct structure with stereochemistry to the unknown problem. To the author's best knowledge, this is the first LMM to generate the C15H22O2 drug compound derivative (S)-ibuprofen ethylester using high dimensional data from 13 detailed spectra. The purpose of this study was to utilize cutting edge natural language processing techniques to evaluate an advanced chemical structure consisting of IR, 1H-NMR, 13C-NMR, DEPT-NMR, GCOSY60, GTOCSY, GHMQC, GHMBC, GNOESY, and expanded views of spectra.

<div align="left">

<br>

Kawchak, K. (2024). High Dimensional and Complex Spectrometric Data Analysis of an Organic Compound using Large Multimodal Models and Chained Outputs. ChemRxiv. https://doi.org/10.26434/chemrxiv-2024-06gf1
[![DOI](https://img.shields.io/badge/DOI-10.26434%2Fchemrxiv--2024--06gf1-blue)](https://doi.org/10.26434/chemrxiv-2024-06gf1)

---


<div align="center">
  <p>LMM Spectrometric Determination of an Organic Compound</p>
<div align="center">

<div align="center">
  <p>August 26, 2024</p>
  <p>Kevin Kawchak</p>
  <p>CEO ChemicalQDevice</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Many machine learning models used in academia and industry that identify organic compounds typically lack the ability to converse over prompts and results, and also require expertise across a number of steps to obtain answers. The purpose of this study was primarily to gain insight into the advantages of current unmodified state of the art Large Multimodal Models (LMMs) across several prompts containing multiple spectra of varying difficulty to evaluate the impact of training data, reasoning, and speed. These readily available and easy to use software for the identification of an organic compound based on a molecular formula and spectra were found to be reproducible across three similar LMMs. To the author's best knowledge, this marks the first time that three GPT variants were each able to correctly identify the organic compound quinoline using a variety of different spectroscopic images. The results were obtained using a 2-step process consisting of a) Uploading high resolution spectral images, and b) Submitting a text prompt with the images that requested a compound determination. The main findings were that 1) Four LMMs provided rationale step-by-step interpretations of 1H-NMR, 13C-NMR, and 3 DEPT-NMR spectra from Prompt A, 2) Three of these LMMs, led by a GPT-5 preview model, combined these interpretations into the correct chemical structure with Prompt A, and 3) Two of these LMMs achieved a top score of 5/5 for also generating sequential explanations reflecting the order of the provided spectra along with most of the correct spectral and molecular formula explanations.

<div align="left">

<br>

Kawchak, K. (2024). LMM Spectrometric Determination of an Organic Compound. ChemRxiv. https://doi.org/10.26434/chemrxiv-2024-qtnkj
[![DOI](https://img.shields.io/badge/DOI-10.26434%2Fchemrxiv--2024--qtnkj-blue)](https://doi.org/10.26434/chemrxiv-2024-qtnkj)

---


<div align="center">
  <p>LMM Chemical Research with Document Retrieval</p>
<div align="center">

<div align="center">
  <p>Kevin Kawchak</p>
  <p>Chief Executive Officer</p>
  <p>ChemicalQDevice</p>
  <p>San Diego, CA</p>
  <p>August 12, 2024</p>
  <p>kevink@chemicalqdevice.com</p>
  <p><strong>Abstract</strong></p>
</div>

<div align="left">
Chemical research is more effectively progressed using Large Multimodal Models (LMMs) combined with Document Retrieval and recently published literature. The methods described here illustrate significant strides over previously tested Large Language Model (LLM) multi-document workflows for characterization assistance and generating new reactions. Here, 3.5 Sonnet, ScholarGPT, and ChatGPT 4o LMMs processed either 5 images or 5 supplementary documents from leading 2024 journals. Each of the three models performed inference on a detailed prompt to produce a response that included context from attachments. In addition, the LMMs were not provided with which of the 5 files contained the answer. The main findings were that 3.5 Sonnet had an average score of 9.8 for images, while two judges awarded high scores to ChatGPT 4o (9.7, 9.4) and ScholarGPT (9.5, 9.4) for document analysis. Judging was performed by a human evaluator for the image uploads, with document processing evaluated by Llama 3.1 405B and Nemotron 4 340B LLMs which correlated well and improved explainability. Highlights include 3.5 Sonnet's ability to interpret a Two-dimensional Nuclear Magnetic Resonance (2D NMR) spectrum accurately, along with Judge Llama 3.1's ability to provide consistent formatted scores with explanations. The results shown here help illustrate AI's continued revitalization of the established chemical research field.

<div align="left">

<br>

Kawchak, K. (2024). LMM Chemical Research with Document Retrieval. ChemRxiv. https://doi.org/10.26434/chemrxiv-2024-p91gm
[![DOI](https://img.shields.io/badge/DOI-10.26434%2Fchemrxiv--2024--p91gm-blue)](https://doi.org/10.26434/chemrxiv-2024-p91gm)
"""

print(f"Embedded README characters: {len(README_TEXT):,}")

Embedded README characters: 126,213


## README parser, reviewed topic rules, and graph definitions

All extraction logic, regex vocabulary, reviewed false-positive corrections, stage definitions, and graph selection rules are editable below.

In [3]:
def edge_key(u: str, v: str, directed: bool = False) -> str:
    return f"{u}->{v}" if directed else "|".join(sorted((u, v)))


def parse_readme(raw: str) -> pd.DataFrame:
    """Parse the 53 README publication blocks into oldest-to-newest rows.

    Abstract text stops before either "Table of Contents" or "Contents" so
    section lists do not contaminate similarity and keyword calculations.
    """
    blocks = re.split(r'\n\s*---\s*\n', raw)
    rows = []
    for block in blocks:
        cleaned = html.unescape(re.sub(r'<[^>]+>', '\n', block))
        lines = [re.sub(r'\s+', ' ', line).strip() for line in cleaned.splitlines()]
        lines = [line for line in lines if line]
        if 'Abstract' not in lines:
            continue
        abstract_idx = lines.index('Abstract')
        title = lines[0]
        date = None
        for candidate in lines[1:abstract_idx]:
            try:
                parsed = dateparser.parse(candidate, fuzzy=False)
                if 2020 <= parsed.year <= 2035:
                    date = pd.Timestamp(parsed.date())
                    break
            except Exception:
                continue
        abstract_lines = []
        for line in lines[abstract_idx + 1:]:
            if (line in {'Table of Contents', 'Contents'}
                    or line.startswith('Kawchak, K.')
                    or line.startswith('[![DOI]')
                    or line.startswith('Disclaimer:')):
                break
            abstract_lines.append(line)
        abstract = ' '.join(abstract_lines).strip()
        doi_match = re.search(r'https://doi\.org/([^\s\)]+)', block)
        if date is not None and len(abstract) >= 50:
            rows.append({
                'title': title,
                'date': date,
                'year': date.year,
                'abstract': abstract,
                'doi': doi_match.group(1) if doi_match else '',
            })
    df = pd.DataFrame(rows).sort_values(['date', 'title']).reset_index(drop=True)
    df['id'] = [f'P{i:02d}' for i in range(1, len(df) + 1)]
    return df


# Explicit README vocabulary. These are intentionally editable.
CONCEPT_PATTERNS = OrderedDict({
    'LLM / conversational AI': r'\b(?:LLM|LLMs|large language model|conversational artificial intelligence|conversational AI|reasoning model)\b',
    'LMM / multimodal AI': r'\b(?:LMM|LMMs|large multimodal model|multimodal model|multimodal AI)\b',
    'Agentic / code generation': r'\b(?:agent|agentic|autonomous AI|code generation|coding agent|repository based|repository-level|single prompt|sub-prompt|FastAPI|GitHub)\b',
    'Document retrieval / RAG': r'\b(?:document retrieval|retrieval|RAG|repository|literature review|paper generation|document guidance|documentation package)\b',
    'Machine learning / optimization': r'\b(?:machine learning|ML\b|bayesian optimization|reinforcement learning|combinatorial optimization|dimension reduction|Monte Carlo|Markov chain)\b',
    'Simulation / digital twin': r'\b(?:simulation|in silico|digital twin|QSP|quantitative systems pharmacology|synthetic patient)\b',
    'Physical AI / robotics': r'\b(?:Physical AI|robot|robotic|robotics|humanoid|pancreaticoduodenectomy|Whipple|surgical)\b',
    'VVUQ / assurance': r'\b(?:VVUQ|verification|validation|uncertainty quantification|assurance|ten-gate|gate suite)\b',
    'Federated learning': r'\bfederated learning\b',
    'MCP / API infrastructure': r'\b(?:MCP|Model Context Protocol|server|API|FastAPI|platform)\b',
    'Clinical trials': r'\b(?:clinical trial|clinical trials|Phase I|Phase 1|Phase II|Phase 2|Phase III|Phase 3|trial protocol|trial site|trial sponsor)\b',
    'PDAC / pancreatic cancer': r'\b(?:PDAC|pancreatic ductal adenocarcinoma|pancreatic cancer|pancreatic)\b',
    'Glioblastoma': r'\b(?:glioblastoma|GBM)\b',
    'Lung adenocarcinoma': r'\blung adenocarcinoma\b',
    'mAb / bioprocess': r'\b(?:mAb|monoclonal antibod|bioprocess|bevacizumab|antibody production)\b',
    'Chemistry / spectroscopy': r'\b(?:spectr|NMR|organic compound|chemical structure|paclitaxel biosynthesis|biosynthesis|molecular formula|chemistry|IR\b)\b',
    'Regulatory / compliance': r'\b(?:FDA|IND|IDE|ICH|GCP|good clinical practice|21 CFR|regulatory|compliance|IRB|NDA|BLA)\b',
    'Patient safety / adverse events': r'\b(?:patient safety|adverse event|adverse events|harm|risk-benefit|risk benefit|clinical hold|safety)\b',
    'Patient matching / decision support': r'\b(?:patient matching|clinical decision support|decision support|patient prediction|eligibility)\b',
    'Cost / efficiency': r'\b(?:cost|costs|economic|economics|efficien|speed|faster|time|throughput|containment|funding)\b',
    'Legislation / policy': r'\b(?:bill|Congress|Federal law|legislation|legislative|Act of 2026|H\. ?R\.|policy|statute)\b',
    'Drug compounds / discovery / synergy': r'\b(?:drug synergy|drug discovery|daraxonrasib|RMC-6236|paclitaxel|drug compound|treatment arm)\b',
})

CONCEPT_GROUP = {
    'LLM / conversational AI': 'AI & computation', 'LMM / multimodal AI': 'AI & computation',
    'Agentic / code generation': 'AI & computation', 'Document retrieval / RAG': 'AI & computation',
    'Machine learning / optimization': 'AI & computation', 'Simulation / digital twin': 'AI & computation',
    'Physical AI / robotics': 'AI & computation', 'VVUQ / assurance': 'AI & computation',
    'Federated learning': 'AI & computation', 'MCP / API infrastructure': 'AI & computation',
    'Clinical trials': 'Clinical & biomedical', 'PDAC / pancreatic cancer': 'Clinical & biomedical',
    'Glioblastoma': 'Clinical & biomedical', 'Lung adenocarcinoma': 'Clinical & biomedical',
    'mAb / bioprocess': 'Clinical & biomedical', 'Chemistry / spectroscopy': 'Clinical & biomedical',
    'Drug compounds / discovery / synergy': 'Clinical & biomedical',
    'Regulatory / compliance': 'Translation & governance',
    'Patient safety / adverse events': 'Translation & governance',
    'Patient matching / decision support': 'Translation & governance',
    'Cost / efficiency': 'Translation & governance', 'Legislation / policy': 'Translation & governance',
}

# Reviewed false-positive corrections. Edit these alongside the regexes.
# P04 uses "regulatory mechanisms" biologically, not regulatory compliance.
# P06 says a generated table had a "matching" row count, not patient matching.
# P07 uses TIME as "tumor immune microenvironment", not elapsed time/efficiency.
# P16 uses a simulated treatment "policy", not legislation or public policy.
CONCEPT_OVERRIDES = {
    'P04': {'Regulatory / compliance': False},
    'P06': {'Patient matching / decision support': False},
    'P07': {'Cost / efficiency': False},
    'P16': {'Legislation / policy': False},
}

# Word-bounded stage vocabulary. The previous unbounded IND/IDE/ICH patterns could
# match ordinary words such as "findings", "provide", and "which"; these cannot.
STAGE_PATTERNS = OrderedDict({
    'Discovery & evidence synthesis': [
        r'\bliterature\b', r'\bdocument retrieval\b', r'\bevidence synthesis\b',
        r'\bmeta-analys(?:is|es)\b', r'\bspectr\w*\b', r'\bbiosynthesis\b',
        r'\b(?:organic |drug )?compounds?\b', r'\bdrug synergy\b',
        r'\bbioprocess\w*\b', r'\bforecast\w*\b', r'\bpaper generation\b', r'\breview\b',
    ],
    'Preclinical / in silico modeling': [
        r'\bsimulat(?:ion|ions|ed|ing)\b', r'\bin silico\b', r'\bdigital twins?\b',
        r'\bQSP\b', r'\bquantitative systems pharmacology\b',
        r'\b(?:patient|clinical|outcome|survival|toxicity) prediction\b',
        r'\bMonte Carlo\b', r'\bMarkov chain\b', r'\bsynthetic patients?\b',
    ],
    'Trial design & protocols': [
        r'\bprotocols?\b', r'\bPhase\s*(?:I{1,3}|[123])\b', r'\bendpoints?\b',
        r'\bcohorts?\b', r'\brandomi[sz]ed\b', r'\btrial design\b', r'\bstudy design\b',
        r'\bdose escalation\b', r'\bschedule of activities\b',
    ],
    'Regulatory & ethics': [
        r'\bFDA\b', r'\bIND\b', r'\bIDE\b', r'\bICH\b', r'\bIRB\b', r'\b21\s*CFR\b',
        r'\bcompliance\b', r'\bgood clinical practice\b', r'\bclinical hold\b',
        r'\binformed consent\b', r'\bethics?\b', r'\bNDA\b', r'\bBLA\b',
        r'\bregulatory (?:material|framework|requirements?|submission|pathway|review|systems?|guidance|compliance|governance|process|package|clocks?|documents?|approval|protections?|implementation|trial)\b',
    ],
    'Site / sponsor infrastructure': [
        r'\bplatforms?\b', r'\bMCP\b', r'\bservers?\b', r'\bAPIs?\b', r'\bFastAPI\b',
        r'\btrial sites?\b', r'\bsponsors?\b', r'\bfederated(?: learning)?\b',
        r'\brepositor(?:y|ies)\b', r'\bGitHub\b', r'\bdocumentation package\b', r'\bon-premises\b',
    ],
    'Patient operations & safety': [
        r'\bpatients?\b', r'\badverse events?\b', r'\bsafety\b', r'\bharms?\b',
        r'\bpatient matching\b', r'\beligib(?:ility|le)\b', r'\bclinical decision support\b',
        r'\bworkflows?\b', r'\brisk[- ]benefit\b', r'\bclinical hold\b', r'\btoxicity\b',
        r'\bpatient priorit(?:y|ization)\b',
    ],
    'Verification & assurance': [
        r'\bVVUQ\b', r'\bverification\b', r'\bvalidation\b',
        r'\buncertainty quantification\b', r'\bassurance\b', r'\bten-gate\b',
        r'\bgate suite\b', r'\baudit(?:able|ed|ing)?\b',
    ],
    'Policy / funding / adoption': [
        r'\bbills?\b', r'\bCongress\b', r'\bFederal law\b', r'\blegislation\b',
        r'\blegislative\b', r'\bAct of 2026\b', r'\bH\.\s*R\.\b', r'\bstatutes?\b',
        r'\bfunding\b', r'\badoption\b', r'\bcosts?\b', r'\btrust\b', r'\bleadership\b',
        r'\bappropriation\b', r'\bfiscal\b',
        r'\b(?:public|federal|state|health|clinical|regulatory) policy\b',
    ],
})


def detect_concepts(df: pd.DataFrame) -> pd.DataFrame:
    presence = pd.DataFrame(index=df.index)
    for concept, pattern in CONCEPT_PATTERNS.items():
        presence[concept] = [
            bool(re.search(pattern, f"{row.title} {row.abstract}", flags=re.I))
            for row in df.itertuples()
        ]
    for publication_id, changes in CONCEPT_OVERRIDES.items():
        row_idx = df.index[df['id'].eq(publication_id)]
        if len(row_idx) != 1:
            raise AssertionError(f'Concept override publication not found: {publication_id}')
        for concept, value in changes.items():
            if concept not in presence.columns:
                raise AssertionError(f'Concept override name not found: {concept}')
            presence.loc[row_idx[0], concept] = bool(value)
    return presence


def concept_match_audit(df: pd.DataFrame, presence: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for idx, row in df.iterrows():
        text = f"{row['title']} {row['abstract']}"
        for concept, pattern in CONCEPT_PATTERNS.items():
            matches = [m.group(0) for m in re.finditer(pattern, text, flags=re.I)]
            override = CONCEPT_OVERRIDES.get(row['id'], {}).get(concept, None)
            if matches or override is not None:
                rows.append({
                    'publication_id': row['id'], 'concept': concept,
                    'detected': bool(presence.loc[idx, concept]),
                    'regex_matches': ' | '.join(matches[:12]),
                    'manual_override': override,
                })
    return pd.DataFrame(rows)


def stage_scores(df: pd.DataFrame) -> pd.DataFrame:
    scores = pd.DataFrame(0.0, index=df.index, columns=STAGE_PATTERNS)
    for stage, patterns in STAGE_PATTERNS.items():
        for idx, row in df.iterrows():
            title_text = row['title']
            full_text = f"{row['title']} {row['abstract']}"
            hits = sum(bool(re.search(pattern, full_text, re.I)) for pattern in patterns)
            title_hits = sum(bool(re.search(pattern, title_text, re.I)) for pattern in patterns)
            raw = hits + 0.75 * title_hits
            scores.loc[idx, stage] = min(1.0, raw / 4.0) if raw > 0 else 0.0
    return scores


def stage_match_audit(df: pd.DataFrame, scores: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for idx, row in df.iterrows():
        text = f"{row['title']} {row['abstract']}"
        for stage, patterns in STAGE_PATTERNS.items():
            matches = []
            for pattern in patterns:
                match = re.search(pattern, text, re.I)
                if match:
                    matches.append(match.group(0))
            if matches:
                rows.append({
                    'publication_id': row['id'], 'stage': stage,
                    'score': float(scores.loc[idx, stage]),
                    'matched_terms': ' | '.join(matches),
                })
    return pd.DataFrame(rows)


def add_strength_lengths(G: nx.Graph, min_len: float = 0.70, max_len: float = 4.2) -> None:
    strengths = np.array([d['strength'] for _, _, d in G.edges(data=True)], dtype=float)
    if not len(strengths):
        return
    lo, hi = float(strengths.min()), float(strengths.max())
    for _, _, d in G.edges(data=True):
        norm = (d['strength'] - lo) / (hi - lo + 1e-12)
        d['strength_norm'] = float(norm)
        d['target_len'] = float(min_len + (max_len - min_len) * (1.0 - norm) ** 1.65)


def connect_components_by_similarity(G: nx.Graph, candidates: list[tuple[float, str, str]]) -> None:
    while not nx.is_connected(G):
        components = list(nx.connected_components(G))
        comp_index = {node: i for i, component in enumerate(components) for node in component}
        for strength, u, v in sorted(candidates, reverse=True):
            if comp_index[u] != comp_index[v] and not G.has_edge(u, v):
                G.add_edge(u, v, strength=float(strength), bridge=True)
                break
        else:
            raise RuntimeError('Could not connect graph components.')


def build_graphs(df: pd.DataFrame, concept_presence: pd.DataFrame, stages: pd.DataFrame):
    corpus = (df['title'] + '. ' + df['abstract']).tolist()
    vectorizer = TfidfVectorizer(
        stop_words='english', ngram_range=(1, 2), min_df=2, max_df=.90,
        sublinear_tf=True, max_features=6000,
    )
    X = vectorizer.fit_transform(corpus)
    similarity = cosine_similarity(X)
    np.fill_diagonal(similarity, 0.0)

    # Graph 1: undirected union of each publication's two strongest neighbors.
    G1 = nx.Graph(name='Publication abstract similarity')
    for i, row in df.iterrows():
        G1.add_node(row['id'], label=row['id'], title=row['title'], year=row['year'], index=i, kind='publication')
    for i in range(len(df)):
        for j in np.argsort(similarity[i])[::-1][:2]:
            G1.add_edge(df.loc[i, 'id'], df.loc[j, 'id'], strength=float(similarity[i, j]), selection='top-2 neighbor')
    candidates = [
        (float(similarity[i, j]), df.loc[i, 'id'], df.loc[j, 'id'])
        for i in range(len(df)) for j in range(i + 1, len(df))
    ]
    connect_components_by_similarity(G1, candidates)
    add_strength_lengths(G1, .70, 4.4)
    communities = list(nx.community.greedy_modularity_communities(G1, weight='strength'))
    for community_id, members in enumerate(communities):
        for node in members:
            G1.nodes[node]['community'] = community_id

    # Graph 2: theme co-occurrence using Ochiai association.
    concepts = list(concept_presence.columns)
    concept_ids = {name: f'T{i:02d}' for i, name in enumerate(concepts, 1)}
    frequency = concept_presence.sum(axis=0).astype(int)
    G2 = nx.Graph(name='Theme co-occurrence')
    for concept in concepts:
        G2.add_node(
            concept_ids[concept], label=concept_ids[concept], title=concept,
            frequency=int(frequency[concept]), group=CONCEPT_GROUP[concept], kind='theme',
        )
    pairs = []
    for i, c1 in enumerate(concepts):
        for c2 in concepts[i + 1:]:
            co = int((concept_presence[c1] & concept_presence[c2]).sum())
            if co:
                association = co / math.sqrt(max(1, int(frequency[c1])) * max(1, int(frequency[c2])))
                pairs.append((association, co, concept_ids[c1], concept_ids[c2]))
    selected = set()
    for node in G2:
        for association, co, u, v in sorted([x for x in pairs if node in x[2:]], reverse=True)[:4]:
            selected.add(tuple(sorted((u, v))))
    lookup2 = {tuple(sorted((u, v))): (association, co) for association, co, u, v in pairs}
    for u, v in selected:
        association, co = lookup2[(u, v)]
        if co >= 2 or association >= .35:
            G2.add_edge(u, v, strength=float(association), count=int(co))
    if not nx.is_connected(G2):
        connect_components_by_similarity(G2, [(association, u, v) for association, _, u, v in pairs])
        for u, v, data in G2.edges(data=True):
            if 'count' not in data:
                association, co = lookup2[tuple(sorted((u, v)))]
                data.update(strength=float(association), count=int(co), bridge=True)
    add_strength_lengths(G2, .85, 4.5)

    # Graph 3: stage-theme associations supported by at least two publications.
    stage_ids = {stage: f'S{i:02d}' for i, stage in enumerate(STAGE_PATTERNS, 1)}
    stage_totals = stages.sum(axis=0)
    concept_totals = concept_presence.sum(axis=0)
    stage_edges = []
    for stage in STAGE_PATTERNS:
        for concept in concepts:
            mask = (stages[stage] > 0) & concept_presence[concept]
            count = int(mask.sum())
            if count < 2:
                continue
            evidence = float(stages.loc[mask, stage].sum())
            association = evidence / math.sqrt(
                max(1e-9, float(stage_totals[stage])) * max(1.0, float(concept_totals[concept]))
            )
            stage_edges.append((association, count, stage_ids[stage], concept_ids[concept]))
    selected3 = set()
    for stage_id in stage_ids.values():
        for association, count, u, v in sorted([x for x in stage_edges if x[2] == stage_id], reverse=True)[:6]:
            selected3.add((u, v))
    # Keep each theme's strongest association when supported by at least two papers.
    for concept_id in concept_ids.values():
        incident = sorted([x for x in stage_edges if x[3] == concept_id], reverse=True)
        if incident:
            _, _, u, v = incident[0]
            selected3.add((u, v))
    included_theme_ids = {v for _, v in selected3}
    G3 = nx.Graph(name='Pipeline-theme association')
    for stage, stage_id in stage_ids.items():
        G3.add_node(stage_id, label=stage_id, title=stage, kind='stage', group='Pipeline stage')
    for concept, concept_id in concept_ids.items():
        if concept_id in included_theme_ids:
            G3.add_node(concept_id, label=concept_id, title=concept, kind='theme', group=CONCEPT_GROUP[concept])
    lookup3 = {(u, v): (association, count) for association, count, u, v in stage_edges}
    for u, v in selected3:
        association, count = lookup3[(u, v)]
        G3.add_edge(u, v, strength=float(association), count=int(count))
    if not nx.is_connected(G3):
        omitted = [(a, u, v) for a, _, u, v in stage_edges if (u, v) not in selected3]
        connect_components_by_similarity(G3, omitted)
        for u, v, data in G3.edges(data=True):
            if 'count' not in data:
                association, count = lookup3[(u, v)]
                data.update(strength=float(association), count=int(count), bridge=True)
    add_strength_lengths(G3, .80, 4.2)

    # Graph 4: earlier-to-later textual precursors. This is not a causal lineage claim.
    G4 = nx.DiGraph(name='Temporal text-similarity precursors')
    for i, row in df.iterrows():
        G4.add_node(row['id'], label=row['id'], title=row['title'], year=row['year'], index=i, kind='publication')
    for i in range(1, len(df)):
        earlier = similarity[i, :i]
        best = int(np.argmax(earlier))
        G4.add_edge(df.loc[best, 'id'], df.loc[i, 'id'], strength=float(earlier[best]), rank=1)
        if i >= 3:
            second = int(np.argsort(earlier)[-2])
            if earlier[second] >= max(.10, .80 * earlier[best]):
                G4.add_edge(df.loc[second, 'id'], df.loc[i, 'id'], strength=float(earlier[second]), rank=2)
    add_strength_lengths(G4, .70, 4.4)
    return (G1, G2, G3, G4), similarity, concept_ids, stage_ids, vectorizer, X


def degree_radius(degree: float, min_degree: float, max_degree: float,
                  min_radius: float, max_radius: float) -> float:
    """Square-root scaling used by all default node radii."""
    if max_degree <= min_degree:
        return float((min_radius + max_radius) / 2.0)
    numerator = math.sqrt(degree) - math.sqrt(min_degree)
    denominator = math.sqrt(max_degree) - math.sqrt(min_degree)
    return float(min_radius + (max_radius - min_radius) * numerator / denominator)


## Build and source audit

This cell verifies source completeness, creates all graph data, and stores detailed concept, stage, publication, and edge audit tables as editable variables.

In [4]:
canonical_readme = '\n'.join(line.rstrip() for line in README_TEXT.splitlines()) + ('\n' if README_TEXT.endswith('\n') else '')
embedded_sha256 = hashlib.sha256(canonical_readme.encode('utf-8')).hexdigest()
if embedded_sha256 != EXPECTED_README_SHA256:
    raise AssertionError(f'Embedded README checksum changed: {embedded_sha256}')
if README_TEXT.count('<strong>Abstract</strong>') != 53:
    raise AssertionError('The embedded README does not contain exactly 53 abstract markers.')

df = parse_readme(README_TEXT)
if len(df) != 53:
    raise AssertionError(f'Expected 53 embedded abstracts; parsed {len(df)}.')
if df['title'].duplicated().any() or df['doi'].duplicated().any():
    raise AssertionError('Duplicate publication title or DOI detected.')
if df['doi'].eq('').any():
    raise AssertionError('At least one publication DOI was not parsed.')
if df['abstract'].str.contains(r'\b(?:Table of Contents|Contents)\b', regex=True).any():
    raise AssertionError('A table-of-contents heading leaked into parsed abstract text.')

concept_presence = detect_concepts(df)
stages = stage_scores(df)
concept_audit = concept_match_audit(df, concept_presence)
stage_audit = stage_match_audit(df, stages)
(G1, G2, G3, G4), similarity, concept_ids, stage_ids, vectorizer, X = build_graphs(df, concept_presence, stages)
GRAPHS = {'graph1': G1, 'graph2': G2, 'graph3': G3, 'graph4': G4}

pub_entries = [(r.id, f"{r.date.strftime('%Y-%m-%d')} — {r.title}", '#274c77') for r in df.itertuples()]
theme_entries = [(cid, f"{concept} (n={int(concept_presence[concept].sum())})", '#355070')
                 for concept, cid in concept_ids.items()]
pipeline_entries = [(sid, stage, '#7b2c83') for stage, sid in stage_ids.items()] + [
    entry for entry in theme_entries if entry[0] in G3.nodes
]
LEGEND_ENTRIES = {
    'graph1': pub_entries, 'graph2': theme_entries,
    'graph3': pipeline_entries, 'graph4': pub_entries,
}

COMMUNITY_LABELS = {
    0: 'Early cancer AI / bioprocess', 1: 'Robotic trials / assurance',
    2: 'PDAC protocols / funding', 3: 'Trial platforms / infrastructure',
    4: 'In silico trials / digital twins', 5: 'Federal policy / H.R. 9510',
    6: 'LLM delivery / validation', 7: 'Trial sites / policy',
    8: 'Multimodal / chemistry', 9: 'ICH / CFR / adaptations',
}
community_palette = list(plt.get_cmap('Set3').colors) + list(plt.get_cmap('Pastel1').colors)
group_colors = {
    'AI & computation': '#b9d8f2', 'Clinical & biomedical': '#cde9c9',
    'Translation & governance': '#f4d8b8', 'Pipeline stage': '#e7c6e7',
}
year_colors = {2024: '#d8e8f7', 2025: '#a9d1ee', 2026: '#76b8df'}
COLOR_KEYS = {
    'graph1': [(COMMUNITY_LABELS[c], community_palette[c % len(community_palette)])
               for c in sorted({G1.nodes[n]['community'] for n in G1})],
    'graph2': [(g, group_colors[g]) for g in ['AI & computation', 'Clinical & biomedical', 'Translation & governance']],
    'graph3': [(g, group_colors[g]) for g in ['Pipeline stage', 'AI & computation', 'Clinical & biomedical', 'Translation & governance']],
    'graph4': [(str(y), year_colors[y]) for y in sorted(df.year.unique())],
}

RADIUS_RULES = {
    'graph1': {'min_radius': 48.34537420613657, 'max_radius': 60.75},
    'graph2': {'min_radius': 30.206207261596575, 'max_radius': 45.0},
    'graph3': {'min_radius': 28.333333333333332, 'max_radius': 45.0},
    'graph4': {'min_radius': 60.41190868531085, 'max_radius': 87.1875},
}


def graph_edge_audit(graph_name: str) -> pd.DataFrame:
    G = GRAPHS[graph_name]
    rows = []
    for u, v, data in G.edges(data=True):
        row = {
            'edge': edge_key(u, v, G.is_directed()),
            'source': u, 'source_title': G.nodes[u].get('title', ''),
            'target': v, 'target_title': G.nodes[v].get('title', ''),
            'strength': float(data['strength']),
            'target_length': float(data['target_len']),
            'bridge': bool(data.get('bridge', False)),
        }
        for field in ('count', 'rank', 'selection'):
            if field in data:
                row[field] = data[field]
        rows.append(row)
    return pd.DataFrame(rows).sort_values(['strength', 'edge'], ascending=[False, True]).reset_index(drop=True)


EDGE_AUDIT = {name: graph_edge_audit(name) for name in GRAPHS}
publication_table = df[['id', 'date', 'title', 'abstract', 'doi']].copy()
source_audit = pd.DataFrame({
    'check': ['canonical README SHA-256', 'publication count', 'unique titles', 'unique DOIs', 'TOC text excluded'],
    'value': [embedded_sha256, len(df), df.title.nunique(), df.doi.nunique(), True],
})
print('Source verification passed.')
display(source_audit)
display(publication_table[['id', 'date', 'title', 'doi']])


Source verification passed.


,check,value
0,canonical README SHA-256,8f9acb01cd2f521c3d2d4e209e194d81662ef4e317641c...
1,publication count,53
2,unique titles,53
3,unique DOIs,53
4,TOC text excluded,True


,id,date,title,doi
0,P01,2024-08-12,LMM Chemical Research with Document Retrieval,10.26434/chemrxiv-2024-p91gm
1,P02,2024-08-26,LMM Spectrometric Determination of an Organic ...,10.26434/chemrxiv-2024-qtnkj
2,P03,2024-09-12,High Dimensional and Complex Spectrometric Dat...,10.26434/chemrxiv-2024-06gf1
3,P04,2024-10-03,Paclitaxel Biosynthesis AI Breakthrough,10.26434/chemrxiv-2024-pqjd3
4,P05,2024-10-27,Monoclonal Antibody Bioprocess Engineering Adv...,10.26434/chemrxiv-2024-3m7m1
5,P06,2024-11-27,mAb Bioprocess Engineering In-Context Table Fo...,10.26434/chemrxiv-2024-jzbj0
6,P07,2024-12-23,Cancer vs. Conversational Artificial Intelligence,10.1101/2024.12.28.630597
7,P08,2025-01-27,Clinical decision support based on Bevacizumab...,10.5281/zenodo.14968162
8,P09,2025-02-25,Cost containment of global monoclonal antibody...,10.5281/zenodo.14968404
9,P10,2025-03-23,Autonomous LLM Agent and scalable Reasoning LL...,10.5281/zenodo.15072843


## Publication-quality renderer and validation

Before each graph is drawn, the renderer deterministically optimizes the weighted layout, relaxes node collisions, routes numeric labels through low-conflict curved alternatives, and repositions annotations. It then audits the actual rendered bounding boxes for label, annotation, and legend overlaps. PNG and linked SVG files are exported only when the notebook runs.

In [5]:
# Publication-quality layout, collision-aware label routing, renderer, and validation.
# The optimization is deterministic and runs on a standard Colab CPU.

def label_rect(center, text, fontsize=8.0, x_units_per_pt=1.0, y_units_per_pt=1.0, pad_pt=2.2):
    lines = str(text).split('\n')
    max_chars = max((len(line) for line in lines), default=1)
    width_pt = max(12.0, fontsize * 0.60 * max_chars + 2 * pad_pt)
    height_pt = max(8.0, fontsize * 1.22 * len(lines) + 2 * pad_pt)
    x, y = map(float, center)
    half_w = 0.5 * width_pt * x_units_per_pt
    half_h = 0.5 * height_pt * y_units_per_pt
    return (x - half_w, y - half_h, x + half_w, y + half_h)


def _rect_overlap_area(a, b):
    dx = min(a[2], b[2]) - max(a[0], b[0])
    dy = min(a[3], b[3]) - max(a[1], b[1])
    return max(0.0, dx) * max(0.0, dy)


def _rect_circle_overlap(rect, center, radius):
    cx, cy = center
    qx = min(max(cx, rect[0]), rect[2])
    qy = min(max(cy, rect[1]), rect[3])
    return (qx - cx) ** 2 + (qy - cy) ** 2 < radius ** 2


def _point_segment_distance(point, start, end):
    point = np.asarray(point, dtype=float)
    start = np.asarray(start, dtype=float)
    end = np.asarray(end, dtype=float)
    vec = end - start
    denom = float(np.dot(vec, vec))
    if denom <= 1e-12:
        return float(np.linalg.norm(point - start))
    t = float(np.clip(np.dot(point - start, vec) / denom, 0.0, 1.0))
    return float(np.linalg.norm(point - (start + t * vec)))


def clipped_edge_points(p, q, ru, rv):
    vec = q - p
    distance = float(np.linalg.norm(vec))
    if distance < 1e-12:
        return p, q
    unit = vec / distance
    return p + unit * (ru * 0.98), q - unit * (rv * 0.98)


def _set_url(artist, href):
    try:
        artist.set_url(href)
    except Exception:
        pass
    return artist


def _draw_edge(ax, start, end, control, directed, linew, color, alpha, mutation_scale, href):
    if control is None:
        if directed:
            artist = FancyArrowPatch(start, end, arrowstyle='-|>', mutation_scale=mutation_scale,
                                     linewidth=linew, color=color, alpha=alpha, shrinkA=0, shrinkB=0,
                                     connectionstyle='arc3,rad=0.0', zorder=1, clip_on=False)
            ax.add_patch(_set_url(artist, href))
        else:
            artist, = ax.plot([start[0], end[0]], [start[1], end[1]], linewidth=linew,
                              color=color, alpha=alpha, zorder=1, solid_capstyle='round')
            _set_url(artist, href)
        return

    control = np.asarray(control, dtype=float)
    path = MplPath([start, control, end], [MplPath.MOVETO, MplPath.CURVE3, MplPath.CURVE3])
    if directed:
        artist = FancyArrowPatch(path=path, arrowstyle='-|>', mutation_scale=mutation_scale,
                                 linewidth=linew, color=color, alpha=alpha, zorder=1, clip_on=False)
    else:
        artist = PathPatch(path, fill=False, linewidth=linew, color=color, alpha=alpha,
                           zorder=1, capstyle='round')
    ax.add_patch(_set_url(artist, href))


def _physical_wrap(text, width_chars, max_lines):
    lines = textwrap.wrap(str(text), width=max(8, int(width_chars)), break_long_words=False,
                          break_on_hyphens=False) or ['']
    if len(lines) > max_lines:
        retained = lines[:max_lines]
        retained[-1] = textwrap.shorten(retained[-1] + ' ' + ' '.join(lines[max_lines:]),
                                         width=max(8, int(width_chars)), placeholder='…')
        lines = retained
    return '\n'.join(lines)


def add_color_key(ax, entries, frame, href):
    ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    _set_url(ax.patch, href)
    heading = ax.text(0, 1, frame['color_key_title'], ha='left', va='top',
                      fontsize=frame['legend_heading_fontsize'], fontweight='bold', color='black')
    _set_url(heading, href)
    n = len(entries)
    cols = 2 if n > 5 else 1
    rows = math.ceil(n / cols)
    top, bottom = 0.78, 0.05
    row_h = (top - bottom) / max(rows - 1, 1) if rows > 1 else 0
    col_w = 1 / cols
    for i, (label, color) in enumerate(entries):
        col, row = i // rows, i % rows
        x, y = col * col_w, top - row * row_h
        patch = Rectangle((x, y - 0.018), 0.036, 0.036, facecolor=color,
                          edgecolor='#4b5563', linewidth=0.45)
        ax.add_patch(_set_url(patch, href))
        txt = ax.text(x + 0.048, y, label, ha='left', va='center',
                      fontsize=frame['color_key_fontsize'], color='#111827')
        _set_url(txt, href)


def draw_legend(ax, entries, frame, href):
    ax.axis('off'); ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    _set_url(ax.patch, href)
    heading = ax.text(0, 1, frame['legend_title'], ha='left', va='top',
                      fontsize=frame['legend_heading_fontsize'], fontweight='bold', color='#111827')
    _set_url(heading, href)

    columns = int(frame['legend_columns'])
    n = len(entries)
    rows = math.ceil(n / columns)
    top, bottom = 0.94, 0.015
    row_h = (top - bottom) / max(rows, 1)
    col_w = 1 / columns
    bbox = ax.get_position()
    axis_w_pt = bbox.width * FIGSIZE[0] * 72
    axis_h_pt = bbox.height * FIGSIZE[1] * 72
    fontsize = float(frame['legend_fontsize'])
    col_pt = axis_w_pt / columns
    code_pt = fontsize * 4.2
    gap_pt = 4.5
    width_chars = max(12, int((col_pt - code_pt - gap_pt - 4) / (0.56 * fontsize)))
    row_pt = axis_h_pt * row_h
    max_lines = max(1, int((row_pt - 1.0) / (fontsize * 1.04)))
    entry_artists = []

    for idx, (item_code, text, color) in enumerate(entries):
        col, row = idx // rows, idx % rows
        x, y = col * col_w, top - row * row_h
        wrapped = _physical_wrap(text, width_chars, max_lines)
        text_x = x + (code_pt + gap_pt) / axis_w_pt
        t1 = ax.text(x, y, item_code, ha='left', va='top', fontsize=fontsize,
                     fontweight='bold', color=color, linespacing=1.0)
        t2 = ax.text(text_x, y, wrapped, ha='left', va='top', fontsize=fontsize,
                     color='#111827', linespacing=1.02)
        _set_url(t1, href); _set_url(t2, href)
        entry_artists.append((t1, t2))
    return entry_artists


def patch_svg_links(svg_path: Path):
    text = svg_path.read_text(encoding='utf-8')
    text = re.sub(r'<a xlink:href="(#scrollTo=[^"]+)"', r'<a target="_top" xlink:href="\1"', text)
    svg_path.write_text(text, encoding='utf-8')


def display_clickable_svg(svg_path: Path):
    svg = svg_path.read_text(encoding='utf-8')
    display(HTML('<div style="width:100%; overflow:auto">' + svg + '</div>'))


PUBLICATION_QUALITY_SETTINGS = {
    'graph1': dict(seed=23, floor=0.15, exponent=3.0, k=1.65, span=(4700.0, 4250.0), node_gap=36.0,
                   node_font=8.4, edge_font=6.7, note_font=8.8),
    'graph2': dict(seed=71, floor=0.20, exponent=3.0, k=1.70, span=(3400.0, 2950.0), node_gap=28.0,
                   node_font=8.8, edge_font=6.9, note_font=9.2),
    'graph3': dict(seed=47, floor=0.30, exponent=3.0, k=1.70, span=(3500.0, 3050.0), node_gap=28.0,
                   node_font=8.8, edge_font=7.0, note_font=9.2),
    'graph4': dict(seed=11, floor=0.15, exponent=3.0, k=1.65, span=(4700.0, 4250.0), node_gap=42.0,
                   node_font=8.4, edge_font=6.7, note_font=8.8),
}
_OPTIMIZED_GRAPHS = set()


def _relax_nodes(pos, radii, gap, iterations=700):
    names = list(pos)
    pos = {n: np.asarray(pos[n], dtype=float).copy() for n in names}
    for iteration in range(iterations):
        max_overlap = 0.0
        shifts = {n: np.zeros(2, dtype=float) for n in names}
        for i, u in enumerate(names):
            for v in names[i + 1:]:
                delta = pos[v] - pos[u]
                dist = float(np.linalg.norm(delta))
                needed = radii[u] + radii[v] + gap
                overlap = needed - dist
                if overlap <= 0:
                    continue
                max_overlap = max(max_overlap, overlap)
                if dist < 1e-9:
                    angle = (i * 0.754877666 + names.index(v) * 0.569840296) % (2 * np.pi)
                    unit = np.array([np.cos(angle), np.sin(angle)])
                else:
                    unit = delta / dist
                push = 0.52 * overlap * unit
                shifts[u] -= push
                shifts[v] += push
        if max_overlap < 0.05:
            break
        damping = 0.82 if iteration < 80 else 0.55
        for n in names:
            pos[n] += damping * shifts[n]
        center = np.mean(np.vstack(list(pos.values())), axis=0)
        for n in names:
            pos[n] -= center
    return pos


def _quadratic_point(p, c, q, t):
    return (1 - t) ** 2 * p + 2 * (1 - t) * t * c + t ** 2 * q


def _route_edge_labels(graph_name, pos, radii, span):
    G = GRAPHS[graph_name]
    frame = FRAME_CONFIG[graph_name]
    edges = EDGE_CONFIG[graph_name]
    span_x, span_y = span
    x_units_per_pt = span_x / (FIGSIZE[0] * 72 * frame['graph_rect'][2])
    y_units_per_pt = span_y / (FIGSIZE[1] * 72 * frame['graph_rect'][3])
    node_clearance = 9.0 * max(x_units_per_pt, y_units_per_pt)
    placed = []
    unrelated_segments = [(np.asarray(pos[a]), np.asarray(pos[b]), edge_key(a, b, G.is_directed()))
                          for a, b in G.edges()]

    ordered = sorted(G.edges(data=True), key=lambda item: (-item[2]['strength'], item[0], item[1]))
    for u, v, data in ordered:
        key = edge_key(u, v, G.is_directed())
        cfg = edges[key]
        p, q = np.asarray(pos[u]), np.asarray(pos[v])
        vec = q - p
        length = max(float(np.linalg.norm(vec)), 1.0)
        perp = np.array([-vec[1], vec[0]]) / length
        box_font = float(cfg['label_fontsize'])
        probe = label_rect((0.0, 0.0), cfg['label_text'], box_font,
                           x_units_per_pt, y_units_per_pt, pad_pt=2.6)
        label_half_h = 0.5 * (probe[3] - probe[1])
        base_offset = max(radii[u], radii[v]) + label_half_h + node_clearance
        offsets = (0.0, base_offset, -base_offset, 1.55 * base_offset, -1.55 * base_offset,
                   2.15 * base_offset, -2.15 * base_offset, 2.85 * base_offset, -2.85 * base_offset)
        along_fracs = (0.0, 0.11, -0.11, 0.21, -0.21)
        unit = vec / length
        midpoint = 0.5 * (p + q)
        best = None
        for offset in offsets:
            for along_frac in along_fracs:
                center = midpoint + perp * offset + unit * (along_frac * length)
                # Choose a quadratic control point whose midpoint passes beneath the label.
                control = 2.0 * center - 0.5 * (p + q)
                rect = label_rect(center, cfg['label_text'], box_font,
                                  x_units_per_pt, y_units_per_pt, pad_pt=2.6)
                score = 0.0
                for n, center_n in pos.items():
                    if _rect_circle_overlap(rect, center_n, radii[n] + node_clearance):
                        score += 1_000_000.0
                for prior in placed:
                    area = _rect_overlap_area(rect, prior)
                    if area:
                        score += 250_000.0 + area
                half_h = 0.5 * (rect[3] - rect[1])
                for a, b, other_key in unrelated_segments:
                    if other_key == key:
                        continue
                    if _point_segment_distance(center, a, b) < half_h * 0.82:
                        score += 1500.0
                x_excess = max(0.0, abs(center[0]) - span_x * 0.55)
                y_excess = max(0.0, abs(center[1]) - span_y * 0.55)
                score += 35.0 * (x_excess + y_excess)
                score += 0.55 * abs(offset) + 0.22 * abs(along_frac) * length
                candidate = (score, abs(offset), abs(along_frac), center, control, rect)
                if best is None or candidate[:3] < best[:3]:
                    best = candidate
        _, offset_abs, _, center, control, rect = best
        cfg['label_xy'] = tuple(map(float, center))
        cfg['control_xy'] = None if offset_abs < 1e-9 else tuple(map(float, control))
        placed.append(rect)
    return x_units_per_pt, y_units_per_pt


def _position_notes(graph_name, pos, radii, span, x_units_per_pt, y_units_per_pt):
    notes = NOTE_CONFIG[graph_name]
    if not notes:
        return
    nodes = NODE_CONFIG[graph_name]
    color_entries = COLOR_KEYS[graph_name]
    note_items = list(notes.items())
    global_center = np.mean(np.vstack(list(pos.values())), axis=0)
    occupied = []
    for n, center in pos.items():
        r = radii[n] + 12 * max(x_units_per_pt, y_units_per_pt)
        occupied.append((center[0] - r, center[1] - r, center[0] + r, center[1] + r))
    for cfg in EDGE_CONFIG[graph_name].values():
        occupied.append(label_rect(cfg['label_xy'], cfg['label_text'], cfg['label_fontsize'],
                                   x_units_per_pt, y_units_per_pt, pad_pt=2.8))

    for i, (note_key, cfg) in enumerate(note_items):
        target_color = matplotlib.colors.to_hex(color_entries[min(i, len(color_entries)-1)][1]).lower()
        members = [n for n in pos if matplotlib.colors.to_hex(nodes[n]['facecolor']).lower() == target_color]
        centroid = np.mean(np.vstack([pos[n] for n in members]), axis=0) if members else global_center
        base_vec = centroid - global_center
        if np.linalg.norm(base_vec) < 1e-9:
            angle0 = 2 * np.pi * i / max(len(note_items), 1)
        else:
            angle0 = math.atan2(base_vec[1], base_vec[0])
        best = None
        radii_try = (0.0, 170.0, 290.0, 430.0, 590.0, 760.0)
        offsets = (0, 30, -30, 60, -60, 90, -90, 120, -120, 180)
        for distance in radii_try:
            for offset in offsets:
                angle = angle0 + math.radians(offset)
                center = centroid + distance * np.array([math.cos(angle), math.sin(angle)])
                rect = label_rect(center, cfg['text'], cfg['fontsize'],
                                  x_units_per_pt, y_units_per_pt, pad_pt=4.0)
                overlap = sum(_rect_overlap_area(rect, other) for other in occupied)
                hard = sum(_rect_overlap_area(rect, other) > 0 for other in occupied)
                expansion = max(0.0, abs(center[0]) - span[0] * 0.56) + max(0.0, abs(center[1]) - span[1] * 0.56)
                score = hard * 1_000_000.0 + overlap + distance * 0.8 + expansion * 150.0
                candidate = (score, distance, abs(offset), center, rect)
                if best is None or candidate[:3] < best[:3]:
                    best = candidate
        cfg['xy'] = tuple(map(float, best[3]))
        occupied.append(best[4])


def apply_publication_quality_design(graph_name):
    if graph_name in _OPTIMIZED_GRAPHS:
        return
    G = GRAPHS[graph_name]
    frame = FRAME_CONFIG[graph_name]
    nodes = NODE_CONFIG[graph_name]
    settings = PUBLICATION_QUALITY_SETTINGS[graph_name]

    if graph_name in {'graph1', 'graph4'}:
        frame['graph_rect'] = (0.025, 0.080, 0.545, 0.805)
        frame['legend_xy'] = (0.590, 0.055)
        frame['legend_wh'] = (0.395, 0.850)
        frame['legend_columns'] = 3
        frame['legend_fontsize'] = 6.55
        frame['legend_heading_fontsize'] = 10.5
        frame['color_key_fontsize'] = 7.25
    else:
        frame['graph_rect'] = (0.025, 0.080, 0.680, 0.805)
        frame['legend_xy'] = (0.730, 0.055)
        frame['legend_wh'] = (0.255, 0.850)
        frame['legend_columns'] = 1
        frame['legend_fontsize'] = 7.65 if graph_name == 'graph2' else 7.35
        frame['legend_heading_fontsize'] = 10.5
        frame['color_key_fontsize'] = 7.45
    frame['title']['fontsize'] = 20.0
    frame['subtitle']['fontsize'] = 10.2
    frame['footer']['fontsize'] = 8.5

    for cfg in nodes.values():
        cfg['label_fontsize'] = settings['node_font']
        cfg['linewidth'] = min(float(cfg['linewidth']), 1.15)
    for cfg in EDGE_CONFIG[graph_name].values():
        cfg['label_fontsize'] = settings['edge_font']
        cfg['label_box_pad'] = 0.14
        cfg['label_box_linewidth'] = 0.40
        cfg['label_box_alpha'] = 0.94
    for cfg in NOTE_CONFIG[graph_name].values():
        cfg['fontsize'] = settings['note_font']
        cfg['box_pad'] = 0.23
        cfg['box_linewidth'] = 0.45
        cfg['box_alpha'] = 0.94

    strengths = np.array([d['strength'] for _, _, d in G.edges(data=True)], dtype=float)
    slo, shi = float(strengths.min()), float(strengths.max())
    H = G.to_undirected().copy()
    for u, v, data in H.edges(data=True):
        norm = (float(data['strength']) - slo) / (shi - slo + 1e-12)
        floor = float(settings['floor'])
        data['layout_weight'] = (floor + (1.0 - floor) * norm) ** float(settings['exponent'])
    raw = nx.spring_layout(H, seed=int(settings['seed']), weight='layout_weight',
                           iterations=850, k=float(settings['k']) / math.sqrt(max(len(H), 1)),
                           scale=1.0, center=(0.0, 0.0))
    arr = np.vstack([raw[n] for n in H])
    span_x, span_y = settings['span']
    # Robust scaling prevents one weakly connected peripheral node from shrinking the entire network.
    qx0, qx1 = np.quantile(arr[:, 0], [0.07, 0.93])
    qy0, qy1 = np.quantile(arr[:, 1], [0.07, 0.93])
    cx, cy = 0.5 * (qx0 + qx1), 0.5 * (qy0 + qy1)
    sx = 0.88 * span_x / max(float(qx1 - qx0), 1e-9)
    sy = 0.88 * span_y / max(float(qy1 - qy0), 1e-9)
    raw = {n: np.array([np.clip((raw[n][0] - cx) * sx, -0.52 * span_x, 0.52 * span_x),
                        np.clip((raw[n][1] - cy) * sy, -0.52 * span_y, 0.52 * span_y)], dtype=float)
           for n in H}
    center = np.mean(np.vstack(list(raw.values())), axis=0)
    raw = {n: p - center for n, p in raw.items()}

    radius_scale = NODE_RADIUS_SCALES[graph_name]
    radii = {n: float(nodes[n]['radius']) * radius_scale for n in G}
    pos = _relax_nodes(raw, radii, float(settings['node_gap']))
    for n in G:
        xy = tuple(map(float, pos[n]))
        nodes[n]['xy'] = xy
        nodes[n]['label_xy'] = xy

    x_units_per_pt, y_units_per_pt = _route_edge_labels(graph_name, pos, radii, settings['span'])
    _position_notes(graph_name, pos, radii, settings['span'], x_units_per_pt, y_units_per_pt)
    _OPTIMIZED_GRAPHS.add(graph_name)


def _bbox_overlap_count(bboxes):
    count = 0
    for i, first in enumerate(bboxes):
        for second in bboxes[i + 1:]:
            if first.overlaps(second):
                count += 1
    return count


def render_graph(graph_name: str):
    apply_publication_quality_design(graph_name)
    G = GRAPHS[graph_name]
    frame = FRAME_CONFIG[graph_name]
    nodes = NODE_CONFIG[graph_name]
    edges = EDGE_CONFIG[graph_name]
    notes = NOTE_CONFIG[graph_name]
    links = {
        'frame': f"#scrollTo=g{graph_name[-1]}_frame",
        'nodes': f"#scrollTo=g{graph_name[-1]}_nodes",
        'edges': f"#scrollTo=g{graph_name[-1]}_edges",
        'notes': f"#scrollTo=g{graph_name[-1]}_notes",
    }

    expected_node_keys = set(G.nodes)
    expected_edge_keys = {edge_key(u, v, G.is_directed()) for u, v in G.edges}
    if set(nodes) != expected_node_keys:
        raise AssertionError(f'{graph_name}: NODE_CONFIG keys do not match graph nodes.')
    if set(edges) != expected_edge_keys:
        raise AssertionError(f'{graph_name}: EDGE_CONFIG keys do not match graph edges.')
    label_mismatches = [
        edge_key(u, v, G.is_directed()) for u, v, data in G.edges(data=True)
        if edges[edge_key(u, v, G.is_directed())]['label_text'] != f"{data['strength']:.2f}"
    ]
    if label_mismatches:
        raise AssertionError(f'{graph_name}: stale edge labels: {label_mismatches[:5]}')

    fig = plt.figure(figsize=FIGSIZE, facecolor='white')
    _set_url(fig.patch, links['frame'])
    ax = fig.add_axes(frame['graph_rect'])
    ax.set_facecolor('white'); ax.axis('off'); ax.set_aspect('equal')
    _set_url(ax.patch, links['frame'])
    lx, ly = frame['legend_xy']; lw, lh = frame['legend_wh']
    color_entries = COLOR_KEYS[graph_name]
    key_cols = 2 if len(color_entries) > 5 else 1
    key_rows = math.ceil(len(color_entries) / key_cols)
    key_needed_pt = 18 + key_rows * (frame['color_key_fontsize'] * 1.85)
    legend_total_pt = FIGSIZE[1] * 72 * lh
    key_frac = float(np.clip(key_needed_pt / legend_total_pt, 0.12, 0.21))
    key_ax = fig.add_axes([lx, ly + lh * (1 - key_frac), lw, lh * key_frac])
    legend_ax = fig.add_axes([lx, ly, lw, lh * (1 - key_frac) - 0.012])

    strengths = np.array([d['strength'] for _, _, d in G.edges(data=True)], dtype=float)
    slo, shi = float(strengths.min()), float(strengths.max())
    pos = {n: np.asarray(nodes[n]['xy'], dtype=float) for n in G}
    radius_scale = NODE_RADIUS_SCALES[graph_name]
    radii = {n: float(nodes[n]['radius']) * radius_scale for n in G}

    for u, v, data in sorted(G.edges(data=True), key=lambda item: item[2]['strength']):
        key = edge_key(u, v, G.is_directed()); cfg = edges[key]
        norm = (data['strength'] - slo) / (shi - slo + 1e-12)
        linew = (0.72 + 2.55 * norm) * float(cfg['linewidth_scale'])
        alpha = np.clip((0.27 + 0.62 * norm) * float(cfg['alpha_scale']), 0.0, 1.0)
        start, end = clipped_edge_points(pos[u], pos[v], radii[u], radii[v])
        _draw_edge(ax, start, end, cfg['control_xy'], bool(frame['directed']), linew,
                   cfg['color'], alpha, 11 + 7 * norm, links['edges'])

    node_patches = []
    for n in G:
        cfg = nodes[n]
        circle = Circle(cfg['xy'], radius=radii[n], facecolor=cfg['facecolor'],
                        edgecolor=cfg['edgecolor'], linewidth=cfg['linewidth'],
                        alpha=cfg['alpha'], zorder=3)
        ax.add_patch(_set_url(circle, links['nodes']))
        node_patches.append(circle)
        txt = ax.text(*cfg['label_xy'], cfg['label_text'], ha=cfg['label_ha'], va=cfg['label_va'],
                      fontsize=cfg['label_fontsize'], fontweight=cfg['label_fontweight'],
                      color=cfg['label_color'], rotation=cfg['label_rotation'], zorder=4)
        _set_url(txt, links['nodes'])

    edge_label_artists = []
    for u, v, data in G.edges(data=True):
        key = edge_key(u, v, G.is_directed()); cfg = edges[key]
        txt = ax.text(*cfg['label_xy'], cfg['label_text'], ha=cfg['label_ha'], va=cfg['label_va'],
                      fontsize=cfg['label_fontsize'], color=cfg['label_color'], zorder=5,
                      bbox=dict(boxstyle=f"round,pad={cfg['label_box_pad']}",
                                facecolor=cfg['label_box_facecolor'], edgecolor=cfg['label_box_edgecolor'],
                                linewidth=cfg['label_box_linewidth'], alpha=cfg['label_box_alpha']))
        _set_url(txt, links['edges'])
        edge_label_artists.append(txt)

    note_artists = []
    for key, cfg in notes.items():
        txt = ax.text(*cfg['xy'], cfg['text'], ha=cfg['ha'], va=cfg['va'], fontsize=cfg['fontsize'],
                      color=cfg['color'], fontweight=cfg['fontweight'], linespacing=cfg['linespacing'], zorder=6,
                      bbox=dict(boxstyle=f"round,pad={cfg['box_pad']}", facecolor=cfg['box_facecolor'],
                                edgecolor=cfg['box_edgecolor'], linewidth=cfg['box_linewidth'], alpha=cfg['box_alpha']))
        _set_url(txt, links['notes'])
        note_artists.append(txt)

    settings = PUBLICATION_QUALITY_SETTINGS[graph_name]
    span_x, span_y = settings['span']
    x_units_per_pt = span_x / (FIGSIZE[0] * 72 * frame['graph_rect'][2])
    y_units_per_pt = span_y / (FIGSIZE[1] * 72 * frame['graph_rect'][3])
    boxes = []
    for n in G:
        x, y = nodes[n]['xy']; r = radii[n]; boxes.append((x-r, y-r, x+r, y+r))
    for cfg in edges.values():
        boxes.append(label_rect(cfg['label_xy'], cfg['label_text'], cfg['label_fontsize'],
                                x_units_per_pt, y_units_per_pt, pad_pt=3.0))
    for cfg in notes.values():
        boxes.append(label_rect(cfg['xy'], cfg['text'], cfg['fontsize'],
                                x_units_per_pt, y_units_per_pt, pad_pt=4.5))
    xmin = min(b[0] for b in boxes); ymin = min(b[1] for b in boxes)
    xmax = max(b[2] for b in boxes); ymax = max(b[3] for b in boxes)
    xpad = 0.035 * (xmax - xmin); ypad = 0.035 * (ymax - ymin)
    ax.set_xlim(xmin - xpad, xmax + xpad); ax.set_ylim(ymin - ypad, ymax + ypad)

    for element in ('title', 'subtitle', 'footer'):
        cfg = frame[element]
        txt = fig.text(*cfg['xy'], cfg['text'], ha=cfg['ha'], va=cfg['va'], fontsize=cfg['fontsize'],
                       fontweight=cfg['fontweight'], color=cfg['color'])
        _set_url(txt, links['frame'])
    add_color_key(key_ax, color_entries, frame, links['frame'])
    legend_entry_artists = draw_legend(legend_ax, LEGEND_ENTRIES[graph_name], frame, links['nodes'])

    # Exact rendered overlap audit in display coordinates.
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    edge_boxes_px = [artist.get_window_extent(renderer).expanded(1.04, 1.12) for artist in edge_label_artists]
    node_boxes_px = [patch.get_window_extent(renderer).expanded(1.02, 1.02) for patch in node_patches]
    note_boxes_px = [artist.get_window_extent(renderer).expanded(1.03, 1.08) for artist in note_artists]
    legend_boxes_px = []
    from matplotlib.transforms import Bbox
    for code_artist, text_artist in legend_entry_artists:
        legend_boxes_px.append(Bbox.union([code_artist.get_window_extent(renderer),
                                           text_artist.get_window_extent(renderer)]))
    edge_label_overlaps = _bbox_overlap_count(edge_boxes_px)
    edge_label_node_overlaps = sum(label.overlaps(node) for label in edge_boxes_px for node in node_boxes_px)
    note_overlaps = _bbox_overlap_count(note_boxes_px) + sum(note.overlaps(node) for note in note_boxes_px for node in node_boxes_px)
    legend_overlaps = _bbox_overlap_count(legend_boxes_px)

    png_path = OUTPUT_DIR / frame['output_name']
    svg_path = png_path.with_suffix('.svg')
    fig.savefig(png_path, dpi=EXPORT_DPI, facecolor='white', format='png', pil_kwargs={'compress_level': 6})
    fig.savefig(svg_path, facecolor='white', format='svg')
    plt.close(fig); gc.collect()
    patch_svg_links(svg_path)

    with PILImage.open(png_path) as image:
        image.load(); pixel_width, pixel_height = image.size

    actual_lengths = np.array([np.linalg.norm(pos[u] - pos[v]) for u, v in G.edges()], dtype=float)
    length_rho = float(spearmanr(strengths, actual_lengths).statistic)
    degrees = np.array([G.degree(n) for n in G], dtype=float)
    radius_values = np.array([radii[n] for n in G], dtype=float)
    radius_rho = float(spearmanr(degrees, radius_values).statistic)
    rule = RADIUS_RULES[graph_name]
    degree_min, degree_max = float(degrees.min()), float(degrees.max())
    expected_radii = np.array([
        degree_radius(G.degree(n), degree_min, degree_max, rule['min_radius'], rule['max_radius'])
        * radius_scale for n in G
    ], dtype=float)
    radius_rule_max_error = float(np.max(np.abs(radius_values - expected_radii)))
    node_overlap = sum(np.linalg.norm(pos[u]-pos[v]) < radii[u]+radii[v]+1.0
                       for i,u in enumerate(G) for v in list(G)[i+1:])
    if not np.isfinite(length_rho) or length_rho >= STRENGTH_LENGTH_RHO_MAX:
        raise AssertionError(f'{graph_name}: edge-strength/length relationship failed: rho={length_rho}')
    if not np.isfinite(radius_rho) or radius_rho < 0.999:
        raise AssertionError(f'{graph_name}: node-radius/degree relationship failed: rho={radius_rho}')
    if radius_rule_max_error > 1e-7:
        raise AssertionError(f'{graph_name}: node radii no longer follow the documented rule: {radius_rule_max_error}')
    if node_overlap:
        raise AssertionError(f'{graph_name}: {node_overlap} node overlaps remain after collision relaxation.')
    if legend_overlaps:
        raise AssertionError(f'{graph_name}: {legend_overlaps} rendered legend-entry overlaps remain.')

    if DISPLAY_CLICKABLE_SVG:
        display_clickable_svg(svg_path)
    return {
        'graph': graph_name, 'nodes': G.number_of_nodes(), 'edges': G.number_of_edges(),
        'node_radius_scale': radius_scale,
        'degree_vs_radius_spearman': radius_rho,
        'radius_rule_max_abs_error': radius_rule_max_error,
        'strength_vs_length_spearman': length_rho,
        'edge_label_mismatches': 0,
        'node_overlaps': int(node_overlap),
        'edge_label_overlaps': int(edge_label_overlaps),
        'edge_label_node_overlaps': int(edge_label_node_overlaps),
        'annotation_overlaps': int(note_overlaps),
        'legend_entry_overlaps': int(legend_overlaps),
        'pixel_width': pixel_width, 'pixel_height': pixel_height,
        'png': str(png_path), 'svg': str(svg_path),
    }


In [6]:
FRAME_CONFIG = {}
NODE_CONFIG = {}
EDGE_CONFIG = {}
NOTE_CONFIG = {}
diagnostics_by_graph = {}


# Graph 1 — Publication abstract similarity

**Nodes:** all 53 publications, indexed oldest to newest. **Edges:** the undirected union of each paper's two closest TF-IDF/cosine neighbors, plus only the strongest bridges required to make the network connected. **Node radius:** square-root scaling of connection count. **Edge label:** cosine similarity. **Edge distance:** inverse-ranked to similarity. **Color:** weighted modularity community; the description boxes summarize the actual member titles. These links are textual relationships, not evidence of causation or citation dependence.

## Frame, titles, legend, and graph background

In [7]:
# GRAPH 1: title, subtitle, footer, legend and blank background link here.
FRAME_CONFIG['graph1'] = {'output_name': '01_publication_similarity_network.png',
 'graph_rect': (0.025, 0.075, 0.705, 0.815),
 'legend_xy': (0.745, 0.055),
 'legend_wh': (0.245, 0.85),
 'title': {'text': 'Graph 1 - ChemicalQDevice Publication Abstract Similarity Network',
           'xy': (0.025, 0.975),
           'fontsize': 22.0,
           'ha': 'left',
           'va': 'top',
           'fontweight': 'bold',
           'color': '#111827'},
 'subtitle': {'text': '53 README abstracts. Each paper connects to its two nearest TF-IDF/cosine neighbors; '
                      'only the strongest necessary bridges join disconnected components.',
              'xy': (0.025, 0.94),
              'fontsize': 11.2,
              'ha': 'left',
              'va': 'top',
              'fontweight': 'normal',
              'color': '#374151'},
 'footer': {'text': 'Node radius is square-root-scaled connection count. Stronger cosine similarities are '
                    'shorter; edge labels report cosine similarity. Communities are descriptive, not causal.',
            'xy': (0.025, 0.018),
            'fontsize': 9.2,
            'ha': 'left',
            'va': 'bottom',
            'fontweight': 'normal',
            'color': '#374151'},
 'legend_title': 'Publication index (oldest → newest)',
 'legend_columns': 2,
 'legend_fontsize': 7.5,
 'legend_heading_fontsize': 11.5,
 'color_key_title': 'Color meaning',
 'color_key_fontsize': 8.3,
 'edge_color': '#6f7782',
 'directed': False}


## Nodes, node labels, and legend entries

In [8]:
# GRAPH 1: every node, node label and legend item links here.
# Radius defaults follow the square-root degree rule validated by the renderer.
NODE_CONFIG['graph1'] = {'P01': {'xy': (5048.315027610999, 2825.0497766896287),
         'radius': 48.34537420613657,
         'facecolor': '#d9d9d9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P01',
         'label_xy': (5048.315027610999, 2825.0497766896287),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P02': {'xy': (4778.185942031286, 3164.291865155899),
         'radius': 48.34537420613657,
         'facecolor': '#d9d9d9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P02',
         'label_xy': (4778.185942031286, 3164.291865155899),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P03': {'xy': (4652.9893581637825, 2947.382053573894),
         'radius': 53.142637586900065,
         'facecolor': '#d9d9d9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P03',
         'label_xy': (4652.9893581637825, 2947.382053573894),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P04': {'xy': (4212.376281510723, 2799.832992347204),
         'radius': 53.142637586900065,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P04',
         'label_xy': (4212.376281510723, 2799.832992347204),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P05': {'xy': (3898.2993005860335, 2970.422255166749),
         'radius': 48.34537420613657,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P05',
         'label_xy': (3898.2993005860335, 2970.422255166749),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P06': {'xy': (3786.8764352055796, 2690.801295865652),
         'radius': 53.142637586900065,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P06',
         'label_xy': (3786.8764352055796, 2690.801295865652),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P07': {'xy': (3385.9912312798033, 2485.191707752843),
         'radius': 53.142637586900065,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P07',
         'label_xy': (3385.9912312798033, 2485.191707752843),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P08': {'xy': (2889.2512191466144, 2637.2020513715925),
         'radius': 48.34537420613657,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P08',
         'label_xy': (2889.2512191466144, 2637.2020513715925),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P09': {'xy': (3099.1003152660837, 2082.506716383628),
         'radius': 48.34537420613657,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P09',
         'label_xy': (3099.1003152660837, 2082.506716383628),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P10': {'xy': (2657.6742970724613, 2142.5348148036715),
         'radius': 53.142637586900065,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P10',
         'label_xy': (2657.6742970724613, 2142.5348148036715),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P11': {'xy': (2270.355310545043, 1972.5018300187514),
         'radius': 48.34537420613657,
         'facecolor': '#8dd3c7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P11',
         'label_xy': (2270.355310545043, 1972.5018300187514),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P12': {'xy': (1771.050788237245, 1888.521494515207),
         'radius': 53.142637586900065,
         'facecolor': '#80b1d3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P12',
         'label_xy': (1771.050788237245, 1888.521494515207),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P13': {'xy': (1560.984279721608, 1631.0596107353733),
         'radius': 48.34537420613657,
         'facecolor': '#80b1d3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P13',
         'label_xy': (1560.984279721608, 1631.0596107353733),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P14': {'xy': (1281.703477247972, 1821.6007594474668),
         'radius': 60.75,
         'facecolor': '#80b1d3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P14',
         'label_xy': (1281.703477247972, 1821.6007594474668),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P15': {'xy': (1187.9682491389542, 2171.743731646562),
         'radius': 48.34537420613657,
         'facecolor': '#80b1d3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P15',
         'label_xy': (1187.9682491389542, 2171.743731646562),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P16': {'xy': (1015.6042750084916, 2405.9103007559274),
         'radius': 48.34537420613657,
         'facecolor': '#80b1d3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P16',
         'label_xy': (1015.6042750084916, 2405.9103007559274),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P17': {'xy': (888.0812022842233, 1412.9634725392375),
         'radius': 53.142637586900065,
         'facecolor': '#b3de69',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P17',
         'label_xy': (888.0812022842233, 1412.9634725392375),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P18': {'xy': (479.79725184418345, 1506.6876468442508),
         'radius': 53.142637586900065,
         'facecolor': '#b3de69',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P18',
         'label_xy': (479.79725184418345, 1506.6876468442508),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P19': {'xy': (488.10099160935175, 1198.134424516665),
         'radius': 57.186917696247164,
         'facecolor': '#b3de69',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P19',
         'label_xy': (488.10099160935175, 1198.134424516665),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P20': {'xy': (54.44480537073369, 1645.2194167998719),
         'radius': 48.34537420613657,
         'facecolor': '#b3de69',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P20',
         'label_xy': (54.44480537073369, 1645.2194167998719),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P21': {'xy': (-2822.8184028857518, -192.42048001127844),
         'radius': 48.34537420613657,
         'facecolor': '#fb8072',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P21',
         'label_xy': (-2822.8184028857518, -192.42048001127844),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P22': {'xy': (-2737.4029149981807, -601.8458286925055),
         'radius': 57.186917696247164,
         'facecolor': '#fb8072',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P22',
         'label_xy': (-2737.4029149981807, -601.8458286925055),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P23': {'xy': (-2400.777960402264, -449.91996607967934),
         'radius': 53.142637586900065,
         'facecolor': '#fb8072',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P23',
         'label_xy': (-2400.777960402264, -449.91996607967934),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P24': {'xy': (-2276.0266509820503, -690.8328633600015),
         'radius': 53.142637586900065,
         'facecolor': '#fb8072',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P24',
         'label_xy': (-2276.0266509820503, -690.8328633600015),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P25': {'xy': (-691.8780601132637, -842.577408564892),
         'radius': 48.34537420613657,
         'facecolor': '#bc80bd',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P25',
         'label_xy': (-691.8780601132637, -842.577408564892),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P26': {'xy': (-884.5783904883904, -689.6370148208341),
         'radius': 53.142637586900065,
         'facecolor': '#bc80bd',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P26',
         'label_xy': (-884.5783904883904, -689.6370148208341),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P27': {'xy': (-795.9607984090065, -617.1660750898762),
         'radius': 53.142637586900065,
         'facecolor': '#bc80bd',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P27',
         'label_xy': (-795.9607984090065, -617.1660750898762),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P28': {'xy': (-3060.6500486766336, -1036.8971158962331),
         'radius': 53.142637586900065,
         'facecolor': '#fb8072',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P28',
         'label_xy': (-3060.6500486766336, -1036.8971158962331),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P29': {'xy': (-1714.5125922866378, -871.883250391624),
         'radius': 48.34537420613657,
         'facecolor': '#fccde5',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P29',
         'label_xy': (-1714.5125922866378, -871.883250391624),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P30': {'xy': (-1282.320541772143, -1651.0422390912147),
         'radius': 53.142637586900065,
         'facecolor': '#fccde5',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P30',
         'label_xy': (-1282.320541772143, -1651.0422390912147),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P31': {'xy': (-2689.3824748633297, -988.2701271464744),
         'radius': 48.34537420613657,
         'facecolor': '#fb8072',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P31',
         'label_xy': (-2689.3824748633297, -988.2701271464744),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P32': {'xy': (-3304.443129820621, -1570.58075487066),
         'radius': 48.34537420613657,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P32',
         'label_xy': (-3304.443129820621, -1570.58075487066),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P33': {'xy': (-1331.207547137789, -1066.7134986695646),
         'radius': 53.142637586900065,
         'facecolor': '#fccde5',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P33',
         'label_xy': (-1331.207547137789, -1066.7134986695646),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P34': {'xy': (-2960.421980682674, -2559.568809727478),
         'radius': 48.34537420613657,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P34',
         'label_xy': (-2960.421980682674, -2559.568809727478),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P35': {'xy': (-2761.2489599896044, -2516.5175428269704),
         'radius': 53.142637586900065,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P35',
         'label_xy': (-2761.2489599896044, -2516.5175428269704),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P36': {'xy': (-3214.4826822440814, -2162.9085247662224),
         'radius': 53.142637586900065,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P36',
         'label_xy': (-3214.4826822440814, -2162.9085247662224),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P37': {'xy': (-1888.492427650548, -2230.4807292611104),
         'radius': 48.34537420613657,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P37',
         'label_xy': (-1888.492427650548, -2230.4807292611104),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P38': {'xy': (-2211.9204016351955, -2584.839547272069),
         'radius': 53.142637586900065,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P38',
         'label_xy': (-2211.9204016351955, -2584.839547272069),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P39': {'xy': (-1304.0965449413927, -1974.0959666565707),
         'radius': 53.142637586900065,
         'facecolor': '#fccde5',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P39',
         'label_xy': (-1304.0965449413927, -1974.0959666565707),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P40': {'xy': (-1147.1171790845292, -2279.124481531573),
         'radius': 57.186917696247164,
         'facecolor': '#fdb462',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P40',
         'label_xy': (-1147.1171790845292, -2279.124481531573),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P41': {'xy': (-879.2632184419789, -2477.9279089102497),
         'radius': 48.34537420613657,
         'facecolor': '#fdb462',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P41',
         'label_xy': (-879.2632184419789, -2477.9279089102497),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P42': {'xy': (-928.7032829540938, -2378.2504912965414),
         'radius': 53.142637586900065,
         'facecolor': '#fdb462',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P42',
         'label_xy': (-928.7032829540938, -2378.2504912965414),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P43': {'xy': (-1006.1980941078344, -2467.965841529383),
         'radius': 57.186917696247164,
         'facecolor': '#fdb462',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P43',
         'label_xy': (-1006.1980941078344, -2467.965841529383),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P44': {'xy': (-1906.0625447777, -3003.560466564551),
         'radius': 48.34537420613657,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P44',
         'label_xy': (-1906.0625447777, -3003.560466564551),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P45': {'xy': (-1700.967781537838, -2991.9958757684853),
         'radius': 48.34537420613657,
         'facecolor': '#ffffb3',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P45',
         'label_xy': (-1700.967781537838, -2991.9958757684853),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P46': {'xy': (-1313.3828247712204, -2671.846646497065),
         'radius': 53.142637586900065,
         'facecolor': '#fdb462',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P46',
         'label_xy': (-1313.3828247712204, -2671.846646497065),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P47': {'xy': (-218.95061204399065, -77.66812494079579),
         'radius': 53.142637586900065,
         'facecolor': '#bebada',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P47',
         'label_xy': (-218.95061204399065, -77.66812494079579),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P48': {'xy': (-326.52145321899775, 4.751551100234073),
         'radius': 48.34537420613657,
         'facecolor': '#bebada',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P48',
         'label_xy': (-326.52145321899775, 4.751551100234073),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P49': {'xy': (258.29753513318246, 687.4389247439512),
         'radius': 48.34537420613657,
         'facecolor': '#bebada',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P49',
         'label_xy': (258.29753513318246, 687.4389247439512),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P50': {'xy': (19.046198877087807, 318.3622223801727),
         'radius': 48.34537420613657,
         'facecolor': '#bebada',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P50',
         'label_xy': (19.046198877087807, 318.3622223801727),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P51': {'xy': (-310.17013000032335, -372.424011099146),
         'radius': 60.75,
         'facecolor': '#bebada',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P51',
         'label_xy': (-310.17013000032335, -372.424011099146),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P52': {'xy': (151.25835614346087, -662.9006664425576),
         'radius': 48.34537420613657,
         'facecolor': '#bebada',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P52',
         'label_xy': (151.25835614346087, -662.9006664425576),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P53': {'xy': (234.20750188315554, -728.2486573788328),
         'radius': 48.34537420613657,
         'facecolor': '#bebada',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P53',
         'label_xy': (234.20750188315554, -728.2486573788328),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0}}


## Edges and numeric edge labels

In [9]:
# GRAPH 1: every edge and numeric edge label links here.
# Set control_xy=(x, y) to bend one edge; None keeps a straight edge.
EDGE_CONFIG['graph1'] = {'P01|P03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.32',
             'label_xy': (4844.902507217809, 2867.635386800805),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P01|P02': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.24',
             'label_xy': (4897.384789724838, 2982.0374070447047),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P02|P03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.54',
             'label_xy': (4730.578953315933, 3047.1842389693697),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P03|P04': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.27',
             'label_xy': (4439.036279466048, 2854.6347326821133),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P04|P05': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.37',
             'label_xy': (4046.261123630746, 2868.416300592382),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P04|P06': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.28',
             'label_xy': (4004.555756437322, 2726.08000304792),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P05|P06': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.46',
             'label_xy': (3859.4313385133146, 2823.9000191305377),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P06|P07': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.28',
             'label_xy': (3595.487998918827, 2570.3432343387435),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P07|P09': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.26',
             'label_xy': (3258.909531489663, 2272.190934047346),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P07|P08': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.23',
             'label_xy': (3131.64626569998, 2541.671882838702),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P08|P10': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.21',
             'label_xy': (2792.1041455886652, 2381.1415257517847),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P09|P10': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.31',
             'label_xy': (2875.7490744250103, 2093.120115488218),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P10|P11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.32',
             'label_xy': (2471.862928340733, 2039.6410402843758),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P11|P12': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.22',
             'label_xy': (2024.0918348252683, 1910.3636611635811),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P12|P13': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.32',
             'label_xy': (1681.1484376769904, 1747.4450509775525),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P12|P14': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.29',
             'label_xy': (1529.0624757184116, 1835.424980382483),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P13|P14': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.40',
             'label_xy': (1410.8139494261106, 1710.8962116039297),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P14|P15': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.35',
             'label_xy': (1216.243464479415, 1991.6949566309765),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P14|P16': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.21',
             'label_xy': (1129.9490597019924, 2105.2372082582574),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P14|P17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.21',
             'label_xy': (1099.678047742372, 1603.039695643988),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P15|P16': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.32',
             'label_xy': (1086.0739605536644, 2277.261596154165),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P17|P19': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.31',
             'label_xy': (697.3533145740623, 1288.3040564057062),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P17|P18': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.29',
             'label_xy': (679.5021696561463, 1440.4967222778746),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P18|P19': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.39',
             'label_xy': (502.7746276547164, 1352.917664977647),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P18|P20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.18',
             'label_xy': (260.66063889832645, 1556.1173414028417),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P19|P20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.21',
             'label_xy': (256.5169958029732, 1407.3642315852046),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P19|P49': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.18',
             'label_xy': (392.2145796165225, 934.2301367303423),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P21|P23': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.27',
             'label_xy': (-2601.3701726252752, -304.0787654752737),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P21|P22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.25',
             'label_xy': (-2760.395253888039, -393.0200701302959),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P22|P23': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.31',
             'label_xy': (-2577.169570503349, -507.9818128290958),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P22|P24': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.28',
             'label_xy': (-2502.9457856115887, -626.7980056940473),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P22|P28': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.21',
             'label_xy': (-2882.5022693496817, -831.649115138004),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P23|P24': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.52',
             'label_xy': (-2322.851025459077, -562.3235350519661),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P24|P31': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.27',
             'label_xy': (-2471.043962019035, -855.7565168840898),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P25|P26': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.52',
             'label_xy': (-799.1338116735783, -779.8479249571461),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P25|P27': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.51',
             'label_xy': (-759.938760651841, -737.26860094315),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P26|P27': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.85',
             'label_xy': (-871.9224928287396, -614.6963253571822),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P26|P33': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.22',
             'label_xy': (-1094.6716809852032, -893.835243207672),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P27|P51': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.25',
             'label_xy': (-562.1434464580423, -476.77607653534477),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P28|P31': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.38',
             'label_xy': (-2877.475997930855, -993.8035098971711),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P28|P32': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.20',
             'label_xy': (-3163.742768959745, -1312.328745808053),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P29|P33': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.37',
             'label_xy': (-1514.290668724719, -952.4391103175533),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P29|P30': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.22',
             'label_xy': (-1477.4290890964694, -1249.8211911502895),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P30|P39': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.48',
             'label_xy': (-1275.3625090282271, -1813.7720461212143),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P30|P33': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.25',
             'label_xy': (-1326.9345185556435, -1360.5654019544531),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P32|P36': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.20',
             'label_xy': (-3239.0080871496716, -1863.6380411490804),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P34|P35': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.61',
             'label_xy': (-2864.3524397450064, -2521.7722167612587),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P34|P36': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.24',
             'label_xy': (-3104.534336396953, -2372.179682322064),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P35|P38': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.22',
             'label_xy': (-2484.0576369764494, -2530.360386081915),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P35|P36': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.21',
             'label_xy': (-3000.5419717601803, -2355.9605236695816),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P37|P38': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.30',
             'label_xy': (-2035.6966089532857, -2420.903429581972),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P37|P39': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.18',
             'label_xy': (-1604.6926475658095, -2083.145826310658),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P38|P44': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.26',
             'label_xy': (-2042.81105976059, -2782.380901753991),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P39|P40': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.46',
             'label_xy': (-1209.4666593084926, -2118.3038571093757),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P40|P42': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.57',
             'label_xy': (-1030.8570019203873, -2313.1464265908157),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P40|P43': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.56',
             'label_xy': (-1062.910727016444, -2363.2868054200558),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P40|P46': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.37',
             'label_xy': (-1212.7133606865254, -2482.9100010318352),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P41|P42': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.92',
             'label_xy': (-844.8567723508933, -2398.7624280637583),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P41|P43': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.80',
             'label_xy': (-949.3029113241437, -2556.689370188427),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P42|P43': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.82',
             'label_xy': (-1005.2890488614406, -2390.423942580951),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P43|P46': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.30',
             'label_xy': (-1148.8791370942263, -2586.3462007632183),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P44|P45': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.63',
             'label_xy': (-1804.441307523005, -2981.353261486302),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P45|P46': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.25',
             'label_xy': (-1520.0028453702573, -2816.3917457051743),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P47|P48': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.75',
             'label_xy': (-294.63098381584547, -65.03469524314177),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P47|P51': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.49',
             'label_xy': (-247.49213354023834, -230.32825734537218),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P47|P50': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.30',
             'label_xy': (-116.80980066311314, 130.47772092171016),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P48|P51': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.34',
             'label_xy': (-299.101234847119, -183.0019395539416),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P49|P50': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.30',
             'label_xy': (155.21706612910822, 492.1752664004842),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P51|P52': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.21',
             'label_xy': (-68.4934799092822, -500.2483167408964),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P51|P53': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.17',
             'label_xy': (-26.49166922183398, -532.7582779890695),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'P52|P53': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.98',
             'label_xy': (223.67482207592724, -656.2987245736762),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97}}


## Explanatory description boxes

In [10]:
# GRAPH 1: every explanatory description box links here.
NOTE_CONFIG['graph1'] = {'community_0': {'text': 'Early cancer AI\n& bioprocess',
                 'xy': (4242.7274879467495, 3203.2674714891336),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_1': {'text': 'Robotic trials\n& assurance',
                 'xy': (-3412.102062548153, -3356.068360680251),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_2': {'text': 'PDAC protocols\n& funding',
                 'xy': (-241.3772987921203, -1039.8107273668033),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_3': {'text': 'Trial platforms\n& infrastructure',
                 'xy': (-3287.5884496462986, -814.3751424732731),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_4': {'text': 'In silico trials\n& digital twins',
                 'xy': (1724.4693213492255, 2509.0138962468445),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_5': {'text': 'Federal policy\n& H.R. 9510',
                 'xy': (-1225.6824197840483, -2852.388588151786),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_6': {'text': 'LLM delivery\n& validation',
                 'xy': (653.9148489116876, 1972.606093105678),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_7': {'text': 'Trial sites\n& policy',
                 'xy': (-1901.132339056789, -1878.0430987085651),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_8': {'text': 'Multimodal\nchemistry',
                 'xy': (5133.100978232916, 3168.143636593416),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
 'community_9': {'text': 'ICH / CFR\nadaptations',
                 'xy': (-976.5460719191383, -884.7385862336212),
                 'fontsize': 11.5,
                 'ha': 'center',
                 'va': 'center',
                 'fontweight': 'normal',
                 'color': 'black',
                 'linespacing': 1.08,
                 'box_pad': 0.28,
                 'box_facecolor': 'white',
                 'box_edgecolor': '#d7dce1',
                 'box_linewidth': 0.55,
                 'box_alpha': 0.98},
}


## Render Graph 1

Run this cell after edits. It outputs only Graph 1 in this section and writes its PNG and clickable SVG to `OUTPUT_DIR`.

In [11]:
diagnostics_by_graph['graph1'] = render_graph('graph1')
display(pd.DataFrame([diagnostics_by_graph['graph1']]))


,graph,nodes,edges,node_radius_scale,degree_vs_radius_spearman,radius_rule_max_abs_error,strength_vs_length_spearman,edge_label_mismatches,node_overlaps,edge_label_overlaps,edge_label_node_overlaps,annotation_overlaps,legend_entry_overlaps,pixel_width,pixel_height,png,svg
0,graph1,53,71,1.2,1.0,0.0,-0.761502,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/01_publicat...,/content/cqd_network_graph_outputs/01_publicat...


# Graph 2 — Research-theme co-occurrence

**Nodes:** 22 explicit README themes. The editable regex rules and four reviewed false-positive corrections are above. The legend's `n=` value is abstract frequency; node radius instead represents network connection count. **Edges:** Ochiai-normalized co-mention across abstracts, selected from each theme's strongest associations. **Edge label:** normalized association. This is descriptive co-occurrence, not statistical inference or causality.

## Frame, titles, legend, and graph background

In [12]:
# GRAPH 2: title, subtitle, footer, legend and blank background link here.
FRAME_CONFIG['graph2'] = {'output_name': '02_theme_cooccurrence_network.png',
 'graph_rect': (0.025, 0.075, 0.705, 0.815),
 'legend_xy': (0.745, 0.055),
 'legend_wh': (0.245, 0.85),
 'title': {'text': 'Graph 2 - ChemicalQDevice Research Theme Co-occurrence Network',
           'xy': (0.025, 0.975),
           'fontsize': 22.0,
           'ha': 'left',
           'va': 'top',
           'fontweight': 'bold',
           'color': '#111827'},
 'subtitle': {'text': 'Themes use explicit README wording plus four reviewed false-positive corrections. Edges '
                      'report Ochiai-normalized abstract co-mentions.',
              'xy': (0.025, 0.94),
              'fontsize': 11.2,
              'ha': 'left',
              'va': 'top',
              'fontweight': 'normal',
              'color': '#374151'},
 'footer': {'text': 'Node radius is square-root-scaled connection count, not abstract frequency. Stronger '
                    'normalized co-mentions are shorter; labels report Ochiai association.',
            'xy': (0.025, 0.018),
            'fontsize': 9.2,
            'ha': 'left',
            'va': 'bottom',
            'fontweight': 'normal',
            'color': '#374151'},
 'legend_title': 'Theme index and abstract frequency',
 'legend_columns': 1,
 'legend_fontsize': 8.2,
 'legend_heading_fontsize': 11.5,
 'color_key_title': 'Color meaning',
 'color_key_fontsize': 8.3,
 'edge_color': '#6f7782',
 'directed': False}


## Nodes, node labels, and legend entries

In [13]:
# GRAPH 2: every node, node label and legend item links here.
# Radius defaults follow the square-root degree rule validated by the renderer.
NODE_CONFIG['graph2'] = {'T01': {'xy': (136.95792552596978, -139.41404621330975),
         'radius': 42.821773229381925,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T01',
         'label_xy': (136.95792552596978, -139.41404621330975),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T02': {'xy': (167.36300044074284, -572.093399213659),
         'radius': 32.5,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T02',
         'label_xy': (167.36300044074284, -572.093399213659),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T03': {'xy': (54.810997209441766, 173.8263531851889),
         'radius': 40.41241452319315,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T03',
         'label_xy': (54.810997209441766, 173.8263531851889),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T04': {'xy': (202.54517172539238, 63.242069893748386),
         'radius': 39.09406539564933,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T04',
         'label_xy': (202.54517172539238, 63.242069893748386),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T05': {'xy': (440.24378913059803, 188.85014707160894),
         'radius': 34.433756729740644,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T05',
         'label_xy': (440.24378913059803, 188.85014707160894),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T06': {'xy': (-125.8633817870681, 211.51286403648658),
         'radius': 39.09406539564933,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T06',
         'label_xy': (-125.8633817870681, 211.51286403648658),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T07': {'xy': (-119.4800893626399, 124.43161998850202),
         'radius': 39.09406539564933,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T07',
         'label_xy': (-119.4800893626399, 124.43161998850202),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T08': {'xy': (2.497775179765542, 375.0576969651861),
         'radius': 34.433756729740644,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T08',
         'label_xy': (2.497775179765542, 375.0576969651861),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T09': {'xy': (-337.46854404818663, 382.6820755592956),
         'radius': 34.433756729740644,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T09',
         'label_xy': (-337.46854404818663, 382.6820755592956),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T10': {'xy': (-327.5015191238513, 154.03190800824623),
         'radius': 36.13743060919757,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T10',
         'label_xy': (-327.5015191238513, 154.03190800824623),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T11': {'xy': (-127.6369249654182, -4.174138692909387),
         'radius': 42.821773229381925,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T11',
         'label_xy': (-127.6369249654182, -4.174138692909387),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T12': {'xy': (-124.7398119869422, -225.20227418231448),
         'radius': 34.433756729740644,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T12',
         'label_xy': (-124.7398119869422, -225.20227418231448),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T13': {'xy': (269.9584968506061, 257.6478573088721),
         'radius': 36.13743060919757,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T13',
         'label_xy': (269.9584968506061, 257.6478573088721),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T14': {'xy': (-300.43097339936145, -435.5261988762811),
         'radius': 30.206207261596575,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T14',
         'label_xy': (-300.43097339936145, -435.5261988762811),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T15': {'xy': (411.6039212628438, -209.51211772727476),
         'radius': 32.5,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T15',
         'label_xy': (411.6039212628438, -209.51211772727476),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T16': {'xy': (239.90995572453517, -547.2107155315117),
         'radius': 32.5,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T16',
         'label_xy': (239.90995572453517, -547.2107155315117),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T17': {'xy': (-247.10414912221685, -21.41253156777379),
         'radius': 36.13743060919757,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T17',
         'label_xy': (-247.10414912221685, -21.41253156777379),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T18': {'xy': (-456.34961959011713, 23.36189380185674),
         'radius': 36.13743060919757,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T18',
         'label_xy': (-456.34961959011713, 23.36189380185674),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T19': {'xy': (346.10158888446, -24.145338775192194),
         'radius': 34.433756729740644,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T19',
         'label_xy': (346.10158888446, -24.145338775192194),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T20': {'xy': (60.956244966488406, 19.807594317209322),
         'radius': 45.0,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T20',
         'label_xy': (60.956244966488406, 19.807594317209322),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T21': {'xy': (-165.3262540896239, 438.80533957551893),
         'radius': 34.433756729740644,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T21',
         'label_xy': (-165.3262540896239, 438.80533957551893),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T22': {'xy': (-1.0475994254180077, -234.56665893149378),
         'radius': 37.67766952966369,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T22',
         'label_xy': (-1.0475994254180077, -234.56665893149378),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0}}


## Edges and numeric edge labels

In [14]:
# GRAPH 2: every edge and numeric edge label links here.
# Set control_xy=(x, y) to bend one edge; None keeps a straight edge.
EDGE_CONFIG['graph2'] = {'T01|T13': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.41',
             'label_xy': (140.87575479914915, 80.07963801045273),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.61',
             'label_xy': (77.80459696692176, -201.27557563996507),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T16': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.30',
             'label_xy': (208.20664677879526, -338.32057884413445),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.57',
             'label_xy': (83.35751020271083, -67.24941175150501),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T04': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.53',
             'label_xy': (152.69636732126082, -32.56628119669119),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T14': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.28',
             'label_xy': (-70.20237026270351, -304.50728474882385),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T15': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.40',
             'label_xy': (279.0629993379611, -155.72678893339798),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T02': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.30',
             'label_xy': (172.50338412544266, -354.3241928849373),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.49',
             'label_xy': (-10.517504952025547, -10.697637713371762),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T01|T12': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.44',
             'label_xy': (0.22949924961990398, -164.37251871225567),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T02|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.28',
             'label_xy': (64.6005494203654, -412.5892093787235),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T02|T16': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '1.00',
             'label_xy': (182.2238023202638, -497.22211810707444),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T03|T06': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.53',
             'label_xy': (-39.200674539065936, 175.05362921392157),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T03|T07': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.60',
             'label_xy': (-27.649707996828532, 132.59836750067456),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T03|T21': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.27',
             'label_xy': (-39.23190217965822, 319.629578827976),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T03|T09': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.35',
             'label_xy': (-130.04979602294605, 299.4387528116537),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T03|T13': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.50',
             'label_xy': (155.74842499921246, 232.77077824860154),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T03|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.54',
             'label_xy': (75.4261965402442, 97.51691103963906),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T03|T04': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.54',
             'label_xy': (139.35067477905852, 132.79217182738708),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T04|T05': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.43',
             'label_xy': (312.5271108681948, 142.82656983311375),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T04|T07': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.46',
             'label_xy': (15.39809249846789, -43.7022062117762),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T04|T15': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.24',
             'label_xy': (290.4072552816421, -85.91005439181576),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T04|T13': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.40',
             'label_xy': (217.9428375811789, 166.79390189231776),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T04|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.55',
             'label_xy': (136.8472110767142, 24.911108967450193),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T05|T13': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.37',
             'label_xy': (347.71969868655015, 204.97875167711874),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T05|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.42',
             'label_xy': (242.79373043698746, 121.84414853240952),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T05|T19': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.37',
             'label_xy': (411.4573909533024, 74.27072079620507),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T06|T10': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.62',
             'label_xy': (-222.02132786598048, 166.42157800206755),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T06|T08': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.50',
             'label_xy': (-76.05508382243143, 304.56562805205976),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T06|T19': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.32',
             'label_xy': (118.3671406471047, 110.20253660415068),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T06|T09': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.45',
             'label_xy': (-243.5135318857598, 282.451104767651),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T06|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.53',
             'label_xy': (-192.747922232767, 104.21204670587787),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T06|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.59',
             'label_xy': (-44.9757959592128, 103.45713282879984),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T07|T21': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.52',
             'label_xy': (-160.54032540496263, 278.9734783603565),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T07|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.64',
             'label_xy': (-196.04245665317876, 62.66700103191159),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T07|T18': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.40',
             'label_xy': (-281.0179501019203, 50.90909259343748),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T07|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.72',
             'label_xy': (-107.59635648350731, 59.11633933464571),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T07|T08': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.46',
             'label_xy': (-41.71589094933951, 241.58026007972933),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T08|T21': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.57',
             'label_xy': (-87.47969419900241, 390.96341321078586),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T08|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.46',
             'label_xy': (209.33833808004914, 226.65962228294964),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T09|T10': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.31',
             'label_xy': (-312.20210589489847, 269.241139073349),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T09|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.35',
             'label_xy': (-268.8648200543306, 185.87233774367294),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T10|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.55',
             'label_xy': (-271.0153628049763, 73.77341782667672),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T10|T18': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.46',
             'label_xy': (-378.6297497274626, 75.58646302983935),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T10|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.50',
             'label_xy': (-238.88784048240538, 60.62986684090718),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T11|T19': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.42',
             'label_xy': (116.8137842736509, 165.68052811630886),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T11|T12': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.52',
             'label_xy': (-108.15434497794355, -114.45182656038409),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T11|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.74',
             'label_xy': (-194.51127777618828, 36.69413623154011),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T11|T18': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.40',
             'label_xy': (-301.1757336330444, -100.02219062651359),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T11|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.43',
             'label_xy': (-56.27852398706764, -114.93977241830157),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T11|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.58',
             'label_xy': (-35.54880316202706, 25.18415777768686),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T11|T14': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.24',
             'label_xy': (-194.53988735923346, -227.65923746369884),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T12|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.69',
             'label_xy': (-61.6469183263228, -213.41591109501417),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T12|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.47',
             'label_xy': (-46.76791272617339, -91.42253473777177),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T13|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.39',
             'label_xy': (180.1490813885653, 125.8173777679711),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T15|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.38',
             'label_xy': (225.53095901495576, -111.28850852472799),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T16|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.28',
             'label_xy': (103.00480194834361, -403.54864146975615),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T17|T18': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.53',
             'label_xy': (-355.50867160282695, -16.698842219197903),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T18|T21': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.19',
             'label_xy': (-327.7666885916749, 242.9424204769093),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T19|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.31',
             'label_xy': (200.5041198623488, -21.79229069423299),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'T20|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#6f7782',
             'label_text': '0.47',
             'label_xy': (47.75314167462189, -111.71800282290516),
             'label_fontsize': 8.0,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97}}


## Explanatory description boxes

In [15]:
# GRAPH 2: every explanatory description box links here.
NOTE_CONFIG['graph2'] = {'group_0': {'text': 'AI & computation',
             'xy': (83.51124078595235, 853.8158130674092),
             'fontsize': 11.5,
             'ha': 'center',
             'va': 'center',
             'fontweight': 'normal',
             'color': 'black',
             'linespacing': 1.08,
             'box_pad': 0.28,
             'box_facecolor': 'white',
             'box_edgecolor': '#d7dce1',
             'box_linewidth': 0.55,
             'box_alpha': 0.98},
 'group_1': {'text': 'Clinical &\nbiomedical',
             'xy': (201.14135261038254, -765.2122519716689),
             'fontsize': 11.5,
             'ha': 'center',
             'va': 'center',
             'fontweight': 'normal',
             'color': 'black',
             'linespacing': 1.08,
             'box_pad': 0.28,
             'box_facecolor': 'white',
             'box_edgecolor': '#d7dce1',
             'box_linewidth': 0.55,
             'box_alpha': 0.98},
 'group_2': {'text': 'Translation &\ngovernance',
             'xy': (-476.6581076954961, 450.53429537370715),
             'fontsize': 11.5,
             'ha': 'center',
             'va': 'center',
             'fontweight': 'normal',
             'color': 'black',
             'linespacing': 1.08,
             'box_pad': 0.28,
             'box_facecolor': 'white',
             'box_edgecolor': '#d7dce1',
             'box_linewidth': 0.55,
             'box_alpha': 0.98},
}


## Render Graph 2

Run this cell after edits. It outputs only Graph 2 in this section and writes its PNG and clickable SVG to `OUTPUT_DIR`.

In [16]:
diagnostics_by_graph['graph2'] = render_graph('graph2')
display(pd.DataFrame([diagnostics_by_graph['graph2']]))


,graph,nodes,edges,node_radius_scale,degree_vs_radius_spearman,radius_rule_max_abs_error,strength_vs_length_spearman,edge_label_mismatches,node_overlaps,edge_label_overlaps,edge_label_node_overlaps,annotation_overlaps,legend_entry_overlaps,pixel_width,pixel_height,png,svg
0,graph2,22,61,0.95,1.0,0.0,-0.744457,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/02_theme_co...,/content/cqd_network_graph_outputs/02_theme_co...


# Graph 3 — Pipeline–theme associations

**Nodes:** eight clinical-research stages and README themes. **Edges:** a stage and theme must co-occur in at least two publications; the graph keeps each stage's six strongest supported pairs and each theme's strongest supported pair. **Strength:** weighted stage-keyword evidence divided by the geometric mean of stage and theme totals. **Node radius:** connection count. This graph summarizes abstract wording and does not establish clinical validity, regulatory sufficiency, or causal evidence.

## Frame, titles, legend, and graph background

In [17]:
# GRAPH 3: title, subtitle, footer, legend and blank background link here.
FRAME_CONFIG['graph3'] = {'output_name': '03_pipeline_theme_association_network.png',
 'graph_rect': (0.025, 0.075, 0.705, 0.815),
 'legend_xy': (0.745, 0.055),
 'legend_wh': (0.245, 0.85),
 'title': {'text': 'Graph 3 - ChemicalQDevice Abstract-Derived Pipeline–Theme Association Network',
           'xy': (0.025, 0.975),
           'fontsize': 22.0,
           'ha': 'left',
           'va': 'top',
           'fontweight': 'bold',
           'color': '#111827'},
 'subtitle': {'text': 'Eight clinical-research stages connect to themes only when at least two README '
                      'publications support the pair. Strength is normalized weighted keyword evidence.',
              'xy': (0.025, 0.94),
              'fontsize': 11.2,
              'ha': 'left',
              'va': 'top',
              'fontweight': 'normal',
              'color': '#374151'},
 'footer': {'text': 'Node radius is square-root-scaled connection count. Stronger stage–theme associations are '
                    'shorter; labels report normalized association, not clinical evidence or causation.',
            'xy': (0.025, 0.018),
            'fontsize': 9.2,
            'ha': 'left',
            'va': 'bottom',
            'fontweight': 'normal',
            'color': '#374151'},
 'legend_title': 'Pipeline stages and supported themes',
 'legend_columns': 1,
 'legend_fontsize': 8.2,
 'legend_heading_fontsize': 11.5,
 'color_key_title': 'Color meaning',
 'color_key_fontsize': 8.3,
 'edge_color': '#75667d',
 'directed': False}


## Nodes, node labels, and legend entries

In [18]:
# GRAPH 3: every node, node label and legend item links here.
# Radius defaults follow the square-root degree rule validated by the renderer.
NODE_CONFIG['graph3'] = {'S01': {'xy': (479.4332893901212, 164.8336930638677),
         'radius': 43.33485466634829,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S01',
         'label_xy': (479.4332893901212, 164.8336930638677),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'S02': {'xy': (-238.41710174796913, 52.113439215216225),
         'radius': 41.5458710674107,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S02',
         'label_xy': (-238.41710174796913, 52.113439215216225),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'S03': {'xy': (107.86013408045575, -268.17165423033197),
         'radius': 41.5458710674107,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S03',
         'label_xy': (107.86013408045575, -268.17165423033197),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'S04': {'xy': (5.194470094019371, 14.883462454079725),
         'radius': 41.5458710674107,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S04',
         'label_xy': (5.194470094019371, 14.883462454079725),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'S05': {'xy': (-307.78094547390214, -93.95050110057427),
         'radius': 43.33485466634829,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S05',
         'label_xy': (-307.78094547390214, -93.95050110057427),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'S06': {'xy': (40.6463337694698, -85.4697766427589),
         'radius': 45.0,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S06',
         'label_xy': (40.6463337694698, -85.4697766427589),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'S07': {'xy': (-160.50283906590963, 408.3745913067524),
         'radius': 41.5458710674107,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S07',
         'label_xy': (-160.50283906590963, 408.3745913067524),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'S08': {'xy': (-6.848202885524691, 362.6636124649838),
         'radius': 41.5458710674107,
         'facecolor': '#e7c6e7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'S08',
         'label_xy': (-6.848202885524691, 362.6636124649838),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T01': {'xy': (263.7482112328157, 80.08876646180472),
         'radius': 28.333333333333332,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T01',
         'label_xy': (263.7482112328157, 80.08876646180472),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T02': {'xy': (710.8469561818505, 92.46924844171956),
         'radius': 28.333333333333332,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T02',
         'label_xy': (710.8469561818505, 92.46924844171956),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T03': {'xy': (-68.67220506507009, 163.98366533284292),
         'radius': 39.600467078786565,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T03',
         'label_xy': (-68.67220506507009, 163.98366533284292),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T04': {'xy': (-536.5782053008807, -113.17387296610471),
         'radius': 32.10901532768311,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T04',
         'label_xy': (-536.5782053008807, -113.17387296610471),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T05': {'xy': (632.6272369795591, 563.2112226719403),
         'radius': 28.333333333333332,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T05',
         'label_xy': (632.6272369795591, 563.2112226719403),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T06': {'xy': (-314.4909229956258, 173.45989720493768),
         'radius': 32.10901532768311,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T06',
         'label_xy': (-314.4909229956258, 173.45989720493768),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T07': {'xy': (-171.83327283827052, 147.9188354277467),
         'radius': 37.44863601130045,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T07',
         'label_xy': (-171.83327283827052, 147.9188354277467),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T08': {'xy': (-249.58672705506083, 447.3737315064359),
         'radius': 35.00619801997391,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T08',
         'label_xy': (-249.58672705506083, 447.3737315064359),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T09': {'xy': (-752.3657122128479, 79.69951462814626),
         'radius': 28.333333333333332,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T09',
         'label_xy': (-752.3657122128479, 79.69951462814626),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T10': {'xy': (-301.8260006876946, -238.81206797811294),
         'radius': 35.00619801997391,
         'facecolor': '#b9d8f2',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T10',
         'label_xy': (-301.8260006876946, -238.81206797811294),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T11': {'xy': (-97.62933911684627, -57.08780903058167),
         'radius': 41.5458710674107,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T11',
         'label_xy': (-97.62933911684627, -57.08780903058167),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T12': {'xy': (-21.513382253380612, -387.5741664116356),
         'radius': 32.10901532768311,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T12',
         'label_xy': (-21.513382253380612, -387.5741664116356),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T13': {'xy': (-662.0010476810962, -342.0939272828329),
         'radius': 28.333333333333332,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T13',
         'label_xy': (-662.0010476810962, -342.0939272828329),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T14': {'xy': (-625.5123682297392, -706.4412920713984),
         'radius': 28.333333333333332,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T14',
         'label_xy': (-625.5123682297392, -706.4412920713984),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T15': {'xy': (924.1640704065148, -38.35455178314523),
         'radius': 28.333333333333332,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T15',
         'label_xy': (924.1640704065148, -38.35455178314523),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T16': {'xy': (645.8513004359667, 336.68896750176503),
         'radius': 28.333333333333332,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T16',
         'label_xy': (645.8513004359667, 336.68896750176503),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T17': {'xy': (-51.185469232358976, -131.3995753578647),
         'radius': 37.44863601130045,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T17',
         'label_xy': (-51.185469232358976, -131.3995753578647),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T18': {'xy': (140.3837208359271, -601.728120923631),
         'radius': 32.10901532768311,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T18',
         'label_xy': (140.3837208359271, -601.728120923631),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T19': {'xy': (228.77913716295052, -467.32689620355706),
         'radius': 28.333333333333332,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T19',
         'label_xy': (228.77913716295052, -467.32689620355706),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T20': {'xy': (113.45695108764316, 105.31280220112303),
         'radius': 41.5458710674107,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T20',
         'label_xy': (113.45695108764316, 105.31280220112303),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T21': {'xy': (-94.53380026418762, 601.7219434614063),
         'radius': 32.10901532768311,
         'facecolor': '#f4d8b8',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T21',
         'label_xy': (-94.53380026418762, 601.7219434614063),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'T22': {'xy': (368.28573044907074, -263.2131813622395),
         'radius': 32.10901532768311,
         'facecolor': '#cde9c9',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'T22',
         'label_xy': (368.28573044907074, -263.2131813622395),
         'label_fontsize': 10.4,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0}}


## Edges and numeric edge labels

In [19]:
# GRAPH 3: every edge and numeric edge label links here.
# Set control_xy=(x, y) to bend one edge; None keeps a straight edge.
EDGE_CONFIG['graph3'] = {'S01|T05': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.44',
             'label_xy': (538.8666233824233, 370.62264372407475),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S01|T15': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.31',
             'label_xy': (693.8497932377035, 45.8413467645283),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S01|T02': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.52',
             'label_xy': (600.1725493709752, 144.7446266523988),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S01|T01': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.49',
             'label_xy': (377.8812981431691, 106.45109965671624),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S01|T16': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.52',
             'label_xy': (550.5292328768793, 262.49115177550453),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S01|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.48',
             'label_xy': (293.53257104763196, 152.9816504250145),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S01|T04': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.37',
             'label_xy': (-22.238269492414574, 2.6808699712532444),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S02|T12': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.41',
             'label_xy': (-162.25051117623582, -183.65711449918047),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S02|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.39',
             'label_xy': (-190.08725186232027, -30.93324820403061),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S02|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.49',
             'label_xy': (-58.89232051653383, 54.98280222321707),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S02|T10': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.44',
             'label_xy': (-234.94732882706091, -101.01573983302515),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S02|T08': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.36',
             'label_xy': (-262.55406707730424, 249.21932173488145),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S02|T06': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.67',
             'label_xy': (-263.4516600780676, 120.93802788368232),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S03|T18': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.48',
             'label_xy': (141.448488313203, -433.26045276935093),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S03|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.44',
             'label_xy': (237.76024132384933, -249.26947328637553),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S03|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.48',
             'label_xy': (17.035389106472486, -151.02565649552025),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S03|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.37',
             'label_xy': (128.4095932541932, -81.69543275350698),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S03|T12': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.58',
             'label_xy': (53.65722573348436, -339.2322401302502),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S03|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.45',
             'label_xy': (17.204150927959116, -212.73185047636542),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S04|T03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.39',
             'label_xy': (-46.200074866867844, 82.26924562010234),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S04|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.79',
             'label_xy': (107.63777671787363, -108.60631487771079),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S04|T22': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.38',
             'label_xy': (177.0067177379739, -136.87305356650668),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S04|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.61',
             'label_xy': (-25.57382867993266, -50.59524939560967),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S04|T10': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.39',
             'label_xy': (-139.43642173171088, -122.71001923937659),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S04|T07': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.50',
             'label_xy': (-92.72807994075383, 68.8811920771898),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S05|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.56',
             'label_xy': (-205.47702057270706, -59.71687259793597),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S05|T04': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.55',
             'label_xy': (-420.79798161404676, -120.00596556222943),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S05|T10': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.63',
             'label_xy': (-289.3141117954593, -165.74455049582886),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S05|T07': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.54',
             'label_xy': (-182.27257318716934, -5.354311013987328),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S05|T03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.44',
             'label_xy': (-307.6349409884497, 112.1384792251842),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S05|T09': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.35',
             'label_xy': (-536.9457559540188, -24.720518443059458),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S05|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.55',
             'label_xy': (-177.07227624100304, -96.15569751997671),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T17': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.46',
             'label_xy': (32.30546943227473, -183.56201178526504),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T18': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.47',
             'label_xy': (108.58406935252899, -340.1081402358689),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T13': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.32',
             'label_xy': (-318.9108064913137, -191.23833163551856),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.51',
             'label_xy': (-56.640554208234946, -208.4197090422485),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.49',
             'label_xy': (92.75143309274524, 3.929815544807793),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T14': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.13',
             'label_xy': (-276.068320397961, -413.51106618015496),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T19': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.45',
             'label_xy': (179.56467630576748, -254.30074825329544),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S06|T03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.47',
             'label_xy': (-59.80847999035448, 19.187860699565327),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S07|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.36',
             'label_xy': (3.1828180977849136, 280.98499480016807),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S07|T08': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.86',
             'label_xy': (-225.0964868521699, 382.07100458252205),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S07|T06': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.44',
             'label_xy': (-222.75959133175346, 281.2568566340951),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S07|T07': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.48',
             'label_xy': (-148.84121426711548, 277.39295528064775),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S07|T21': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.56',
             'label_xy': (-142.7692194181481, 510.25178936853695),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S07|T03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.42',
             'label_xy': (-98.0730281885564, 292.3844996398592),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S08|T03': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.44',
             'label_xy': (-20.86184966378782, 258.0653130896627),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S08|T11': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.41',
             'label_xy': (-3.3686417127985067, 142.21858464672098),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S08|T20': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.35',
             'label_xy': (69.85907175616711, 241.7271200193395),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S08|T08': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.55',
             'label_xy': (-149.96378326670825, 342.7041661259103),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S08|T07': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.50',
             'label_xy': (-75.50080784704576, 244.65822111787043),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97},
 'S08|T21': {'control_xy': None,
             'linewidth_scale': 1.0,
             'alpha_scale': 1.0,
             'color': '#75667d',
             'label_text': '0.59',
             'label_xy': (-66.61415263765113, 476.35223265664916),
             'label_fontsize': 8.6,
             'label_ha': 'center',
             'label_va': 'center',
             'label_color': '#111111',
             'label_box_pad': 0.18,
             'label_box_facecolor': 'white',
             'label_box_edgecolor': '#b8c0c8',
             'label_box_linewidth': 0.55,
             'label_box_alpha': 0.97}}


## Explanatory description boxes

In [20]:
# GRAPH 3: every explanatory description box links here.
NOTE_CONFIG['graph3'] = {'group_0': {'text': 'Pipeline stages',
             'xy': (-4.502915571493062, 659.040080450134),
             'fontsize': 11.5,
             'ha': 'center',
             'va': 'center',
             'fontweight': 'normal',
             'color': 'black',
             'linespacing': 1.08,
             'box_pad': 0.28,
             'box_facecolor': 'white',
             'box_edgecolor': '#d7dce1',
             'box_linewidth': 0.55,
             'box_alpha': 0.98},
 'group_1': {'text': 'AI & computation',
             'xy': (-520.4016466193376, 921.9215664220261),
             'fontsize': 11.5,
             'ha': 'center',
             'va': 'center',
             'fontweight': 'normal',
             'color': 'black',
             'linespacing': 1.08,
             'box_pad': 0.28,
             'box_facecolor': 'white',
             'box_edgecolor': '#d7dce1',
             'box_linewidth': 0.55,
             'box_alpha': 0.98},
 'group_2': {'text': 'Clinical &\nbiomedical',
             'xy': (396.38142581336865, -1087.1056198550345),
             'fontsize': 11.5,
             'ha': 'center',
             'va': 'center',
             'fontweight': 'normal',
             'color': 'black',
             'linespacing': 1.08,
             'box_pad': 0.28,
             'box_facecolor': 'white',
             'box_edgecolor': '#d7dce1',
             'box_linewidth': 0.55,
             'box_alpha': 0.98},
 'group_3': {'text': 'Translation &\ngovernance',
             'xy': (514.3487702285146, -753.3080585990824),
             'fontsize': 11.5,
             'ha': 'center',
             'va': 'center',
             'fontweight': 'normal',
             'color': 'black',
             'linespacing': 1.08,
             'box_pad': 0.28,
             'box_facecolor': 'white',
             'box_edgecolor': '#d7dce1',
             'box_linewidth': 0.55,
             'box_alpha': 0.98},
}


## Render Graph 3

Run this cell after edits. It outputs only Graph 3 in this section and writes its PNG and clickable SVG to `OUTPUT_DIR`.

In [21]:
diagnostics_by_graph['graph3'] = render_graph('graph3')
display(pd.DataFrame([diagnostics_by_graph['graph3']]))


,graph,nodes,edges,node_radius_scale,degree_vs_radius_spearman,radius_rule_max_abs_error,strength_vs_length_spearman,edge_label_mismatches,node_overlaps,edge_label_overlaps,edge_label_node_overlaps,annotation_overlaps,legend_entry_overlaps,pixel_width,pixel_height,png,svg
0,graph3,30,52,0.95,1.0,0.0,-0.546498,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/03_pipeline...,/content/cqd_network_graph_outputs/03_pipeline...


# Graph 4 — Temporal text-similarity precursors

**Nodes:** all 53 publications. **Arrows:** for each later paper, the most textually similar earlier abstract is retained; a second earlier match is added only when it is at least 0.10 and at least 80% of the best match. **Node radius:** total incoming plus outgoing links. **Edge label:** cosine similarity. The word “precursor” is temporal and textual only; arrows do not claim authorship dependence, replication, influence, or causation.

## Frame, titles, legend, and graph background

In [22]:
# GRAPH 4: title, subtitle, footer, legend and blank background link here.
FRAME_CONFIG['graph4'] = {'output_name': '04_temporal_text_similarity_precursor_network.png',
 'graph_rect': (0.025, 0.075, 0.705, 0.815),
 'legend_xy': (0.745, 0.055),
 'legend_wh': (0.245, 0.85),
 'title': {'text': 'Graph 4 - ChemicalQDevice Temporal Text-Similarity Precursor Network',
           'xy': (0.025, 0.975),
           'fontsize': 22.0,
           'ha': 'left',
           'va': 'top',
           'fontweight': 'bold',
           'color': '#111827'},
 'subtitle': {'text': 'Arrows run from earlier publications to the closest later abstract by TF-IDF/cosine. A '
                      'near-tied second earlier match is retained when supported.',
              'xy': (0.025, 0.94),
              'fontsize': 11.2,
              'ha': 'left',
              'va': 'top',
              'fontweight': 'normal',
              'color': '#374151'},
 'footer': {'text': 'Node radius is square-root-scaled total precursor-link count. Stronger textual matches '
                    'are shorter; labels report cosine similarity. Arrows do not establish intellectual '
                    'dependence.',
            'xy': (0.025, 0.018),
            'fontsize': 9.2,
            'ha': 'left',
            'va': 'bottom',
            'fontweight': 'normal',
            'color': '#374151'},
 'legend_title': 'Publication index (oldest → newest)',
 'legend_columns': 2,
 'legend_fontsize': 7.5,
 'legend_heading_fontsize': 11.5,
 'color_key_title': 'Color meaning',
 'color_key_fontsize': 8.3,
 'edge_color': '#6f7782',
 'directed': True}


## Nodes, node labels, and legend entries

In [23]:
# GRAPH 4: every node, node label and legend item links here.
# Radius defaults follow the square-root degree rule validated by the renderer.
NODE_CONFIG['graph4'] = {'P01': {'xy': (-3885.685794424698, 1636.8703004355011),
         'radius': 69.38456493479724,
         'facecolor': '#d8e8f7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P01',
         'label_xy': (-3885.685794424698, 1636.8703004355011),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P02': {'xy': (-3678.152586448981, 2191.7220547483225),
         'radius': 69.38456493479724,
         'facecolor': '#d8e8f7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P02',
         'label_xy': (-3678.152586448981, 2191.7220547483225),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P03': {'xy': (-3381.2806484564703, 2033.747805825835),
         'radius': 69.38456493479724,
         'facecolor': '#d8e8f7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P03',
         'label_xy': (-3381.2806484564703, 2033.747805825835),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P04': {'xy': (-3214.547645554165, 1435.7903270321221),
         'radius': 76.26952622888624,
         'facecolor': '#d8e8f7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P04',
         'label_xy': (-3214.547645554165, 1435.7903270321221),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P05': {'xy': (-3000.102231440103, 951.5706628551892),
         'radius': 76.26952622888624,
         'facecolor': '#d8e8f7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P05',
         'label_xy': (-3000.102231440103, 951.5706628551892),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P06': {'xy': (-3387.027525849325, 750.9738826552949),
         'radius': 76.26952622888624,
         'facecolor': '#d8e8f7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P06',
         'label_xy': (-3387.027525849325, 750.9738826552949),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P07': {'xy': (-3191.806622465181, 269.20457548558966),
         'radius': 76.26952622888624,
         'facecolor': '#d8e8f7',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P07',
         'label_xy': (-3191.806622465181, 269.20457548558966),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P08': {'xy': (-3820.1410976582038, 218.83776032099718),
         'radius': 69.38456493479724,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P08',
         'label_xy': (-3820.1410976582038, 218.83776032099718),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P09': {'xy': (-2559.3525212556897, 404.71694950911075),
         'radius': 82.07381709333261,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P09',
         'label_xy': (-2559.3525212556897, 404.71694950911075),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P10': {'xy': (-2867.80556961872, -113.4398673015854),
         'radius': 69.38456493479724,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P10',
         'label_xy': (-2867.80556961872, -113.4398673015854),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P11': {'xy': (-2434.3921409754726, -344.16652917533315),
         'radius': 69.38456493479724,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P11',
         'label_xy': (-2434.3921409754726, -344.16652917533315),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P12': {'xy': (-1885.836194539076, -21.06609614663531),
         'radius': 82.07381709333261,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P12',
         'label_xy': (-1885.836194539076, -21.06609614663531),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P13': {'xy': (-1985.7856497428122, -611.2395191466749),
         'radius': 69.38456493479724,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P13',
         'label_xy': (-1985.7856497428122, -611.2395191466749),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P14': {'xy': (-1615.5145089153957, -877.5060215462178),
         'radius': 76.26952622888624,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P14',
         'label_xy': (-1615.5145089153957, -877.5060215462178),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P15': {'xy': (-1802.2084947291223, -1436.3879104555813),
         'radius': 69.38456493479724,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P15',
         'label_xy': (-1802.2084947291223, -1436.3879104555813),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P16': {'xy': (-2037.1472622735687, -1958.3036519925968),
         'radius': 60.41190868531085,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P16',
         'label_xy': (-2037.1472622735687, -1958.3036519925968),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P17': {'xy': (-1165.5961703385908, -245.89636449647406),
         'radius': 87.1875,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P17',
         'label_xy': (-1165.5961703385908, -245.89636449647406),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P18': {'xy': (-901.1953927013054, -832.7728422799681),
         'radius': 76.26952622888624,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P18',
         'label_xy': (-901.1953927013054, -832.7728422799681),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P19': {'xy': (-741.0742103158809, -527.3695403648035),
         'radius': 82.07381709333261,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P19',
         'label_xy': (-741.0742103158809, -527.3695403648035),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P20': {'xy': (-258.53767842169515, -1126.8764974214507),
         'radius': 76.26952622888624,
         'facecolor': '#a9d1ee',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P20',
         'label_xy': (-258.53767842169515, -1126.8764974214507),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P21': {'xy': (625.606120796373, -981.0720777582846),
         'radius': 82.07381709333261,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P21',
         'label_xy': (625.606120796373, -981.0720777582846),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P22': {'xy': (1167.8264578954584, -497.3057708012637),
         'radius': 87.1875,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P22',
         'label_xy': (1167.8264578954584, -497.3057708012637),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P23': {'xy': (1131.3832847610672, -1072.5766996701502),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P23',
         'label_xy': (1131.3832847610672, -1072.5766996701502),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P24': {'xy': (1447.6146120624512, -1034.7647050071646),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P24',
         'label_xy': (1447.6146120624512, -1034.7647050071646),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P25': {'xy': (1557.936048399438, -236.3790338499917),
         'radius': 87.1875,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P25',
         'label_xy': (1557.936048399438, -236.3790338499917),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P26': {'xy': (1822.9453876649288, -47.45478853453589),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P26',
         'label_xy': (1822.9453876649288, -47.45478853453589),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P27': {'xy': (1946.5666401770172, 35.37279974181248),
         'radius': 60.41190868531085,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P27',
         'label_xy': (1946.5666401770172, 35.37279974181248),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P28': {'xy': (629.7813931327219, -392.6885080183177),
         'radius': 82.07381709333261,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P28',
         'label_xy': (629.7813931327219, -392.6885080183177),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P29': {'xy': (1902.902022003161, -1524.3276518481487),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P29',
         'label_xy': (1902.902022003161, -1524.3276518481487),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P30': {'xy': (2195.941561265151, -807.5725335594499),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P30',
         'label_xy': (2195.941561265151, -807.5725335594499),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P31': {'xy': (774.7951605190556, 102.95646318176513),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P31',
         'label_xy': (774.7951605190556, 102.95646318176513),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P32': {'xy': (85.36322621431688, -85.10602672853243),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P32',
         'label_xy': (85.36322621431688, -85.10602672853243),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P33': {'xy': (1944.0435584216852, -2035.914296063193),
         'radius': 60.41190868531085,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P33',
         'label_xy': (1944.0435584216852, -2035.914296063193),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P34': {'xy': (-472.6126441296933, 390.408791576187),
         'radius': 82.07381709333261,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P34',
         'label_xy': (-472.6126441296933, 390.408791576187),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P35': {'xy': (-235.42163066599724, 553.9880160314857),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P35',
         'label_xy': (-235.42163066599724, 553.9880160314857),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P36': {'xy': (-555.2869126149947, 1032.9221886555551),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P36',
         'label_xy': (-555.2869126149947, 1032.9221886555551),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P37': {'xy': (1575.4471164071065, 536.5716610051422),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P37',
         'label_xy': (1575.4471164071065, 536.5716610051422),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P38': {'xy': (2020.4953608063681, 1081.6481796638566),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P38',
         'label_xy': (2020.4953608063681, 1081.6481796638566),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P39': {'xy': (2599.3600259180207, -847.0759221589923),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P39',
         'label_xy': (2599.3600259180207, -847.0759221589923),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P40': {'xy': (3019.914438608529, -934.2776901322887),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P40',
         'label_xy': (3019.914438608529, -934.2776901322887),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P41': {'xy': (3239.020352908215, -1178.144403715803),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P41',
         'label_xy': (3239.020352908215, -1178.144403715803),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P42': {'xy': (3318.7621096254206, -1310.673754813611),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P42',
         'label_xy': (3318.7621096254206, -1310.673754813611),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P43': {'xy': (3418.1268942898296, -1187.3161791590949),
         'radius': 76.26952622888624,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P43',
         'label_xy': (3418.1268942898296, -1187.3161791590949),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P44': {'xy': (2435.804487351853, 1617.7259720115649),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P44',
         'label_xy': (2435.804487351853, 1617.7259720115649),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P45': {'xy': (2598.6417689871932, 1830.57431148376),
         'radius': 60.41190868531085,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P45',
         'label_xy': (2598.6417689871932, 1830.57431148376),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P46': {'xy': (3490.3937890066904, -677.9146753563736),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P46',
         'label_xy': (3490.3937890066904, -677.9146753563736),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P47': {'xy': (700.9222467043528, 549.6173595547676),
         'radius': 87.1875,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P47',
         'label_xy': (700.9222467043528, 549.6173595547676),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P48': {'xy': (867.0480501015073, 566.5818296548778),
         'radius': 60.41190868531085,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P48',
         'label_xy': (867.0480501015073, 566.5818296548778),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P49': {'xy': (-278.22500350548506, -66.27998830060936),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P49',
         'label_xy': (-278.22500350548506, -66.27998830060936),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P50': {'xy': (264.95592566260467, 323.1971655052851),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P50',
         'label_xy': (264.95592566260467, 323.1971655052851),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P51': {'xy': (780.3091298317755, 982.7649552619034),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P51',
         'label_xy': (780.3091298317755, 982.7649552619034),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P52': {'xy': (886.3160166228873, 1688.5372010453443),
         'radius': 69.38456493479724,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P52',
         'label_xy': (886.3160166228873, 1688.5372010453443),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0},
 'P53': {'xy': (906.5129508954367, 1825.568332561858),
         'radius': 60.41190868531085,
         'facecolor': '#76b8df',
         'edgecolor': '#202833',
         'linewidth': 1.3,
         'alpha': 0.98,
         'label_text': 'P53',
         'label_xy': (906.5129508954367, 1825.568332561858),
         'label_fontsize': 10.0,
         'label_ha': 'center',
         'label_va': 'center',
         'label_fontweight': 'bold',
         'label_color': '#0d1117',
         'label_rotation': 0.0}}


## Edges and numeric edge labels

In [24]:
# GRAPH 4: every edge and numeric edge label links here.
# Set control_xy=(x, y) to bend one edge; None keeps a straight edge.
EDGE_CONFIG['graph4'] = {'P01->P02': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.24',
              'label_xy': (-3800.4066782822847, 1921.2111195203718),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P01->P04': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.23',
              'label_xy': (-3544.416358964433, 1555.356225073757),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P02->P03': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.54',
              'label_xy': (-3521.73632482129, 2127.7318361461093),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P03->P04': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.27',
              'label_xy': (-3279.144242080655, 1740.0028208735137),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P04->P05': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.37',
              'label_xy': (-3090.3467380351813, 1201.1995973228697),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P05->P06': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.46',
              'label_xy': (-3185.3957220035245, 835.5150241328197),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P05->P09': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.24',
              'label_xy': (-2764.3495775185565, 690.5379075878697),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P06->P07': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.28',
              'label_xy': (-3271.502100257651, 517.3486732416163),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P06->P08': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.18',
              'label_xy': (-3587.8656860049814, 472.112196217758),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P07->P08': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.23',
              'label_xy': (-3504.386526833153, 224.21891938208842),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P07->P09': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.26',
              'label_xy': (-2879.6782734083154, 356.089942561308),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P09->P10': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.31',
              'label_xy': (-2697.1765428426693, 135.87431189375627),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P09->P12': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.18',
              'label_xy': (-2211.7792496265183, 208.93308766012174),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P10->P11': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.32',
              'label_xy': (-2642.152791949672, -211.9982743654287),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P11->P12': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.22',
              'label_xy': (-2170.2020078619703, -165.48930164382705),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P12->P13': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.32',
              'label_xy': (-1917.036546247211, -319.33236228134194),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P12->P17': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.19',
              'label_xy': (-1519.6916234880264, -114.18165595208104),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P13->P14': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.40',
              'label_xy': (-1789.9892304376986, -729.547756138365),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P14->P15': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.35',
              'label_xy': (-1691.047692008911, -1162.8976529395507),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P14->P17': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.21',
              'label_xy': (-1406.819221661083, -550.1158432694651),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P15->P16': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.32',
              'label_xy': (-1902.3298584772924, -1705.1549391387068),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P17->P18': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.29',
              'label_xy': (-1015.7787543783611, -531.3977446926998),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P17->P19': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.31',
              'label_xy': (-942.788883593452, -370.72685838876464),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P17->P34': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.13',
              'label_xy': (-833.120079731467, 87.52032079514349),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P18->P19': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.39',
              'label_xy': (-837.4287137355788, -671.5283875114632),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P18->P20': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.18',
              'label_xy': (-571.4291853638683, -961.3878769112326),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P19->P20': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.21',
              'label_xy': (-484.2307067881476, -814.5866820964883),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P19->P49': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.18',
              'label_xy': (-523.9528740989031, -282.4669116265945),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P20->P21': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.10',
              'label_xy': (180.11725608397546, -1033.2541440217744),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P21->P22': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.25',
              'label_xy': (883.6626599912721, -724.5580101229721),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P21->P23': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.27',
              'label_xy': (881.965931714517, -1007.6377258994089),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P21->P28': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.19',
              'label_xy': (607.495211707973, -686.7369604915765),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P22->P23': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.31',
              'label_xy': (1168.7114307587276, -786.1516277975144),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P22->P25': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.15',
              'label_xy': (1351.4725344863407, -349.7853133624106),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P22->P28': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.21',
              'label_xy': (894.9826743491802, -464.6497787143029),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P22->P37': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.15',
              'label_xy': (1352.490279194265, 27.18172403988533),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P23->P24': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.52',
              'label_xy': (1287.4597004579869, -1036.6159531019207),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P24->P25': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.16',
              'label_xy': (1482.5239111405408, -632.7735155469542),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P24->P29': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.21',
              'label_xy': (1689.8420287654799, -1265.983509142868),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P25->P26': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.52',
              'label_xy': (1680.4540463032508, -127.90832657716209),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P25->P30': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.20',
              'label_xy': (1890.375610811748, -506.9672860685275),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P25->P47': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.11',
              'label_xy': (1115.3139073654304, 141.22856341001628),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P26->P27': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.85',
              'label_xy': (1848.0189525123972, 48.78954744905981),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P28->P31': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.38',
              'label_xy': (684.5348236549883, -139.67179009872785),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P28->P32': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.20',
              'label_xy': (347.6847062563523, -256.39823460997826),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P29->P30': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.22',
              'label_xy': (2030.9264281595283, -1158.3884127760323),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P29->P33': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.37',
              'label_xy': (1941.8848094491307, -1778.6402887803806),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P30->P39': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.48',
              'label_xy': (2399.358437292493, -809.885344147811),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P31->P32': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.20',
              'label_xy': (435.3777398786799, -10.499108083802273),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P31->P37': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.14',
              'label_xy': (1165.288349080709, 337.9198880201042),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P32->P34': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.16',
              'label_xy': (-206.89630659162913, 137.0783004596218),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P34->P35': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.61',
              'label_xy': (-363.3136734183356, 485.67844570857375),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P34->P36': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.24',
              'label_xy': (-533.5307684448921, 709.1459410483737),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P35->P36': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.21',
              'label_xy': (-412.0196101241894, 782.3248395692951),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P35->P47': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.12',
              'label_xy': (232.8475199818547, 572.6288100346193),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P37->P38': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.30',
              'label_xy': (1783.1380156540126, 821.2210636472537),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P38->P44': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.26',
              'label_xy': (2212.7245861926385, 1361.6373630828582),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P39->P40': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.46',
              'label_xy': (2813.24471543721, -873.2787303638552),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P40->P41': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.54',
              'label_xy': (3142.085149747449, -1044.8744258511642),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P40->P46': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.37',
              'label_xy': (3246.2578031699013, -789.769605942962),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P41->P42': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.92',
              'label_xy': (3222.338938868746, -1278.4361012402333),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P41->P43': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.80',
              'label_xy': (3334.199186290003, -1072.8742354168098),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P42->P43': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.82',
              'label_xy': (3419.8436025347314, -1290.3970497825665),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P43->P46': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.30',
              'label_xy': (3435.2213096376395, -929.9144306734066),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P44->P45': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.63',
              'label_xy': (2504.361552495392, 1733.9897487914202),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P47->P48': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.75',
              'label_xy': (772.8102604954681, 667.5304946297044),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P47->P50': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.30',
              'label_xy': (491.7747630008925, 419.3943916941331),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P47->P51': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.49',
              'label_xy': (723.398348274298, 769.3467352561562),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P49->P50': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.30',
              'label_xy': (-17.83237423869067, 144.075552351967),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P51->P52': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.21',
              'label_xy': (813.5221343746223, 1338.6235991383176),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97},
 'P52->P53': {'control_xy': None,
              'linewidth_scale': 1.0,
              'alpha_scale': 1.0,
              'color': '#6f7782',
              'label_text': '0.98',
              'label_xy': (979.5166925262763, 1744.8043833173215),
              'label_fontsize': 8.0,
              'label_ha': 'center',
              'label_va': 'center',
              'label_color': '#111111',
              'label_box_pad': 0.18,
              'label_box_facecolor': 'white',
              'label_box_edgecolor': '#b8c0c8',
              'label_box_linewidth': 0.55,
              'label_box_alpha': 0.97}}


## Explanatory description boxes

In [25]:
# GRAPH 4: every explanatory description box links here.
NOTE_CONFIG['graph4'] = {'year_2024': {'text': '2024 foundations',
               'xy': (-4501.587824691532, 1757.8615341582417),
               'fontsize': 11.5,
               'ha': 'center',
               'va': 'center',
               'fontweight': 'normal',
               'color': 'black',
               'linespacing': 1.08,
               'box_pad': 0.28,
               'box_facecolor': 'white',
               'box_edgecolor': '#d7dce1',
               'box_linewidth': 0.55,
               'box_alpha': 0.98},
 'year_2025': {'text': '2025 simulations\n& validation',
               'xy': (-3985.111548342461, -1236.7664722284644),
               'fontsize': 11.5,
               'ha': 'center',
               'va': 'center',
               'fontweight': 'normal',
               'color': 'black',
               'linespacing': 1.08,
               'box_pad': 0.28,
               'box_facecolor': 'white',
               'box_edgecolor': '#d7dce1',
               'box_linewidth': 0.55,
               'box_alpha': 0.98},
 'year_2026': {'text': '2026 trials,\npolicy & funding',
               'xy': (3838.7511505640414, -144.38790766129898),
               'fontsize': 11.5,
               'ha': 'center',
               'va': 'center',
               'fontweight': 'normal',
               'color': 'black',
               'linespacing': 1.08,
               'box_pad': 0.28,
               'box_facecolor': 'white',
               'box_edgecolor': '#d7dce1',
               'box_linewidth': 0.55,
               'box_alpha': 0.98},
}


## Render Graph 4

Run this cell after edits. It outputs only Graph 4 in this section and writes its PNG and clickable SVG to `OUTPUT_DIR`.

In [26]:
diagnostics_by_graph['graph4'] = render_graph('graph4')
display(pd.DataFrame([diagnostics_by_graph['graph4']]))


,graph,nodes,edges,node_radius_scale,degree_vs_radius_spearman,radius_rule_max_abs_error,strength_vs_length_spearman,edge_label_mismatches,node_overlaps,edge_label_overlaps,edge_label_node_overlaps,annotation_overlaps,legend_entry_overlaps,pixel_width,pixel_height,png,svg
0,graph4,53,71,1.15,1.0,0.0,-0.781462,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/04_temporal...,/content/cqd_network_graph_outputs/04_temporal...


# Final validation and audit-file export

In [27]:
if set(diagnostics_by_graph) != set(GRAPHS):
    missing = sorted(set(GRAPHS) - set(diagnostics_by_graph))
    raise AssertionError(f'Render each graph section before final export. Missing: {missing}')

diagnostics_df = pd.DataFrame([diagnostics_by_graph[name] for name in ('graph1', 'graph2', 'graph3', 'graph4')])
diagnostics_df.to_csv(OUTPUT_DIR / 'network_graph_diagnostics.csv', index=False)
publication_table.to_csv(OUTPUT_DIR / 'publication_index_with_abstracts.csv', index=False)
concept_presence.assign(publication_id=df['id']).to_csv(OUTPUT_DIR / 'publication_concept_matrix.csv', index=False)
stages.assign(publication_id=df['id']).to_csv(OUTPUT_DIR / 'publication_stage_scores.csv', index=False)
concept_audit.to_csv(OUTPUT_DIR / 'concept_match_audit.csv', index=False)
stage_audit.to_csv(OUTPUT_DIR / 'stage_match_audit.csv', index=False)
for graph_name, audit_df in EDGE_AUDIT.items():
    audit_df.to_csv(OUTPUT_DIR / f'{graph_name}_edge_audit.csv', index=False)

display(diagnostics_df)
print(f'All verified outputs created in: {OUTPUT_DIR}')


,graph,nodes,edges,node_radius_scale,degree_vs_radius_spearman,radius_rule_max_abs_error,strength_vs_length_spearman,edge_label_mismatches,node_overlaps,edge_label_overlaps,edge_label_node_overlaps,annotation_overlaps,legend_entry_overlaps,pixel_width,pixel_height,png,svg
0,graph1,53,71,1.20,1.0,0.0,-0.761502,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/01_publicat...,/content/cqd_network_graph_outputs/01_publicat...
1,graph2,22,61,0.95,1.0,0.0,-0.744457,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/02_theme_co...,/content/cqd_network_graph_outputs/02_theme_co...
2,graph3,30,52,0.95,1.0,0.0,-0.546498,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/03_pipeline...,/content/cqd_network_graph_outputs/03_pipeline...
3,graph4,53,71,1.15,1.0,0.0,-0.781462,0,0,0,0,0,0,3840,2720,/content/cqd_network_graph_outputs/04_temporal...,/content/cqd_network_graph_outputs/04_temporal...


All verified outputs created in: /content/cqd_network_graph_outputs


## Interpretation safeguards and verified corrections

- The embedded README canonical checksum matches the uploaded file after trailing-whitespace normalization, and all 53 dated publication blocks and DOIs are present.
- Parsing stops before both **Table of Contents** and **Contents** headings; this removes the site-document package's contents list from P29's abstract.
- Pipeline-stage patterns use word boundaries. In particular, `IND`, `IDE`, and `ICH` no longer match ordinary words such as “findings,” “provide,” or “which.”
- Four reviewed concept false positives are stored explicitly in `CONCEPT_OVERRIDES`: biological “regulatory mechanisms,” a table row-count “matching,” the TIME tumor-immunology acronym, and a simulated treatment “policy.”
- TF-IDF/cosine values are textual similarities, not clinical-effect correlations.
- Theme associations are normalized co-mentions, not causal or inferential statistics.
- Pipeline links require at least two supporting publications and summarize wording only.
- Temporal arrows identify earlier textual matches, not intellectual lineage, replication, or causation.
- Graph outputs are generated when the user runs the notebook; no images or execution outputs are embedded in this delivered `.ipynb`.
- Publication-quality validation now requires zero node overlaps and zero rendered legend-entry overlaps; additional edge-label and annotation overlap counts are included in the diagnostics table.
